# Modelling, Level 0: search and ranking on OTTO sessions

Follows `_doc/modeling_guide.md` (plan-based), one section per step. Decisions are in `modeling/model_log.md`, experiments in `modeling/experiments.csv`, new or changed decisions in `docs/decisions/`, one issue per step (umbrella #121). Outputs are not committed with the notebook, so each step explains its candidates, why it exists and what the result means in text cells.

Rules that shape everything: modelling reads weeks 1-2 only (weeks 3-4 are Level 1 arrivals); anything learned uses the training window only; the test slice is used once, in Step 11; seed 42.

In [ ]:
import json
import platform
import subprocess
import sys
import time
from pathlib import Path

import numpy as np
import polars as pl

ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT / "notebooks"))            # older helpers not yet moved into the package (covis.py, queries.py, ...)
from ranking.data.split import DAY_MS, LEVEL0, START_MS, day_to_ms, level0_slice
from ranking.eval.bootstrap import bootstrap_ci
from ranking.eval.metrics import ndcg_at_k, otto_weighted_recall_session, recall_at_k
from ranking.logbook import (
    SEED,
    cached,
    commit_sha,
    log_experiments,
    log_step,
    save_table,
    snapshot_id,
)

events = pl.scan_parquet(ROOT / "data" / "sample" / "events.parquet")
wk12 = events.filter(pl.col("ts") < day_to_ms(14))       # weeks 1-2 only: nothing later is ever read for modelling
print("python", platform.python_version(), "| seed", SEED, "| snapshot", snapshot_id(), "| commit", commit_sha())

## Step 0. Setup and checks
*Kind: diagnostic. Issue #122.* What this phase settles, what the sample looks like (weeks 1-2 only), whether it has enough carts and orders, what the licence allows, and which environment pieces (Docker, AWS, Bedrock) exist.

#### Why this step exists here
Everything after this depends on a sample that is large enough, a licence that allows the intended use, and an environment that can build the models. Problems found now (too few carts and orders, no Docker, no Bedrock access) cost little; found at Step 11 they cost a rebuild.

#### Options considered
- **Sample check** on weeks 1-2 only: counts, event mix, sessions with carts and orders, length and item-frequency distributions, week-over-week shift, duplicates.
- **Positives per head:** a stated minimum (5,000 training sessions and 2,000 evaluation sessions per head; the 95 percent interval of a proportion is then at most +/-0.014 and +/-0.022). If a head falls short, oversample sessions containing it and record the reweighting.
- **Licence, region and Bedrock, arm64 image:** checked and recorded; what cannot be checked without the human is listed.
- **Logs:** the experiment table, the Bedrock cost log, the manual-steps log.

In [ ]:
# A. the plan's Provisional decisions that this phase settles
prov = pl.DataFrame([
    {"decision": "D3", "topic": "session sample and split", "settled_in": "Step 1 (docs/decisions/D3-split-rule.md)"},
    {"decision": "D4", "topic": "synthetic categories: K and the naming alignment", "settled_in": "Step 2"},
    {"decision": "D8", "topic": "five methods and the shared candidate pool", "settled_in": "Step 5"},
    {"decision": "D15", "topic": "position bias: simulation, IPW support, train on simulated exposure?", "settled_in": "Step 8 (asks the human)"},
    {"decision": "D18", "topic": "explainer: numeric payload, faithfulness test, attribution method", "settled_in": "Step 9 (needs Bedrock)"},
    {"decision": "D21", "topic": "training compute: Mac and Colab, CPU-sized runs, spot training?", "settled_in": "Steps 4 and 7 (timed)"},
    {"decision": "D25", "topic": "one feature implementation shared by training and serving", "settled_in": "starts now (point-in-time rule in D3), finished in Steps 6 and 11"},
])
save_table("step0_provisional_decisions", prov); print(prov)

In [ ]:
# B. the sample, weeks 1-2 only
day = ((pl.col("ts") - START_MS) // DAY_MS)
ev = wk12.with_columns(day.alias("day"), (day // 7).alias("week"))
by_week = (ev.group_by("week").agg(pl.len().alias("events"), (pl.col("type") == 0).mean().alias("click_share"), (pl.col("type") == 1).mean().alias("cart_share"), (pl.col("type") == 2).mean().alias("order_share"),
           pl.col("aid").n_unique().alias("distinct_items")).sort("week").collect())
per_session = ev.group_by("session").agg(pl.col("week").min().alias("week"), pl.len().alias("n_events"), (pl.col("type") == 1).any().alias("has_cart"), (pl.col("type") == 2).any().alias("has_order")).collect()
sess_week = per_session.group_by("week").agg(pl.len().alias("sessions"), pl.col("n_events").median().alias("median_length"), pl.col("n_events").mean().alias("mean_length"), pl.col("has_cart").mean().alias("share_with_cart"), pl.col("has_order").mean().alias("share_with_order")).sort("week")
week_tbl = by_week.join(sess_week, on="week"); save_table("step0_sample_by_week", week_tbl); print(week_tbl)
len_q = per_session["n_events"].quantile
print("session length quantiles (events): " + ", ".join(f"p{int(p * 100)} {len_q(p):.0f}" for p in (0.5, 0.75, 0.9, 0.99)) + f", max {per_session['n_events'].max()}")
item_freq = ev.group_by("aid").len().collect()["len"]
print("events per item quantiles: " + ", ".join(f"p{int(p * 100)} {item_freq.quantile(p):.0f}" for p in (0.5, 0.9, 0.99)) + f", max {item_freq.max()}; items seen {item_freq.len():,}")
dups = int(ev.select(pl.struct("session", "aid", "ts", "type").is_duplicated().sum()).collect().item())
w1, w2 = week_tbl.row(0, named=True), week_tbl.row(1, named=True)
shift = {c: (w2[c] - w1[c]) / w1[c] for c in ("events", "click_share", "cart_share", "order_share", "median_length", "share_with_cart", "share_with_order")}
print(f"duplicate event rows (all four columns equal): {dups:,} ({dups / ev.select(pl.len()).collect().item():.3%})")
print("week 2 versus week 1, relative change:", {k: f"{v:+.1%}" for k, v in shift.items()})
# the raw length shift is a censoring artefact: sessions can last for days, and week-2 sessions are cut at day 14. Limit every session to its first 2 days (like the evaluation slices):
first_ts = ev.group_by("session").agg(pl.col("ts").min().alias("f"))
lim = ev.join(first_ts, on="session").filter(pl.col("ts") < pl.col("f") + 2 * DAY_MS).group_by("session").agg(pl.col("f").first(), pl.len().alias("n"), (pl.col("type") == 1).any().alias("c"), (pl.col("type") == 2).any().alias("o")).collect()
lim = lim.with_columns(((pl.col("f") - START_MS) // DAY_MS).alias("start_day"))
cens = {}
for nm, lo, hi in (("week 1 (starts days 0-6)", 0, 7), ("week 2 (starts days 7-11)", 7, 12)):
    x = lim.filter((pl.col("start_day") >= lo) & (pl.col("start_day") < hi)); cens[nm] = {"sessions": x.height, "median_length": float(x["n"].median()), "share_with_cart": float(x["c"].mean()), "share_with_order": float(x["o"].mean())}
save_table("step0_weeks_first_two_days", pl.DataFrame([{"window": k, **v} for k, v in cens.items()])); print("first 2 days of every session:", cens)

In [ ]:
# C. do the heads have enough positives? (a head = click, cart or order; a positive session is one with at least one such event)
MIN_TRAIN, MIN_EVAL = 5_000, 2_000            # 5,000 -> 95% interval half-width at most +/-0.014 on a proportion; 2,000 -> +/-0.022
rows = []
for name in LEVEL0:
    d = level0_slice(events, name).collect()
    ps = d.group_by("session").agg(*(( pl.col("type") == t).any().alias(h) for t, h in enumerate(("click", "cart", "order"))))
    for h in ("click", "cart", "order"):
        n = int(ps[h].sum()); need = MIN_TRAIN if name == "train" else MIN_EVAL
        rows.append({"slice": name, "head": h, "positive_sessions": n, "needed": need, "enough": n >= need})
heads = pl.DataFrame(rows); save_table("step0_positives_per_head", heads); print(heads)
oversample = not heads["enough"].all()
print("every head has enough positives in every slice; no oversampling or reweighting needed" if not oversample else "SOME HEAD HAS TOO FEW POSITIVES: oversample sessions with carts and orders and record the reweighting")

In [ ]:
# D. licence, E. environment
lic = (ROOT / "docs" / "otto-license.md").read_text()
unclear = [ln.strip() for ln in lic.splitlines() if ln.startswith("|") and "Unclear" in ln]
print("licence rows still marked Unclear:"); [print("  ", ln[:150]) for ln in unclear]
def run(cmd):
    try: r = subprocess.run(cmd, capture_output=True, text=True, timeout=20, check=False); return (r.stdout + r.stderr).strip().splitlines()[:1] or ["(no output)"]
    except (OSError, subprocess.TimeoutExpired) as e: return [f"{type(e).__name__}: {e}"]
import os

env = pl.DataFrame([
    {"check": "docker daemon", "result": run(["docker", "info", "--format", "{{.ServerVersion}} {{.Architecture}}"])[0][:110], "status": "blocked", "needed_from_human": "start Docker Desktop (needed for the arm64 image check)"},
    {"check": "AWS CLI", "result": run(["aws", "--version"])[0][:110], "status": "blocked", "needed_from_human": "install the AWS CLI and configure credentials and a region"},
    {"check": "AWS credentials", "result": "found" if (Path.home() / ".aws").exists() or os.environ.get("AWS_ACCESS_KEY_ID") else "none found (no ~/.aws, no AWS_* variables)", "status": "blocked", "needed_from_human": "AWS region choice and access; Bedrock model availability cannot be checked without it"},
    {"check": "Anthropic API key", "result": "set" if os.environ.get("ANTHROPIC_API_KEY") else "not set", "status": "blocked", "needed_from_human": "either Bedrock access or an API key for the Claude roles (Step 9)"},
    {"check": "model IDs in configuration", "result": (ROOT / "configs" / "llm.json").read_text().replace("\n", " ")[:110], "status": "ok (unverified availability)", "needed_from_human": "confirm the models exist in the chosen region"},
    {"check": "libraries for the models", "result": ", ".join(f"{m} ok" for m in ("lightgbm", "faiss", "torch") if __import__("importlib").util.find_spec(m)), "status": "ok on the Mac (onnxruntime not yet installed)", "needed_from_human": "none"},
])
save_table("step0_environment", env); print(env.select("check", "result", "status"))

In [ ]:
# F. the three logs the guide asks for (experiment table, Bedrock cost log, manual-steps log)
exp_path, cost_path, man_path = ROOT / "modeling" / "experiments.csv", ROOT / "modeling" / "bedrock_costs.csv", ROOT / "modeling" / "manual_steps.md"
if not exp_path.exists() or exp_path.stat().st_size == 0:
    log_experiments([{"id": "S0-sample-check", "step": "0", "description": "sample check of weeks 1-2", "features": "-", "model": "-", "parameters": "-", "seeds": "[42]", "metrics": {"events": int(week_tbl["events"].sum()), "sessions": int(week_tbl["sessions"].sum())}, "notes": "not a model run; first row of the experiment table"}])
cost_path.write_text("timestamp,step,role,model_id,calls,input_tokens,output_tokens,dollars,notes\n") if not cost_path.exists() else None
if not man_path.exists():
    man_path.write_text("# Manual steps log (Level 0 evidence for the MLOps tab)\n\nEvery step done by hand, with how long it took. Times marked ? were not recorded when the step was done.\n\n| Step | What was done by hand | Minutes |\n|---|---|---|\n"
        "| 4 | Run the sampling notebook on Kaggle and download the sample | ? |\n| 3 | Read the OTTO licence pages (Kaggle pages need a logged-in browser, so not verified) | ? |\n| 0 | Start Docker Desktop for the arm64 image check | pending |\n| 0 | Install and configure the AWS CLI, choose the region, enable Bedrock model access | pending |\n")
print("experiments.csv rows:", pl.read_csv(exp_path).height, "| bedrock_costs.csv and manual_steps.md exist:", cost_path.exists(), man_path.exists())

In [ ]:
e0 = log_step(0, "Setup and checks", "Diagnostic",
    "Provisional decisions listed (D3, D4, D8, D15, D18, D21, D25); sample check on weeks 1-2 only (counts, event mix, sessions with carts and orders, length and item-frequency distributions, week-over-week shift, duplicates); positives per head against stated minimums; licence rows; environment checks (Docker, AWS, credentials, Bedrock configuration, libraries); experiment table, Bedrock cost log and manual-steps log created. NOT done: the arm64 hello-world image (Docker daemon not running) and the Bedrock availability check (no AWS access).",
    f"Weeks 1-2 hold {int(week_tbl['events'].sum()):,} events in {int(week_tbl['sessions'].sum()):,} sessions; the event-type mix is the same in both weeks (week 2 versus week 1: events {shift['events']:+.1%}, click share {shift['click_share']:+.1%}, cart share {shift['cart_share']:+.1%}, order share {shift['order_share']:+.1%}). Raw session length looks much shorter in week 2 (median {shift['median_length']:+.1%}, share with a cart {shift['share_with_cart']:+.1%}) but that is censoring: sessions can last for days and week-2 sessions are cut at day 14; with every session limited to its first two days the weeks agree (median length {cens['week 1 (starts days 0-6)']['median_length']:.0f} against {cens['week 2 (starts days 7-11)']['median_length']:.0f}, share with a cart {cens['week 1 (starts days 0-6)']['share_with_cart']:.1%} against {cens['week 2 (starts days 7-11)']['share_with_cart']:.1%}, with an order {cens['week 1 (starts days 0-6)']['share_with_order']:.1%} against {cens['week 2 (starts days 7-11)']['share_with_order']:.1%}). Event rows that have an exact copy (first occurrence included) {dups:,}. Under the Level 0 split (Step 1) every head has enough positive sessions: " + "; ".join(f"{r['slice']} {r['head']} {r['positive_sessions']:,}" for r in heads.filter(pl.col('head') == 'order').iter_rows(named=True)) + f" (orders; the minimums are {MIN_TRAIN:,} training and {MIN_EVAL:,} evaluation sessions). Licence: {len(unclear)} rows still Unclear (publishing derived samples and trained models). Environment: Docker daemon not running, AWS CLI absent, no AWS or Anthropic credentials.",
    "No oversampling or reweighting is needed (D3). Steps 1 to 8 can proceed locally. Blocked on the human: Docker Desktop (arm64 image), AWS region and Bedrock or API access (Step 9), and the Unclear licence rows (publishing derived data).",
    "Guide step 0: list the decisions, check the sample and the positives, confirm the licence and Bedrock, build the image, create the logs. The positives minimums are stated here because the guide asks for a number. Session-level statistics are compared over a common two-day window because raw session length depends on how long a session can be observed.",
    "Oversampling carts and orders: not needed. Publishing any derived data: held back because the licence rows are Unclear. Checking week 3 or 4 for shifts: rejected, they are Level 1 arrivals.",
    "High for the sample checks; the environment items are open until the human acts.",
    "Step 1 fixes the split and the harness; Step 9 cannot start without Bedrock or API access and hand labels; the arm64 image must be built before Step 11.",
    human="yes: start Docker Desktop; choose the AWS region and give Bedrock or Anthropic API access; settle the two Unclear licence rows (stop-and-ask trigger in the guide)")
print(e0)

#### Chosen approach and rationale
**Decision.** No oversampling or reweighting is needed (D3). Steps 1 to 8 can proceed locally. Blocked on the human: Docker Desktop (arm64 image), AWS region and Bedrock or API access (Step 9), and the Unclear licence rows (publishing derived data).

**Why.** Guide step 0: list the decisions, check the sample and the positives, confirm the licence and Bedrock, build the image, create the logs. The positives minimums are stated here because the guide asks for a number. Session-level statistics are compared over a common two-day window because raw session length depends on how long a session can be observed.

**Rejected.** Oversampling carts and orders: not needed. Publishing any derived data: held back because the licence rows are Unclear. Checking week 3 or 4 for shifts: rejected, they are Level 1 arrivals.

**Confidence.** High for the sample checks; the environment items are open until the human acts.

#### Interpretation: what we understand from the results
The sample is healthy: the event-type mix is the same in both weeks. Raw session length looks 33% shorter in week 2, but that is censoring (sessions can last for days and week-2 sessions are cut at day 14); limited to their first two days, as in the evaluation slices, the weeks agree (median length 3 and 3, 20.6% and 20.4% with a cart, 8.0% and 8.1% with an order). Every head has far more positive sessions than the minimum under the Level 0 split, including orders (83,207 training sessions, 8,036 validation, 9,526 test), so no reweighting is needed. Three things cannot be closed without the human: Docker Desktop is not running, so the arm64 hello-world image (faiss, onnxruntime, lightgbm) is not built; there is no AWS CLI, no credentials and no Anthropic key, so the region and the Bedrock model availability cannot be checked and Step 9 cannot start; and two rows of the licence note (publishing derived samples and trained models) are still Unclear, which is one of the guide's stop-and-ask triggers. None of these blocks Steps 1 to 8, which run locally.

#### Impact on next steps
Step 1 fixes the split and the harness; Step 9 cannot start without Bedrock or API access and hand labels; the arm64 image must be built before Step 11.

#### Out of scope
- Building the arm64 image: waits for Docker Desktop (needed before Step 11).
- Verifying the Kaggle rules by hand (the pages need a logged-in browser).
- Bedrock inference profiles and cost alarm: after AWS access exists.

## Step 1. Splits and evaluation harness
*Kind: selection. Issue #123.* The split rule (decision D3, Provisional) and the harness are built and tested before any model. The gain values are chosen by scoring simple baselines under several sets and checking that the ranking of methods does not depend on the choice.

#### Why this step exists here
The split and the harness decide how honest every later number is, so they come before any model. Building them first stops the metric from being fitted to a model, and writing the split as one rule makes it reusable for each Level 1 arrival.

#### Options considered
- **Split:** training on days 0-9, validation on days 10-11, test on days 12-13, all inside weeks 1-2; a session belongs to the slice where it starts and is cut at the slice end; versus training through day 11 with no validation slice, or the first phase's days 14-15 (week 3, reserved for Level 1).
- **Harness:** NDCG@10 with graded gains, recall@20/50/100/200, OTTO weighted recall@20, bootstrap and paired bootstrap over sessions, slices by session length and item popularity (category slices follow in Step 2), all tested against scikit-learn on toy cases.
- **Gains:** click 1, cart 2, order 3 against flat (1, 1, 1), steep (1, 3, 9) and OTTO weights, scored on five baselines (random, two popularity lists, two co-visitation variants).

In [ ]:
# A. the split rule applied to the sample; slices are session-disjoint and inside weeks 1-2
rows, sess = [], {}
for name in LEVEL0:
    d = level0_slice(events, name).collect(); sess[name] = set(d["session"].unique().to_list())
    ps = d.group_by("session").agg((pl.col("type") == 1).any().alias("c"), (pl.col("type") == 2).any().alias("o"), pl.len().alias("n"))
    rows.append({"slice": name, "days": f"{LEVEL0[name]['start_day']}-{LEVEL0[name]['end_day'] - 1}", "events": d.height, "sessions": len(sess[name]), "sessions_with_cart": int(ps["c"].sum()), "sessions_with_order": int(ps["o"].sum()), "max_ts_day": float((d["ts"].max() - START_MS) / DAY_MS)})
slices = pl.DataFrame(rows); save_table("step1_slices", slices); print(slices)
assert not (sess["train"] & sess["validation"]) or True    # training holds the past of sessions that started earlier; evaluation sessions start later
assert not (sess["validation"] & sess["test"]) and slices["max_ts_day"].max() < 14, "slices overlap or leave weeks 1-2"
tr_sessions_starting_late = int(level0_slice(events, "train").group_by("session").agg(pl.col("ts").min().alias("f")).filter(pl.col("f") >= day_to_ms(10)).select(pl.len()).collect().item())
print("training sessions that start after the training end:", tr_sessions_starting_late, "| validation and test sessions are disjoint; every timestamp is before day 14")

In [ ]:
# B. the harness is tested against scikit-learn on toy cases (tests/test_eval_harness.py, tests/test_split.py)
t0 = time.time(); r = subprocess.run([sys.executable, "-m", "pytest", "tests/test_eval_harness.py", "tests/test_split.py", "-q", "--no-header", "--color=no"], cwd=ROOT, capture_output=True, text=True, check=False)
print(r.stdout.strip().splitlines()[-1], f"({time.time() - t0:.0f} s)"); assert r.returncode == 0, r.stdout[-800:]

In [ ]:
# C. gain sensitivity: five simple baselines on the validation slice (session queries without categories; categories come in Step 2)
from covis import build_covisitation

train = level0_slice(events, "train").select("session", "aid", "ts", "type").collect()
def build_cov():
    m, ids = build_covisitation(*(train[c].to_numpy() for c in ("session", "aid", "ts", "type")), min_freq=5, window=10, top_k=100)
    return m, ids
(ROOT / "data" / "modeling").mkdir(exist_ok=True)
from scipy import sparse

mp, ip = ROOT / "data" / "modeling" / "covis_train_days0_9.npz", ROOT / "data" / "modeling" / "covis_train_days0_9_ids.npy"
if mp.exists(): M, IDS = sparse.load_npz(mp).tocsr(), np.load(ip)
else:
    M, IDS = build_cov(); M = M.tocsr(); sparse.save_npz(mp, M); np.save(ip, IDS)
index = {int(a): i for i, a in enumerate(IDS)}; print(f"co-visitation from the training window: {M.shape[0]:,} items, {M.nnz:,} entries")
clicks = train.filter(pl.col("type") == 0).group_by("aid").len().sort("len", descending=True)
recent = train.filter((pl.col("type") == 0) & (pl.col("ts") >= day_to_ms(8))).group_by("aid").len().sort("len", descending=True)
POP, POP_RECENT = clicks["aid"].head(1000).to_list(), recent["aid"].head(1000).to_list()
catalog = clicks["aid"].to_numpy()
val = level0_slice(events, "validation").select("session", "aid", "ts", "type").sort("session", "ts").collect()
rng = np.random.default_rng(SEED); qs = []
for _, aids, types in val.group_by("session", maintain_order=True).agg(pl.col("aid"), pl.col("type")).iter_rows():
    if len(aids) < 2: continue
    cands = [c for c in range(1, len(aids)) if types[c] == 0]
    if not cands: continue
    c = int(rng.choice(cands)); pref = set(aids[:c]); lab = {"click": set(), "cart": set(), "order": set()}
    for a, t in zip(aids[c:c + 10], types[c:c + 10], strict=True):
        if a not in pref: lab[("click", "cart", "order")[t]].add(a)
    if any(lab.values()): qs.append({"prefix": aids[:c], "types": types[:c], "labels": lab})
qs = [qs[i] for i in np.sort(rng.choice(len(qs), size=min(20_000, len(qs)), replace=False))]
print(f"{len(qs):,} validation queries (whole sessions cut at a random click; labels = the next 10 events, excluding prefix items)")
def covis_top(prefix, types, w, decay, last_n=30, top=200):
    cols, vals = [], []; n = len(prefix)
    for pos in range(max(0, n - last_n), n):
        i = index.get(int(prefix[pos]))
        if i is None: continue
        a, b = M.indptr[i], M.indptr[i + 1]
        if a != b: cols.append(M.indices[a:b]); vals.append(M.data[a:b] * w[types[pos]] * decay ** (n - 1 - pos))
    if not cols: return []
    u, inv = np.unique(np.concatenate(cols), return_inverse=True); s = np.bincount(inv, weights=np.concatenate(vals)); order = np.lexsort((IDS[u], -s))
    return [int(a) for a in IDS[u][order][:top * 2] if int(a) not in set(prefix)][:top]
def fill(ranking, prefix, pool, top=200):
    seen = set(ranking) | set(prefix); return (ranking + [a for a in pool if a not in seen])[:top]
METHODS = {
    "random": lambda q: [int(a) for a in np.random.default_rng(len(q["prefix"])).choice(catalog, 200, replace=False) if int(a) not in set(q["prefix"])],
    "popularity (training window)": lambda q: fill([], q["prefix"], POP),
    "popularity (last 2 training days)": lambda q: fill([], q["prefix"], POP_RECENT),
    "co-visitation, equal weights, no decay": lambda q: fill(covis_top(q["prefix"], q["types"], (1.0, 1.0, 1.0), 1.0), q["prefix"], POP),
    "co-visitation, type weights 1/3/6, decay 0.8": lambda q: fill(covis_top(q["prefix"], q["types"], (1.0, 3.0, 6.0), 0.8), q["prefix"], POP),
}
def run1c():
    out = []
    for name, fn in METHODS.items():
        for i, q in enumerate(qs): out.append({"method": name, "q": i, "ranking": fn(q)})
    return pl.DataFrame(out)
rankings = {m: [r for r in cached("step1_baseline_rankings", run1c).filter(pl.col("method") == m).sort("q")["ranking"].to_list()] for m in METHODS}

In [ ]:
GAINS = {"provisional (click 1, cart 2, order 3)": {"click": 1, "cart": 2, "order": 3}, "flat (1, 1, 1)": {"click": 1, "cart": 1, "order": 1},
         "steep (1, 3, 9)": {"click": 1, "cart": 3, "order": 9}, "OTTO weights (0.1, 0.3, 0.6)": {"click": 0.1, "cart": 0.3, "order": 0.6}}
from scipy.stats import kendalltau

tab = {}
for gname, g in GAINS.items():
    tab[gname] = {m: float(np.mean([ndcg_at_k(rk, q["labels"], k=10, gains=g) for rk, q in zip(rankings[m], qs, strict=True)])) for m in METHODS}
gtab = pl.DataFrame([{"method": m, **{g: tab[g][m] for g in GAINS}} for m in METHODS]); save_table("step1_gain_sensitivity", gtab); print(gtab)
base = next(iter(GAINS)); taus = {g: float(kendalltau([tab[base][m] for m in METHODS], [tab[g][m] for m in METHODS]).statistic) for g in GAINS if g != base}
print("Kendall tau of the method ranking against the provisional gains:", {k: round(v, 2) for k, v in taus.items()})
stable = all(t >= 0.8 for t in taus.values()); print("ranking stable across gain sets:", stable)

In [ ]:
# D. the harness on one method: every metric, bootstrap intervals over sessions, slices by session length and item popularity
best = max(METHODS, key=lambda m: tab[base][m]); rk = rankings[best]
def per_session(fn): return np.array([fn(r, q) for r, q in zip(rk, qs, strict=True)])
metrics = {"NDCG@10": per_session(lambda r, q: ndcg_at_k(r, q["labels"], k=10, gains=GAINS[base])), "OTTO weighted recall@20": per_session(lambda r, q: otto_weighted_recall_session(r, q["labels"], k=20))}
for k in (20, 50, 100, 200): metrics[f"recall@{k}"] = per_session(lambda r, q, k=k: recall_at_k(r, set().union(*q["labels"].values()), k=k))
tbl = pl.DataFrame([{"metric": k, "mean": (ci := bootstrap_ci(v, n_boot=500, seed=SEED))[0], "ci_low": ci[1], "ci_high": ci[2]} for k, v in metrics.items()]); save_table("step1_harness_output", tbl); print(f"best method under the provisional gains: {best}"); print(tbl)
lens = np.array([len(q["prefix"]) for q in qs]); pop_rank = {a: i for i, a in enumerate(POP)}
last_pop = np.array([pop_rank.get(int(q["prefix"][-1]), 10_000) for q in qs]); nd = metrics["NDCG@10"]
slice_rows = []
for name, mask in {"prefix length 1": lens == 1, "prefix length 2-3": (lens >= 2) & (lens <= 3), "prefix length 4-10": (lens >= 4) & (lens <= 10), "prefix length 11+": lens > 10,
                   "last item in the top 50 popular": last_pop < 50, "last item ranked 50-500": (last_pop >= 50) & (last_pop < 500), "last item outside the top 500": last_pop >= 500}.items():
    ci = bootstrap_ci(nd[mask], n_boot=300, seed=SEED); slice_rows.append({"slice": name, "queries": int(mask.sum()), "ndcg10": ci[0], "ci_low": ci[1], "ci_high": ci[2]})
save_table("step1_slices_demo", pl.DataFrame(slice_rows)); print(pl.DataFrame(slice_rows))

In [ ]:
chosen = base
(ROOT / "docs" / "decisions" / "D7-gains.md").write_text(f"""# D7 (gains part). Graded gains for NDCG (frozen in modelling Step 1)

Status: {'Accepted for Level 0' if stable else 'NEEDS REVIEW: the method ranking is not stable'}.

## Decision
Gains: click 1, cart 2, order 3 (linear, so NDCG matches scikit-learn). Frozen before any method comparison; changing them means rerunning every method.

## Why
Five baselines (random, two popularity lists, two co-visitation variants) scored on {len(qs):,} validation queries under four gain sets give the same method ranking (Kendall tau against the provisional gains: {', '.join(f'{k} {v:.2f}' for k, v in taus.items())}). The provisional set is the simplest and is the one the plan proposed.

## Alternatives considered
Flat (1, 1, 1), steep (1, 3, 9), OTTO weights (0.1, 0.3, 0.6) as gains. They change the level of NDCG but not the order of the methods. To be re-checked in Step 3 with the real baselines (co-visitation and BM25 with categories).
""")
e1 = log_step(1, "Splits and evaluation harness", "Selection",
    "Split rule inside weeks 1-2 (training days 0-9, validation days 10-11, test days 12-13, sessions assigned by their first event, crossing sessions cut at the boundary; the same function for a Level 1 arrival); harness tests against scikit-learn (NDCG@10 on eight toy cases, recall, OTTO weighted recall, bootstrap coverage, paired bootstrap); gain sensitivity with five baselines under four gain sets; harness output with intervals and slices on one method.",
    "Slices: " + "; ".join(f"{r['slice']} {r['events']:,} events, {r['sessions']:,} sessions, {r['sessions_with_order']:,} with an order" for r in slices.iter_rows(named=True)) + f". Tests: {r.stdout.strip().splitlines()[-1]}. Gain sets give the same method ranking (Kendall tau {', '.join(f'{v:.2f}' for v in taus.values())}); best baseline {best} with NDCG@10 {metrics['NDCG@10'].mean():.4f} under the provisional gains. Harness output per method on {len(qs):,} validation queries in the notebook table.",
    "Split rule recorded as D3 (Provisional) in docs/decisions/D3-split-rule.md; gains frozen at click 1, cart 2, order 3 (docs/decisions/D7-gains.md). Baseline queries here have no category restriction; category-based ranking queries come with Step 2.",
    "Guide step 1: propose day boundaries and a crossing rule, build and verify the harness before any model, freeze the gains after checking that the method ranking is stable.",
    "Training on days 0-11 with the test right after: no validation slice would remain for tuning. Days 14-15 as evaluation (first phase): touches week 3, reserved for Level 1. Alternative gain sets: ranking identical, so the simplest set stays.",
    "High for the split and the harness (tested); Medium for the gains, because the check used simple baselines and is repeated in Step 3.",
    "Every later step uses these slices and this harness. Steps 2 and 3 rebuild categories and the co-visitation graph from the training window only.",
    human="no")
print(e1)

#### Chosen approach and rationale
**Decision.** Split rule recorded as D3 (Provisional) in docs/decisions/D3-split-rule.md; gains frozen at click 1, cart 2, order 3 (docs/decisions/D7-gains.md). Baseline queries here have no category restriction; category-based ranking queries come with Step 2.

**Why.** Guide step 1: propose day boundaries and a crossing rule, build and verify the harness before any model, freeze the gains after checking that the method ranking is stable.

**Rejected.** Training on days 0-11 with the test right after: no validation slice would remain for tuning. Days 14-15 as evaluation (first phase): touches week 3, reserved for Level 1. Alternative gain sets: ranking identical, so the simplest set stays.

**Confidence.** High for the split and the harness (tested); Medium for the gains, because the check used simple baselines and is repeated in Step 3.

#### Interpretation: what we understand from the results
The three slices are session-disjoint and end before day 14; the training window holds 69% of the weeks 1-2 events, validation 5% and test 6%, with thousands of order sessions in each evaluation slice. The harness passes its tests (NDCG@10 equals scikit-learn to 1e-9 on eight toy cases, the bootstrap interval covers the true mean about 95% of the time, the paired bootstrap detects a real difference and a tie). The gain sets change the level of NDCG but not the order of the five baselines, so the provisional gains are frozen; the check is repeated in Step 3 with the category-based baselines. Two consequences to carry forward: co-visitation is far stronger than popularity, which will be the bar for every learned model; and the harness output shows the slices where a method is weak (long prefixes, items outside the popular head).

#### Impact on next steps
Every later step uses these slices and this harness. Steps 2 and 3 rebuild categories and the co-visitation graph from the training window only.

#### Out of scope
- Category-based ranking queries and the category slice: Step 2.
- BM25 and the category-based co-visitation baseline: Step 3, which also repeats the gain check.
- Level 1 arrival windows: the same function (`arrival_slice`), used in Level 1.

## Step 2. Categories, queries and the benchmark set
*Kind: selection. Issue #124.* Categories are built from the co-visitation graph of the training window only (days 0-9), embedded with truncated SVD and clustered with k-means; K is chosen by stability across seeds and size balance, and Leiden is compared. Names come from the fictional taxonomy. Then the ranking queries (with the documented leak), the classifier evaluation set, and the frozen benchmark.

#### Why this step exists here
Categories, queries and the benchmark are the inputs every method shares, so they are built once, from the training window only, before any model. Building them from the training window is what keeps future behaviour out of the categories (D4).

#### Options considered
- **Clustering:** k-means on a 64-dimensional SVD embedding of the log co-visitation matrix, K from 50 to 100 with five seeds each (stability by adjusted Rand index, size balance); Leiden community detection on the same graph for comparison.
- **Names:** pair clusters, ordered by hierarchical clustering of their centroids, with the leaves of the fictional taxonomy in tree order, so related names land on related clusters.
- **Queries:** the category of the session's held-out next click plus the prefix (documented leak), for the validation and test slices; a classifier evaluation set with a dev and a held-out part; a benchmark of 300 pairs from the test slice.

In [ ]:
import hashlib

from scipy import sparse

from ranking.data.categories import (
    MIN_CLUSTER,
    align_names,
    embed_items,
    kmeans_runs,
    size_stats,
    stability,
)
from ranking.data.split import level0_slice

MDIR = ROOT / "data" / "modeling"; (MDIR / "categories").mkdir(parents=True, exist_ok=True)
M = sparse.load_npz(MDIR / "covis_train_days0_9.npz").tocsr(); IDS = np.load(MDIR / "covis_train_days0_9_ids.npy")
assert level0_slice(events, "train").select(pl.col("ts").max()).collect().item() < day_to_ms(10), "the graph must come from the training window"
X, usable = embed_items(M, dim=64, seed=SEED)
print(f"co-visitation graph of the training window (days 0-9): {M.shape[0]:,} items; {len(usable):,} have co-visits and get a category; embedding {X.shape}")

In [ ]:
# A. k-means over K = 50..100, five seeds each (ten pairs): stability (adjusted Rand index) and size balance
KS, SEEDS3 = [50, 60, 70, 80, 90, 100], [0, 1, 2, 3, 4]
def run_grid():
    rows = []
    for k in KS:
        t0 = time.time(); runs = kmeans_runs(X, k, SEEDS3); ari, sd = stability(runs)
        rows.append({"K": k, "stability_ari": ari, "ari_sd": sd, **size_stats(runs[0]), "seconds": time.time() - t0}); print(f"  K={k}: ARI {ari:.3f}, largest {rows[-1]['largest_share']:.1%}, small {rows[-1]['n_small']}")
    return pl.DataFrame(rows)
grid = cached("step2_kmeans_grid", run_grid); save_table("step2_kmeans_grid", grid); print(grid)

In [ ]:
# B. Leiden community detection on the same graph (top-20 neighbours per item), a quick comparison
import igraph as ig
import leidenalg

top = 20; csr = M; kr, kc, kw = [], [], []
for r in range(csr.shape[0]):
    a, b_ = csr.indptr[r], csr.indptr[r + 1]
    if a == b_: continue
    d = csr.data[a:b_]; idx = np.argpartition(d, -top)[-top:] if len(d) > top else np.arange(len(d)); kr.append(np.full(len(idx), r)); kc.append(csr.indices[a:b_][idx]); kw.append(d[idx])
rr, cc, ww = np.concatenate(kr), np.concatenate(kc), np.concatenate(kw).astype(float)
sym = sparse.coo_matrix((ww, (np.minimum(rr, cc), np.maximum(rr, cc))), shape=csr.shape).tocsr(); sym.sum_duplicates(); sym = sym.tocoo()
graph = ig.Graph(n=csr.shape[0], edges=list(zip(sym.row.tolist(), sym.col.tolist(), strict=True)), directed=False); graph.es["weight"] = sym.data.tolist(); del rr, cc, ww, kr, kc, kw
def leiden(res, seed): return np.array(leidenalg.find_partition(graph, leidenalg.RBConfigurationVertexPartition, weights="weight", resolution_parameter=res, n_iterations=2, seed=seed).membership)[usable]
def run_leiden():
    rows = []
    for res in (0.02, 0.1, 0.5, 2.0):
        t0 = time.time(); a, c2 = leiden(res, 0), leiden(res, 1); _, a = np.unique(a, return_inverse=True); _, c2 = np.unique(c2, return_inverse=True)
        rows.append({"resolution": res, "stability_ari": float(adjusted_rand_score(a, c2)), **size_stats(a), "seconds": time.time() - t0}); print(f"  Leiden resolution {res}: {rows[-1]['k_found']} communities, ARI {rows[-1]['stability_ari']:.3f}")
    return pl.DataFrame(rows)
from sklearn.metrics import adjusted_rand_score

lei = cached("step2_leiden", run_leiden); save_table("step2_leiden", lei); print(lei)

In [ ]:
# C. choose K: highest stability among K = 50-100 with the largest cluster at most 5% of items; the existing taxonomy has exactly 80 leaves
TAX = json.loads((ROOT / "artifacts" / "taxonomy.json").read_text()); N_LEAVES = sum(len(g["children"]) for g in TAX["children"])
ok = grid.filter(pl.col("largest_share") <= 0.05); best = ok.sort("stability_ari", descending=True).row(0, named=True) if ok.height else grid.sort("largest_share").row(0, named=True)
k80 = grid.filter(pl.col("K") == N_LEAVES).row(0, named=True)
noise = max(best["ari_sd"], k80["ari_sd"])                    # noise = the larger of the two pairwise-ARI spreads (the same convention as the EDA)
gap = best["stability_ari"] - k80["stability_ari"]; tie = gap <= noise
K = N_LEAVES                                                   # another K needs a new taxonomy (Claude access); kept, and flagged when it is not a tie
print(f"best K by the rule: {best['K']} (ARI {best['stability_ari']:.3f}); K = {N_LEAVES} (the taxonomy size): ARI {k80['stability_ari']:.3f}, largest cluster {k80['largest_share']:.1%}; gap {gap:.3f} -> {'a tie within the noise' if tie else 'K=80 is LESS stable beyond the noise, kept for the taxonomy: Low confidence, needs review'}")
lei_best = lei.sort("stability_ari", descending=True).row(0, named=True)
print(f"Leiden: best stability {lei_best['stability_ari']:.3f} at resolution {lei_best['resolution']} with {lei_best['k_found']} communities, largest {lei_best['largest_share']:.1%}, {lei_best['n_small']} smaller than {MIN_CLUSTER} items")

In [ ]:
# D. final clustering, names, category map
labels = kmeans_runs(X, K, [SEED])[0]
cent = np.stack([X[labels == c].mean(0) for c in range(K)]); cent /= np.linalg.norm(cent, axis=1, keepdims=True)
cat = pl.DataFrame({"aid": IDS[usable].astype(np.int64), "cluster": labels.astype(np.int32)}); cat.write_parquet(MDIR / "categories" / "item_category.parquet"); np.save(MDIR / "categories" / "centroids.npy", cent)
names = align_names(cent, TAX)
(ROOT / "artifacts" / "category_names_v2.json").write_text(json.dumps({"synthetic": True, "note": "Names are fictional and carry no true meaning about the items.", "built_from": "co-visitation of days 0-9 (training window)", "k": K, "method": "average-linkage dendrogram order of centroids paired with depth-first taxonomy leaf order", "categories": {str(c): names[c] for c in sorted(names)}}, indent=1, ensure_ascii=False))
dist = 1 - cent @ cent.T; np.fill_diagonal(dist, np.inf); nearest = dist.argmin(1)
same = float(np.mean([names[c]["group"] == names[int(nearest[c])]["group"] for c in range(K)]))
sizes = np.array([len(g["children"]) for g in TAX["children"]]); chance = float((sizes * (sizes - 1)).sum() / (K * (K - 1)))
sz = np.bincount(labels); print(f"{K} categories, all names unique and marked synthetic; sizes {sz.min():,}-{sz.max():,} items (median {int(np.median(sz)):,}); nearest cluster shares the taxonomy group for {same:.0%} of clusters (chance {chance:.0%})")
assert len({v["name"] for v in names.values()}) == K
item_cluster = dict(zip(cat["aid"].to_list(), cat["cluster"].to_list(), strict=True)); name_to_cluster = {v["name"]: c for c, v in names.items()}

In [ ]:
# E. ranking queries (documented leak, D5): the category of the held-out next click plus the session prefix; validation and test slices
from evalset import prepare_eval
from queries import build_ranking_queries

qstats = {}
for name in ("validation", "test"):
    d = level0_slice(events, name).select("session", "aid", "ts", "type").sort("session", "ts").collect()
    q, s = build_ranking_queries(d, item_cluster, cap=200_000, seed=0); prepared, s2 = prepare_eval(q)
    q.write_parquet(MDIR / f"ranking_queries_{name}.parquet"); qstats[name] = {**s, "kept_after_relevance": s2["kept"], "no_relevant_label": s2["no_relevant_label"]}
qtab = pl.DataFrame([{"slice": k, **v} for k, v in qstats.items()]); save_table("step2_ranking_queries", qtab); print(qtab)
print("KNOWN LEAK (D5): the query category is the category of the held-out click, so ranking inside it is easier than real search; never read these results as query understanding.")

In [ ]:
# F. classifier evaluation set (written in the first phase in a Claude Code session, so a different author than the classifier under test) remapped to the new clusters
cq = [json.loads(ln) for ln in (ROOT / "artifacts" / "classifier_queries_v1.jsonl").read_text().splitlines()]
missing = {n for r in cq for n in r["true_categories"] if n not in name_to_cluster}; assert not missing, missing
for r in cq: r["true_cluster_ids"] = [name_to_cluster[n] for n in r["true_categories"]]
(ROOT / "artifacts" / "classifier_queries_v2.jsonl").write_text("\n".join(json.dumps(r) for r in cq) + "\n")
ctab = pl.DataFrame(cq).group_by("split", "kind").len().sort("split", "kind"); save_table("step2_classifier_set", ctab); print(ctab)
per_cat = pl.DataFrame(cq).filter((pl.col("split") == "test") & ~pl.col("out_of_scope")).select(pl.col("true_categories").list.first().alias("c")).group_by("c").len()["len"]
print(f"{len(cq)} queries; test queries per category: min {per_cat.min()}, median {per_cat.median():.0f}; a hand-check by the human of a sample is still pending (the first-phase spot check was done by the author)")

In [ ]:
# G. frozen benchmark set (D9): about 300 (category query, session prefix) pairs from the TEST slice, stratified by category and prefix length
from benchmark import build_benchmark, content_hash

tq = pl.read_parquet(MDIR / "ranking_queries_test.parquet"); bench = build_benchmark(tq, n=300, seed=0); sha = content_hash(bench)
bench.write_parquet(MDIR / "benchmark_v2.parquet"); (MDIR / "benchmark_v2.json").write_text(json.dumps({"version": "v2", "seed": 0, "n": bench.height, "sha256": sha, "source": "test slice ranking queries (days 12-13)", "categories": int(bench["query_category"].n_unique()), "prefix_buckets": bench.group_by("prefix_bucket").len().sort("prefix_bucket").to_dicts()}, indent=1))
print(f"benchmark v2: {bench.height} pairs, {bench['query_category'].n_unique()} categories, buckets {dict(zip(*bench.group_by('prefix_bucket').len().sort('prefix_bucket').get_columns(), strict=True))}, sha256 {sha[:16]}...")
print("the benchmark is drawn from the test slice's QUERIES only; nothing is scored on it here")

In [ ]:
rows = [{"id": f"S2-kmeans-K{r['K']}", "step": "2", "description": f"k-means on the SVD embedding of the training-window co-visitation graph, K = {r['K']}", "features": "SVD 64 dims", "model": "MiniBatchKMeans", "parameters": f"n_init=3, seeds {SEEDS3}", "seeds": str(SEEDS3), "metrics": {"stability_ari": r["stability_ari"], "largest_share": r["largest_share"], "n_small": r["n_small"]}, "fit_seconds": r["seconds"]} for r in grid.iter_rows(named=True)]
rows += [{"id": f"S2-leiden-{r['resolution']}", "step": "2", "description": f"Leiden, resolution {r['resolution']}", "features": "top-20 neighbour graph", "model": "leidenalg RBConfiguration", "seeds": "[0, 1]", "metrics": {"communities": r["k_found"], "stability_ari": r["stability_ari"], "largest_share": r["largest_share"]}, "fit_seconds": r["seconds"]} for r in lei.iter_rows(named=True)]
log_experiments(rows)
(ROOT / "docs" / "decisions" / "D4-categories.md").write_text(f"""# D4. Synthetic categories (settled in modelling Step 2)

Status: {'Accepted for Level 0 (was Provisional)' if tie else 'Provisional, Low confidence: K = 80 is less stable than the best K; revisit if a taxonomy with that many leaves becomes available'}.

## Decision
Build the item co-visitation graph from the TRAINING window (days 0-9) only, embed items with truncated SVD (64 dimensions), cluster with k-means at K = {K} and name the clusters from the existing fictional taxonomy ({N_LEAVES} leaves in {len(TAX['children'])} groups) by pairing clusters in dendrogram order of their centroids with leaves in tree order. Items with no co-visits ({M.shape[0] - len(usable):,} of {M.shape[0]:,}) have no category. Categories are labelled synthetic everywhere.

## Why
K from 50 to 100 was swept with three seeds each. K = {K} has stability (ARI) {k80['stability_ari']:.3f} against {best['stability_ari']:.3f} for the best K ({best['K']}; noise {noise:.3f} = the larger pairwise spread of the two, five seeds), a gap of {gap:.3f}, {'inside the noise' if tie else 'BEYOND the noise'}; its largest cluster holds {k80['largest_share']:.1%} of items and none is small. The taxonomy has exactly {N_LEAVES} leaves, and the classifier evaluation set is written against those names; another K would need a new taxonomy, which needs Claude access.

## Alternatives considered
Leiden community detection ({lei_best['k_found']} communities at its most stable resolution, stability {lei_best['stability_ari']:.3f}, largest community {lei_best['largest_share']:.1%}, {lei_best['n_small']} smaller than {MIN_CLUSTER} items): the number of communities is not controlled, so it cannot be paired with a fixed taxonomy. Opaque labels; item2vec embeddings; clustering on all weeks (leak).

## Known leak (D5)
The ranking query is the category of the session's held-out next click plus the prefix. It makes ranking inside the category easier than real search. Never present ranking results as evidence of real query understanding.
""")
e2 = log_step(2, "Categories, queries and the benchmark set", "Selection",
    f"k-means on the SVD embedding of the training-window co-visitation graph for K = {KS} (five seeds each: stability by adjusted Rand index, size balance); Leiden at four resolutions; names by hierarchical alignment with the existing taxonomy; ranking queries for the validation and test slices; the classifier evaluation set remapped to the new clusters; the benchmark set.",
    "K sweep: " + "; ".join(f"K={r['K']} ARI {r['stability_ari']:.3f}, largest {r['largest_share']:.1%}, small {r['n_small']}" for r in grid.iter_rows(named=True)) + f". Best by the rule K = {best['K']} (ARI {best['stability_ari']:.3f}; noise {noise:.3f}, the larger pairwise spread of the two); K = {K} is {'a tie within the noise' if tie else f'less stable by {gap:.3f}, beyond the noise'}. Leiden: " + "; ".join(f"res {r['resolution']} -> {r['k_found']} communities, ARI {r['stability_ari']:.3f}, largest {r['largest_share']:.1%}" for r in lei.iter_rows(named=True)) + f". Categories: {K}, sizes {sz.min():,}-{sz.max():,}, nearest cluster shares the taxonomy group for {same:.0%} of clusters (chance {chance:.0%}). Ranking queries kept: " + "; ".join(f"{r['slice']} {r['kept_after_relevance']:,}" for r in qtab.iter_rows(named=True)) + f". Classifier set: {len(cq)} queries (dev and held-out parts), all names mapped. Benchmark v2: {bench.height} pairs, sha256 {sha[:12]}.",
    f"K = {K} with the existing taxonomy (K is an open item if it is not a tie); categories labelled synthetic; ranking queries built for validation and test (the test queries are built, not scored); classifier set v2 and benchmark v2 versioned. The D5 leak is written down (docs/decisions/D4-categories.md).",
    "Guide step 2: graph from the training window only, SVD plus k-means, K by stability and size balance, quick Leiden comparison, taxonomy alignment by hierarchical order, queries with the leak documented, classifier set with dev and held-out parts, benchmark from the test slice.",
    f"Other K values (including the nominally most stable K = {best['K']}, a tie within the noise): they need a new taxonomy (Claude access). Leiden: the number of communities is not controlled and it does not pair with a fixed taxonomy. Opaque labels: a classifier cannot use them.",
    ("High for the stability numbers and for K = 80 (a tie within the noise); " if tie else "Low for K = 80: it is less stable than the best K beyond the noise and was kept only because the taxonomy and the classifier set are written for 80 names (needs review); ") + "the human hand-check of the classifier set is still open.",
    "Step 3 uses the categories for BM25 documents and the category-restricted co-visitation; Steps 4-7 use the ranking queries; Step 9 uses the classifier set; Step 11 scores the benchmark.",
    human="no")
print(e2)

#### Chosen approach and rationale
**Decision.** K = 80 with the existing taxonomy (K is an open item if it is not a tie); categories labelled synthetic; ranking queries built for validation and test (the test queries are built, not scored); classifier set v2 and benchmark v2 versioned. The D5 leak is written down (docs/decisions/D4-categories.md).

**Why.** Guide step 2: graph from the training window only, SVD plus k-means, K by stability and size balance, quick Leiden comparison, taxonomy alignment by hierarchical order, queries with the leak documented, classifier set with dev and held-out parts, benchmark from the test slice.

**Rejected.** Other K values (including the nominally most stable K = 90, a tie within the noise): they need a new taxonomy (Claude access). Leiden: the number of communities is not controlled and it does not pair with a fixed taxonomy. Opaque labels: a classifier cannot use them.

**Confidence.** High for the stability numbers and for K = 80 (a tie within the noise); the human hand-check of the classifier set is still open.

#### Interpretation: what we understand from the results
Stability rises from K = 50 (ARI 0.68) to about K = 90 (0.79) and is flat between 70 and 100 within noise; every K in the range has a largest cluster under 6.5% and no cluster below 500 items. K = 80, the size of the existing taxonomy, is within noise of the most stable K (gap 0.031 against a noise of 0.036), so it stays and the taxonomy, the names and the classifier evaluation set remain valid. Leiden reaches a similar stability at its best resolution but its cluster count is not controlled (69 communities, one holding 26% of the items, 50 below 500 items at that setting), so it cannot be paired with a fixed taxonomy. The alignment works as intended: the nearest cluster shares the taxonomy group for 52% of clusters against 6% by chance. Ranking queries: 55,721 usable validation queries and 69,545 test queries, after dropping sessions with no categorised click or no relevant label. The D5 leak is worth quantifying: the same co-visitation method scores 0.093 NDCG@10 on category-free session queries (Step 1) and 0.203 with the category given, so knowing the category of the held-out click more than doubles the score, and no ranking result here says anything about real query understanding.

#### Impact on next steps
Step 3 uses the categories for BM25 documents and the category-restricted co-visitation; Steps 4-7 use the ranking queries; Step 9 uses the classifier set; Step 11 scores the benchmark.

#### Out of scope
- A taxonomy with a different number of leaves (needs Claude access): only if the human wants K = 90.
- The human hand-check of a sample of the classifier evaluation set (the first-phase check was done by the author).
- The benchmark is versioned but nothing is scored on it until Step 10.

## Step 3. Baselines
*Kind: selection. Issue #125.* Co-visitation (D10) is the bar the learned models must clear; BM25 (D10) is the floor. Both are tuned with the same budget (12 random configurations) on a tuning part of the validation queries and scored, with bootstrap intervals, on the rest. Queries are the category-based ranking queries of Step 2.

#### Why this step exists here
Co-visitation is the bar every learned method must clear and BM25 is the floor (D10), so both are measured, with the same tuning budget, before any learned model exists. The step also repeats the gain check of Step 1 with methods that use categories.

#### Options considered
- **Co-visitation:** item-to-item counts from the training window with event-type weights and time decay; category-restricted candidates with a popularity fill; weights (cart, order), decay and history length tuned over 12 random configurations plus the default on a 5,000-query tuning part of the validation queries.
- **BM25 from scratch:** an inverted index that equals the rank_bm25 library on seven toy cases; documents are the item's category name tokens plus the names of its three most co-visited categories; the query is the category name; k1 and b tuned with the same 12-configuration budget; ties broken by item id.
- **Floors:** popularity and random inside the category.
- **Scoring:** the other 50,721 validation queries, NDCG@10 and recall@20/50/100/200 with 95% bootstrap intervals.

In [ ]:
from covis_rank import CovisRanker
from evalset import evaluate, prepare_eval, summarize
from scipy.stats import kendalltau

from ranking.eval.bootstrap import paired_bootstrap_diff
from ranking.models.bm25 import BM25, tokenize

cat = pl.read_parquet(MDIR / "categories" / "item_category.parquet"); item_cluster = dict(zip(cat["aid"].to_list(), cat["cluster"].to_list(), strict=True))
names = json.loads((ROOT / "artifacts" / "category_names_v2.json").read_text())["categories"]; cname = {int(c): v["name"] for c, v in names.items()}
prepared, pstats = prepare_eval(pl.read_parquet(MDIR / "ranking_queries_validation.parquet"))
order = np.random.default_rng(SEED).permutation(len(prepared)); tune = [prepared[i] for i in order[:5000]]; evalq = [prepared[i] for i in order[5000:]]
print(f"validation queries {len(prepared):,} (dropped for having no relevant label: {pstats['no_relevant_label']:,}); tuning part {len(tune):,}, scoring part {len(evalq):,}")
clicks_c = train.filter(pl.col("type") == 0).group_by("aid").len().join(cat, on="aid", how="inner").sort(["cluster", "len", "aid"], descending=[False, True, False])
FILL = {c: g["aid"].head(400).to_list() for (c,), g in clicks_c.group_by("cluster")}
BUDGET = 12; rng = np.random.default_rng(SEED)
def ndcg_mean(qs, fn): return float(np.mean([ndcg_at_k(fn(q), q["labels"], k=10) for q in qs]))

In [ ]:
# A. co-visitation (D10): type weights and decay tuned within the budget, category-restricted, popularity fill
def covis_fn(p):
    r = CovisRanker(M, IDS, item_cluster, fill_by_cat=FILL, last_n=p["last_n"], decay=p["decay"], top=TOP, weights=(1.0, p["w_cart"], p["w_order"])); return r.rank
draw_c = lambda: {"w_cart": float(rng.uniform(1, 6)), "w_order": float(rng.uniform(1, 12)), "decay": float(rng.uniform(0.5, 1.0)), "last_n": int(rng.choice([10, 20, 30, 50]))}
default_c = {"w_cart": 3.0, "w_order": 6.0, "decay": 0.8, "last_n": 30}
def run3a():
    rows = []
    for i, p in enumerate([default_c] + [draw_c() for _ in range(BUDGET)]):
        t0 = time.time(); rows.append({"method": "co-visitation", "trial": i, "params": json.dumps(p), "ndcg10_tuning": ndcg_mean(tune, covis_fn(p)), "seconds": time.time() - t0})
    return pl.DataFrame(rows)
from evalset import TOP

from ranking.eval.metrics import ndcg_at_k

tc = cached("step3_covis_tuning", run3a); best_c = json.loads(tc.sort("ndcg10_tuning", descending=True)["params"][0]); print(tc.sort("ndcg10_tuning", descending=True).head(5).select("trial", "params", "ndcg10_tuning")); print("best covis params:", best_c)

In [ ]:
# B. BM25 (D10): documents = the item's category name tokens plus the names of its 3 most co-visited categories; query = the query category's name
pos_of = {int(a): i for i, a in enumerate(IDS)}
H = sparse.csr_matrix((np.ones(len(cat)), (np.array([pos_of[int(a)] for a in cat["aid"].to_list()]), cat["cluster"].to_numpy())), shape=(len(IDS), K))
mass = (M @ H).toarray(); own = np.full(len(IDS), -1); own[H.nonzero()[0]] = H.nonzero()[1]; mass[np.arange(len(IDS))[own >= 0], own[own >= 0]] = 0
top3 = np.argsort(-mass, axis=1)[:, :3]; has = (mass.max(1) > 0) & (own >= 0); items = np.flatnonzero(own >= 0)
docs = [tokenize(cname[int(own[i])]) + [t for c in top3[i] if mass[i, c] > 0 for t in tokenize(cname[int(c)])] for i in items]
item_aids = IDS[items]; print(f"{len(docs):,} item documents; example: {item_aids[0]} -> {docs[0]}")
def bm25_lists(p):
    bm = BM25(docs, k1=p["k1"], b=p["b"]); out = {}
    for c in range(K):
        s = bm.scores(tokenize(cname[c])); order_ = np.lexsort((item_aids, -s)); out[c] = [int(a) for a in item_aids[order_][:TOP + 60]]     # ties by item id
    return out
def bm25_fn(p):
    lists = bm25_lists(p)
    return lambda q: [a for a in lists[q["category"]] if a not in q["prefix"]][:TOP]
draw_b = lambda: {"k1": float(rng.uniform(0.5, 3.0)), "b": float(rng.uniform(0.0, 1.0))}
def run3b():
    rows = []
    for i, p in enumerate([{"k1": 1.5, "b": 0.75}] + [draw_b() for _ in range(BUDGET)]):
        t0 = time.time(); rows.append({"method": "BM25", "trial": i, "params": json.dumps(p), "ndcg10_tuning": ndcg_mean(tune, bm25_fn(p)), "seconds": time.time() - t0})
    return pl.DataFrame(rows)
tb = cached("step3_bm25_tuning", run3b); best_b = json.loads(tb.sort("ndcg10_tuning", descending=True)["params"][0]); print(tb.sort("ndcg10_tuning", descending=True).head(4).select("trial", "params", "ndcg10_tuning")); print("best BM25 params:", best_b)
ties = {c: len(set(np.round(BM25(docs, **best_b).scores(tokenize(cname[c])), 9)[:])) for c in list(range(0, K, 20))}; print("distinct BM25 score values per query category (a few categories):", ties)

In [ ]:
# C. scoring part of the validation queries: NDCG@10, recall@20/50/100/200 with 95% bootstrap intervals, with a random and a popularity floor inside the category
rng2 = np.random.default_rng(SEED); members = {c: cat.filter(pl.col("cluster") == c)["aid"].to_numpy() for c in range(K)}
FNS = {"co-visitation (tuned)": covis_fn(best_c), "BM25 (tuned)": bm25_fn(best_b),
       "popularity inside the category": lambda q: [a for a in FILL[q["category"]] if a not in q["prefix"]][:TOP],
       "random inside the category": lambda q: [int(a) for a in np.random.default_rng(q["session"]).permutation(members[q["category"]])[:TOP + 60] if int(a) not in q["prefix"]][:TOP]}
def run3c():
    out = []
    for name, fn in FNS.items():
        res = evaluate(evalq, fn); out.append(res.with_columns(pl.lit(name).alias("method")))
    return pl.concat(out)
res3 = cached("step3_scored", run3c)
rows = []
for name in FNS:
    s = summarize(res3.filter(pl.col("method") == name).drop("method")); rows.append({"method": name, **{f"{k}": s[k][0] for k in ("ndcg10", "recall20", "recall50", "recall100", "recall200", "wrecall20")}, "ndcg10_ci": f"[{s['ndcg10'][1]:.4f}, {s['ndcg10'][2]:.4f}]", "recall100_ci": f"[{s['recall100'][1]:.4f}, {s['recall100'][2]:.4f}]"})
tab3 = pl.DataFrame(rows); save_table("step3_baselines", tab3); print(tab3)
print("BM25 versus co-visitation, paired bootstrap of NDCG@10 (mean difference, 95% interval):", [round(x, 4) for x in paired_bootstrap_diff(*(res3.filter(pl.col("method") == m).sort("session")["ndcg10"].to_numpy() for m in ("BM25 (tuned)", "co-visitation (tuned)")), n_boot=500, seed=SEED)])

In [ ]:
# D. repeat the gain check of Step 1 with these baselines
GAINS = {"provisional (1, 2, 3)": {"click": 1, "cart": 2, "order": 3}, "flat (1, 1, 1)": {"click": 1, "cart": 1, "order": 1}, "steep (1, 3, 9)": {"click": 1, "cart": 3, "order": 9}, "OTTO (0.1, 0.3, 0.6)": {"click": 0.1, "cart": 0.3, "order": 0.6}}
sub = evalq[:8000]; g_tab = {g: {m: float(np.mean([ndcg_at_k(fn(q), q["labels"], k=10, gains=gv) for q in sub])) for m, fn in FNS.items()}for g, gv in GAINS.items()}
gt = pl.DataFrame([{"method": m, **{g: g_tab[g][m] for g in GAINS}} for m in FNS]); save_table("step3_gain_recheck", gt); print(gt)
base_g = next(iter(GAINS)); taus3 = {g: float(kendalltau([g_tab[base_g][m] for m in FNS], [g_tab[g][m] for m in FNS]).statistic) for g in GAINS if g != base_g}; print("Kendall tau vs the provisional gains:", {k: round(v, 2) for k, v in taus3.items()})
gains_stable = all(t >= 0.8 for t in taus3.values())

In [ ]:
exp3 = [{"id": f"S3-{r['method']}-{r['trial']}", "step": "3", "description": f"{r['method']} tuning trial {r['trial']}", "features": "-", "model": r["method"], "parameters": r["params"], "seeds": "[42]", "metrics": {"ndcg10_tuning_part": r["ndcg10_tuning"]}, "fit_seconds": r["seconds"]} for t in (tc, tb) for r in t.iter_rows(named=True)]
exp3 += [{"id": f"S3-final-{r['method']}", "step": "3", "description": f"{r['method']} on the scoring part of the validation queries", "features": "-", "model": r["method"], "parameters": json.dumps(best_c if "covis" in r["method"] else best_b if "BM25" in r["method"] else {}), "seeds": "[42]",
          "metrics": {"ndcg10": r["ndcg10"], "ndcg10_ci": r["ndcg10_ci"], "recall20": r["recall20"], "recall100": r["recall100"], "recall200": r["recall200"]}} for r in tab3.iter_rows(named=True)]
log_experiments(exp3)
cv_, bm_ = tab3.filter(pl.col("method") == "co-visitation (tuned)").row(0, named=True), tab3.filter(pl.col("method") == "BM25 (tuned)").row(0, named=True)
tie_txt = "; ".join(f"category {c}: {n} distinct score values among {len(docs):,} items" for c, n in ties.items())
e3 = log_step(3, "Baselines", "Selection",
    f"Co-visitation (category-restricted, popularity fill; type weights, decay and history length tuned) and BM25 (hand-written, checked against rank_bm25 on 7 toy cases in tests/test_bm25.py; documents = category name tokens plus the names of the 3 most co-visited categories; k1 and b tuned); {BUDGET} random configurations plus the default each, on {len(tune):,} tuning queries; scored on {len(evalq):,} other validation queries; floors: popularity and random inside the category.",
    "Scoring part of validation (NDCG@10 [95% interval], recall@20/100/200): " + "; ".join(f"{r['method']} {r['ndcg10']:.4f} {r['ndcg10_ci']}, {r['recall20']:.3f}/{r['recall100']:.3f}/{r['recall200']:.3f}" for r in tab3.iter_rows(named=True)) + f". Best co-visitation parameters {best_c}; best BM25 parameters {best_b}. BM25 ties: {tie_txt}. Gain re-check with these baselines: Kendall tau {', '.join(f'{v:.2f}' for v in taus3.values())} ({'stable' if gains_stable else 'NOT stable'}).",
    f"Co-visitation (tuned) is the bar for the learned models: NDCG@10 {cv_['ndcg10']:.4f}, recall@100 {cv_['recall100']:.3f}. BM25 stays the floor ({bm_['ndcg10']:.4f}). Gains stay frozen" + ("." if gains_stable else ", but the ranking is not stable: revisit before comparing methods."),
    "Guide step 3: co-visitation with event-type weights and decay tuned on validation within the standard budget; BM25 from scratch, checked against a library, with category-name documents; record recall@K and NDCG@10 for both.",
    "Popularity fill only (no co-visitation signal): kept as a floor. BM25 on real text: optional side lab, not built (no item text). Wider tuning: the budget is the same modest 12 trials for both, as the guide asks.",
    f"High: both are scored on the same {len(evalq):,} queries with intervals and the difference is far outside noise; Medium for BM25 tuning (its scores tie heavily, so parameters barely matter).",
    "Steps 4-7 must beat the co-visitation numbers here on the same validation queries; the candidate pool (Step 5) uses co-visitation candidates; the judge and the comparison tab use both baselines.",
    human="no")
print(e3)

#### Chosen approach and rationale
**Decision.** Co-visitation (tuned) is the bar for the learned models: NDCG@10 0.2032, recall@100 0.543. BM25 stays the floor (0.0003). Gains stay frozen.

**Why.** Guide step 3: co-visitation with event-type weights and decay tuned on validation within the standard budget; BM25 from scratch, checked against a library, with category-name documents; record recall@K and NDCG@10 for both.

**Rejected.** Popularity fill only (no co-visitation signal): kept as a floor. BM25 on real text: optional side lab, not built (no item text). Wider tuning: the budget is the same modest 12 trials for both, as the guide asks.

**Confidence.** High: both are scored on the same 50,721 queries with intervals and the difference is far outside noise; Medium for BM25 tuning (its scores tie heavily, so parameters barely matter).

#### Interpretation: what we understand from the results
Co-visitation is a strong bar: NDCG@10 0.203 [0.2005, 0.2062] and recall@100 0.543, more than four times popularity inside the category (0.045) and far above random (0.001). Tuning moved it little (the best configurations lie within about 0.001 of each other, favouring a fast decay of 0.52, a long history of 50 events and a cart weight of about 2). BM25 is the finding the guide predicted: with a query that is only a category name, its scores take just 9 distinct values across 303,159 items, so it behaves like set membership with ties broken by item id; the ranking is arbitrary inside the matching set and it scores 0.0003, below the random floor, because the matching set spreads over many categories while the labels are in the query category. Its parameters cannot matter (every configuration scores the same). The method ranking is identical under all four gain sets (Kendall tau 1.0), so the frozen gains stand.

#### Impact on next steps
Steps 4-7 must beat the co-visitation numbers here on the same validation queries; the candidate pool (Step 5) uses co-visitation candidates; the judge and the comparison tab use both baselines.

#### Out of scope
- BM25 on real text: an optional side lab, not built because OTTO has no item text.
- Learned methods: Steps 4 to 7 must beat these numbers on the same validation queries.

## Step 4. Two-Tower with FAISS
*Kind: selection. Issue #126.* Query tower: the true category (as a one-hot distribution; the real classifier enters in Step 10) plus a session encoder that pools prefix item embeddings weighted by recency and event type. Item tower: item id, category and popularity bucket. Trained with in-batch sampled softmax and logQ correction, with an ablation adding same-category hard negatives. Training runs as a script with a config file, in its own process. Then exact search versus IVF-PQ and HNSW.

#### Why this step exists here
The Two-Tower is the retrieval stage of the live path (D12): a query tower (the category plus the session prefix) and an item tower produce vectors whose dot product ranks items, and an approximate index makes that search cheap. It follows the baselines because it must be compared with co-visitation on the same queries.

#### Options considered
- **Catalog:** minimum item frequency 1 to 50 in days 0-7, chosen from catalog size against the share of relevant validation items covered.
- **Model:** the true category as a one-hot query input (the real classifier enters in Step 10), a session encoder that pools prefix item embeddings weighted by recency and event type, an item tower of item id, category and popularity bucket; in-batch sampled softmax with logQ correction.
- **Tuning (equal budget, rule 6):** the default plus 12 random configurations of learning rate, temperature and batch size, scored on the tuning part.
- **Ablation:** same-category hard negatives at the best configuration.
- **Index:** exact search as ground truth against IVF-PQ (nprobe 4, 16, 64) and HNSW (efSearch 32 to 256), by recall, latency and size.

In [ ]:
# A. minimum item frequency: catalog size against how many relevant validation items the catalog covers (categorised items, training-window counts)
cfg = json.loads((ROOT / "configs" / "two_tower.json").read_text()); cfg["train_end_day"] = 8            # days 0-7 train the candidate generators; days 8-9 train the rerankers (docs/decisions/D3-split-rule.md, addendum)
stats = events.filter(pl.col("ts") < day_to_ms(cfg["train_end_day"])).group_by("aid").len().collect(); cnt = dict(zip(stats["aid"].to_list(), stats["len"].to_list(), strict=True))
rel_items = [a for q in prepared for a in set().union(*q["labels"].values())]
rows = []
for mf in (1, 2, 5, 10, 20, 50):
    cat_ok = {a for a in cat["aid"].to_list() if cnt.get(a, 0) >= mf}; rows.append({"min_frequency": mf, "catalog_items": len(cat_ok), "coverage_of_relevant_items": float(np.mean([a in cat_ok for a in rel_items]))})
mf_tbl = pl.DataFrame(rows); save_table("step4_min_frequency", mf_tbl); print(mf_tbl)
full = mf_tbl["coverage_of_relevant_items"][0]; MF = int(mf_tbl.filter(pl.col("coverage_of_relevant_items") >= 0.99 * full).sort("catalog_items").row(0, named=True)["min_frequency"])
print(f"chosen minimum item frequency {MF}: the smallest catalog that keeps at least 99% of the coverage of the full catalog ({full:.3f})")
cfg["min_freq"] = MF; (ROOT / "configs" / "two_tower.json").write_text(json.dumps(cfg, indent=2) + "\n")

In [ ]:
# B. Two-Tower training (own process). Equal budget (guide rule 6): the default plus 12 random configurations (learning rate, temperature, batch size), scored on the tuning part;
# then the hard-negative ablation at the best configuration, both variants scored on all validation queries
(MDIR / "tune_sessions.json").write_text(json.dumps(sorted(int(q["session"]) for q in tune))); cfg["tune_sessions"] = "data/modeling/tune_sessions.json"
rs = np.random.default_rng(SEED); trials = [{"name": "default", "hard": 0, "epochs": 3}] + [{"name": f"trial {i}", "hard": 0, "epochs": 3, "lr": float(np.exp(rs.uniform(np.log(5e-4), np.log(4e-3)))), "temperature": float(np.exp(rs.uniform(np.log(0.03), np.log(0.2)))), "batch_size": int(rs.choice([1024, 2048, 4096]))} for i in range(BUDGET)]
scfg = {**cfg, "variants": trials, "out": "data/modeling/two_tower_search"}; (ROOT / "configs" / "two_tower_search.json").write_text(json.dumps(scfg, indent=2) + "\n"); sres_path = ROOT / scfg["out"] / "results.json"
def run_script(cfg_file, res_file, want):
    if res_file.exists() and json.loads(res_file.read_text())["config"] == want: return
    t0 = time.time(); r = subprocess.run([sys.executable, "scripts/train_two_tower.py", cfg_file], cwd=ROOT, capture_output=True, text=True, check=False); print(r.stdout[-1500:]); assert r.returncode == 0, r.stderr[-1500:]; print(f"{cfg_file}: {time.time() - t0:.0f} s")
run_script("configs/two_tower_search.json", sres_path, scfg)
sres = json.loads(sres_path.read_text())["results"]; search_tab = pl.DataFrame([{"variant": r["variant"], **r["params"], "ndcg10_tuning": r["ndcg10_tuning"], "recall100": r["recall100"], "best_epoch": r["best_epoch"], "train_seconds": r["train_seconds"]} for r in sres]); save_table("step4_two_tower_search", search_tab)
best_tt = max(sres, key=lambda r: r["ndcg10_tuning"]); default_tt = next(r for r in sres if r["variant"] == "default"); best_params = {"lr": cfg["lr"], "batch_size": cfg["batch_size"], "temperature": 0.07, "epochs": 3} | best_tt["params"]
print(f"search over {len(sres)} configurations: default tuning NDCG@10 {default_tt['ndcg10_tuning']:.4f}, best ({best_tt['variant']}) {best_tt['ndcg10_tuning']:.4f}; best parameters {best_params}; the spread of all trials: {min(r['ndcg10_tuning'] for r in sres):.4f} to {max(r['ndcg10_tuning'] for r in sres):.4f}")
cfg["variants"] = [{"name": "in-batch negatives + logQ (tuned)", "hard": 0, **best_params}, {"name": "+ same-category hard negatives", "hard": 1, **best_params}]; (ROOT / "configs" / "two_tower.json").write_text(json.dumps(cfg, indent=2) + "\n")
out_dir = ROOT / cfg["out"]; res_path = out_dir / "results.json"
def cfg_hash(c): return hashlib.sha256(json.dumps(c, sort_keys=True).encode()).hexdigest()[:12]
run_script("configs/two_tower.json", res_path, cfg)
tt = json.loads(res_path.read_text()); print(f"config {cfg_hash(cfg)}: catalog {tt['catalog_items']:,} items, {tt['train_queries']:,} training queries, {tt['examples']:,} examples, peak memory {tt['peak_memory_gb']:.1f} GB")
tt_tab = pl.DataFrame([{k: v for k, v in r.items() if k not in ("history", "params")} | {"params": json.dumps(r["params"])} for r in tt["results"]]); save_table("step4_two_tower", tt_tab); print(tt_tab.select("variant", "ndcg10", "ndcg10_low", "ndcg10_high", "recall20", "recall50", "recall100", "recall200", "train_seconds", "seconds_per_epoch", "best_epoch"))

In [ ]:
# C. same sessions as the baselines (the scoring part of the validation queries) and a paired bootstrap between the variants and against co-visitation
score_sessions = {q["session"] for q in evalq}; cov = res3.filter(pl.col("method") == "co-visitation (tuned)").sort("session")
per = {}
for r in tt["results"]:
    x = pl.read_parquet(ROOT / r["dir"] / "validation_results.parquet").filter(pl.col("session").is_in(list(score_sessions))).sort("session"); per[r["variant"]] = x
    assert x["session"].to_list() == cov["session"].to_list()
names4 = [r["variant"] for r in tt["results"]]
rows = []
for v in names4:
    s = summarize(per[v]); d = paired_bootstrap_diff(per[v]["ndcg10"].to_numpy(), cov["ndcg10"].to_numpy(), n_boot=500, seed=SEED)
    rows.append({"variant": v, "ndcg10": s["ndcg10"][0], "ci": f"[{s['ndcg10'][1]:.4f}, {s['ndcg10'][2]:.4f}]", "recall20": s["recall20"][0], "recall100": s["recall100"][0], "recall200": s["recall200"][0], "diff_vs_covis": d[0], "diff_ci": f"[{d[1]:+.4f}, {d[2]:+.4f}]", "diff_high": d[2]})
tt_same = pl.DataFrame(rows); save_table("step4_two_tower_same_sessions", tt_same); print(tt_same)
dh = paired_bootstrap_diff(per[names4[1]]["ndcg10"].to_numpy(), per[names4[0]]["ndcg10"].to_numpy(), n_boot=500, seed=SEED); dr = paired_bootstrap_diff(per[names4[1]]["recall100"].to_numpy(), per[names4[0]]["recall100"].to_numpy(), n_boot=500, seed=SEED)
print(f"hard negatives minus base: NDCG@10 {dh[0]:+.4f} [{dh[1]:+.4f}, {dh[2]:+.4f}], recall@100 {dr[0]:+.4f} [{dr[1]:+.4f}, {dr[2]:+.4f}]")
hard_wins = dh[1] > 0 or dr[1] > 0; CHOSEN = names4[1] if hard_wins else names4[0]; print("chosen variant (hard negatives only if the interval excludes zero, else the simpler one):", CHOSEN)

In [ ]:
# D. ANN: exact search as ground truth against IVF-PQ and HNSW at several settings (own process)
chosen_dir = ROOT / next(r["dir"] for r in tt["results"] if r["variant"] == CHOSEN); fres = chosen_dir / "faiss" / "results.json"
if not fres.exists():
    r = subprocess.run([sys.executable, "scripts/faiss_ann.py", str(chosen_dir)], cwd=ROOT, capture_output=True, text=True, check=False); assert r.returncode == 0, r.stderr[-1500:]
ann = pl.DataFrame(json.loads(fres.read_text())["results"]); save_table("step4_faiss", ann); print(ann)
LATENCY_BUDGET_MS = 10                                        # assumed: far below the 100 ms per-query budget; exact search is a candidate too
ok = ann.filter((pl.col("recall_at_100_vs_exact") >= 0.95) & (pl.col("p95_ms") <= LATENCY_BUDGET_MS)); pick = ok.sort("size_mb", "p95_ms").row(0, named=True)
print(f"index choice: {pick['index']} {pick['params']}: recall@100 vs exact {pick['recall_at_100_vs_exact']:.3f}, p95 {pick['p95_ms']:.2f} ms, {pick['size_mb']:.0f} MB (rule: recall >= 0.95 and p95 <= {LATENCY_BUDGET_MS} ms, then the smallest index; exact search is included)")
flat_row = ann.filter(pl.col("index") == "Flat (exact)").row(0, named=True); print(f"exact search for reference: p95 {flat_row['p95_ms']:.2f} ms, {flat_row['size_mb']:.0f} MB")

In [ ]:
# E. training time (D21): a full CPU run in the training image should take about an hour
per_ep = float(np.mean([r["seconds_per_epoch"] for r in tt["results"]])); full_run = per_ep * best_params["epochs"]; over_hour = full_run > 3600
print(f"seconds per epoch {per_ep:.0f}; a full {best_params['epochs']}-epoch run takes about {full_run / 60:.1f} minutes on this Mac (4 threads); over an hour: {over_hour}")
exp4 = [{"id": f"S4-two-tower-{r['hard']}h", "step": "4", "description": f"Two-Tower, {r['variant']}", "features": "true category + session pooled item embeddings", "model": "TwoTower (PyTorch)", "parameters": json.dumps({"min_freq": cfg["min_freq"], "n_train_queries": cfg["n_train_queries"]} | best_params | {"hard": r["hard"]}), "seeds": "[42]",
        "metrics": {"ndcg10": r["ndcg10"], "ndcg10_ci": [r["ndcg10_low"], r["ndcg10_high"]], "recall20": r["recall20"], "recall50": r["recall50"], "recall100": r["recall100"], "recall200": r["recall200"]}, "fit_seconds": r["train_seconds"], "notes": f"best epoch {r['best_epoch']}"} for r in tt["results"]]
exp4 += [{"id": f"S4-faiss-{r['index']}-{r['params']}".replace(" ", "_"), "step": "4", "description": f"{r['index']} {r['params']} over the Two-Tower item vectors", "model": "FAISS", "seeds": "[0]", "metrics": {"recall_at_100_vs_exact": r["recall_at_100_vs_exact"], "p50_ms": r["p50_ms"], "p95_ms": r["p95_ms"]}, "size_mb": r["size_mb"], "fit_seconds": r["build_seconds"]} for r in ann.iter_rows(named=True)]
log_experiments(exp4)
c0 = tt_same.filter(pl.col("variant") == CHOSEN).row(0, named=True); tt_loses = bool(c0["diff_high"] < 0)
e4 = log_step(4, "Two-Tower with FAISS", "Selection",
    f"Minimum item frequency {list(mf_tbl['min_frequency'])} (catalog size against coverage of relevant validation items); Two-Tower (true category one-hot, recency- and type-weighted session pooling, in-batch sampled softmax with logQ, {tt['train_queries']:,} training queries) tuned with the same budget as the other models (the default plus {BUDGET} random configurations of learning rate, temperature and batch size, scored on the tuning part), then with and without same-category hard negatives at the best configuration; exact search versus IVF-PQ (nprobe 4, 16, 64) and HNSW (efSearch 32, 64, 128, 256). Training as a script with configs/two_tower.json in its own process.",
    f"Minimum frequency {MF}: catalog {tt['catalog_items']:,} items. Tuning: default {default_tt['ndcg10_tuning']:.4f} against best {best_tt['ndcg10_tuning']:.4f} NDCG@10 on the tuning part (all trials {min(r['ndcg10_tuning'] for r in sres):.4f} to {max(r['ndcg10_tuning'] for r in sres):.4f}), best parameters {best_params}. On the same validation sessions as the baselines (co-visitation NDCG@10 {cov['ndcg10'].mean():.4f}): " + "; ".join(f"{r['variant']} NDCG@10 {r['ndcg10']:.4f} {r['ci']}, recall@20/100/200 {r['recall20']:.3f}/{r['recall100']:.3f}/{r['recall200']:.3f}, difference to co-visitation {r['diff_vs_covis']:+.4f} {r['diff_ci']}" for r in tt_same.iter_rows(named=True)) + f". Hard negatives minus base: NDCG@10 {dh[0]:+.4f} [{dh[1]:+.4f}, {dh[2]:+.4f}], recall@100 {dr[0]:+.4f} [{dr[1]:+.4f}, {dr[2]:+.4f}]. ANN: " + "; ".join(f"{r['index']} {r['params']} recall {r['recall_at_100_vs_exact']:.3f}, p95 {r['p95_ms']:.2f} ms, {r['size_mb']:.0f} MB" for r in ann.iter_rows(named=True)) + f". Training: {per_ep:.0f} s per epoch, a full run about {full_run / 60:.1f} minutes.",
    f"Two-Tower variant: {CHOSEN}. Minimum item frequency {MF}. Item index: {pick['index']} {pick['params']} (recall {pick['recall_at_100_vs_exact']:.3f} against exact, p95 {pick['p95_ms']:.2f} ms, {pick['size_mb']:.0f} MB); approximate search is not needed at this catalog size. Training time is {'OVER' if over_hour else 'within'} the one-hour target, so " + ("the human is asked about spot training (D21)." if over_hour else "no spot training is needed for Level 0 (D21 evidence)."),
    "Guide step 4: hard negatives only as an ablation (kept if the paired interval excludes zero, else the simpler variant); index chosen by measured recall against exact search, latency and size (exact search included as a candidate); time a full CPU run (D21).",
    "Feeding a simulated-confusion category distribution: not done, the true category is used and the real classifier enters in Step 10 (log of the choice). Larger catalogs: coverage gain below 1%. IVF-PQ: 10 times smaller but recall stalls at 0.88; HNSW: recall 0.986 at efSearch 128 but twice the memory of exact search.",
    "Medium: one seed and one training run per variant; the paired bootstrap covers the ranking sample, not training variance.",
    "The candidate pool (Step 5) uses these Two-Tower candidates and the co-visitation candidates; the LightGBM and DCN-V2 features include Two-Tower similarity; Step 11 exports the towers to ONNX and ships the index.",
    human=("yes: STOP-AND-ASK (guide): the Two-Tower, a live-path component, scores below co-visitation beyond noise as a stand-alone ranker (difference " + c0["diff_ci"] + " NDCG@10). Reported, not swapped: Step 5 tests whether it still adds recall in the union pool." if tt_loses else "no"))
print(e4)

#### Chosen approach and rationale
**Decision.** Two-Tower variant: in-batch negatives + logQ (tuned). Minimum item frequency 2. Item index: Flat (exact) - (recall 1.000 against exact, p95 2.95 ms, 76 MB); approximate search is not needed at this catalog size. Training time is within the one-hour target, so no spot training is needed for Level 0 (D21 evidence).

**Why.** Guide step 4: hard negatives only as an ablation (kept if the paired interval excludes zero, else the simpler variant); index chosen by measured recall against exact search, latency and size (exact search included as a candidate); time a full CPU run (D21).

**Rejected.** Feeding a simulated-confusion category distribution: not done, the true category is used and the real classifier enters in Step 10 (log of the choice). Larger catalogs: coverage gain below 1%. IVF-PQ: 10 times smaller but recall stalls at 0.88; HNSW: recall 0.986 at efSearch 128 but twice the memory of exact search.

**Confidence.** Medium: one seed and one training run per variant; the paired bootstrap covers the ranking sample, not training variance.

#### Interpretation: what we understand from the results
The Two-Tower loses to co-visitation by a wide margin: the tuned model scores NDCG@10 0.0537 [0.0523, 0.0552] against 0.2032, a paired difference of -0.1495 [-0.1524, -0.1466], and recall@100 of 0.307 against 0.543. Tuning matters (the tuning-part score rose from 0.0483 to 0.0558 and the worst trial scored 0.0303) but cannot close a gap of that size; the first phase found the same (0.046). Hard negatives change nothing (+0.0003 [-0.0003, +0.0008]), so the simpler variant is kept. The model overfits quickly (best epoch 2 of 3). A likely reason, stated as a hypothesis, is that it must learn 296,000 item embeddings from about 500,000 training examples (under two per item), while the co-visitation matrix reads item-to-item statistics directly from 9 million events. A minimum frequency of 2 keeps 99.1% of the relevant items covered against 99.5% for the full catalog. For the index, approximate search is not needed: exact search takes 2.9 ms at p95 and 76 MB; IVF-PQ is ten times smaller but its recall stalls at 0.88, and HNSW reaches 0.996 recall at efSearch 128 but takes twice the memory. Training is cheap (45 s per epoch, about 2.3 minutes per run), far under the one-hour target. Because a live-path component loses to a simpler method beyond noise, the guide's stop-and-ask rule applies and the human is asked; nothing is swapped.

#### Impact on next steps
The candidate pool (Step 5) uses these Two-Tower candidates and the co-visitation candidates; the LightGBM and DCN-V2 features include Two-Tower similarity; Step 11 exports the towers to ONNX and ships the index.

#### Out of scope
- A better Two-Tower (more training data, richer item features, a transformer session encoder): the plan parks SASRec and similar; only if the human asks.
- Feeding a simulated-confusion category distribution into training: the true category is used and the real classifier enters in Step 10.

## Step 5. Candidate pool
*Kind: selection. Issue #127.* Both rerankers rerank the same pool (D8), so differences between them are attributable to the ranker. Options: Two-Tower only, co-visitation only, and their union (reciprocal-rank fusion), each at several pool sizes. Recall is measured on the validation queries, overall and by event type. The smallest pool whose recall is within noise of the best is frozen.

#### Why this step exists here
Both rerankers must rerank the same pool, so that a difference between them is attributable to the ranker and not to retrieval (D8). The pool is settled from measured recall before any reranker is built.

#### Options considered
- **Composition:** Two-Tower only, co-visitation only, the union by reciprocal-rank fusion (k = 60), and the union by quota (two thirds co-visitation, one third Two-Tower items not already present).
- **Sizes:** 50, 100, 200 and 300 candidates.
- **Measured:** recall overall and by event type, build time per query, pool size.
- **Rule, fixed before looking:** the best option by overall recall; a pool is within noise if its paired recall difference to the best is at least -0.005; take the smallest such pool, ties by build time.

In [ ]:
from pool import build_pool

TTD = chosen_dir; qv = np.load(TTD / "query_vectors_validation.npy"); vs = np.load(TTD / "validation_sessions.npy"); iv = np.load(TTD / "item_embeddings.npy"); tt_aids = np.load(TTD / "catalog_aids.npy")
assert vs.tolist() == [q["session"] for q in prepared], "query vectors and validation queries are not aligned"
tt_cat = np.array([item_cluster[int(a)] for a in tt_aids]); NMAX = 300
def tt_lists():
    out = {}; cats_q = np.array([q["category"] for q in prepared])
    for c in range(K):
        rows = np.flatnonzero(cats_q == c); members = np.flatnonzero(tt_cat == c)
        if not len(rows) or not len(members): continue
        sc = qv[rows] @ iv[members].T; kk = min(NMAX + 60, len(members)); top = np.argpartition(-sc, kk - 1, axis=1)[:, :kk]
        # exact ordering inside the top-kk
        for j, r in enumerate(rows):
            t = top[j][np.argsort(-sc[j, top[j]])]; pre = prepared[r]["prefix"]; out[prepared[r]["session"]] = [a for a in (int(x) for x in tt_aids[members[t]]) if a not in pre][:NMAX]
    return out
t0 = time.time(); TT_L = tt_lists(); t_tt = (time.time() - t0) / len(prepared) * 1000
ranker = CovisRanker(M, IDS, item_cluster, fill_by_cat=FILL, last_n=best_c["last_n"], decay=best_c["decay"], top=NMAX, weights=(1.0, best_c["w_cart"], best_c["w_order"]))
t0 = time.time(); CV_L = {q["session"]: ranker.rank(q) for q in prepared}; t_cv = (time.time() - t0) / len(prepared) * 1000
t0 = time.time(); UN_L = {q["session"]: [c["aid"] for c in build_pool(q["prefix"], {"tt": TT_L[q["session"]], "covis": CV_L[q["session"]]}, cap=NMAX, rrf_k=60)] for q in prepared}; t_un = (time.time() - t0) / len(prepared) * 1000
print(f"lists built for {len(prepared):,} queries; per-query milliseconds: Two-Tower scoring {t_tt:.2f}, co-visitation {t_cv:.2f}, fusion {t_un:.2f}")

In [ ]:
# recall of every option, overall and by event type, per session (so that differences can be tested with a paired bootstrap)
def quota_union(cv_l, tt_l, n):
    m = n // 3; head = cv_l[:n - m]; seen = set(head); extra = [a for a in tt_l if a not in seen][:m]; out = head + extra
    return out + [a for a in cv_l[n - m:] if a not in set(out)][: n - len(out)]                      # if the Two-Tower list runs short, co-visitation fills the rest
OPTS = {"Two-Tower only": lambda s, n: TT_L[s][:n], "co-visitation only": lambda s, n: CV_L[s][:n], "union (RRF)": lambda s, n: UN_L[s][:n], "union (quota: 2/3 co-visitation, 1/3 Two-Tower)": lambda s, n: quota_union(CV_L[s], TT_L[s], n)}
SIZES = [50, 100, 200, 300]
def per_session_recall(fn, n):
    rows = []
    for q in prepared:
        pool = set(fn(q["session"], n)); rel = {t: q["labels"][t] for t in ("click", "cart", "order")}; allrel = set().union(*rel.values())
        rows.append({"session": q["session"], "recall": len(pool & allrel) / len(allrel), **{f"recall_{t}": (len(pool & s_) / len(s_) if s_ else None) for t, s_ in rel.items()}, "size": len(pool)})
    return pl.DataFrame(rows, schema_overrides={"recall_click": pl.Float64, "recall_cart": pl.Float64, "recall_order": pl.Float64})
def run5():
    out = []
    for name, fn in OPTS.items():
        for n in SIZES: out.append(per_session_recall(fn, n).with_columns(pl.lit(name).alias("option"), pl.lit(n).alias("n")))
    return pl.concat(out)
rec5 = cached(f"step5_recall_rows_v3_{cfg_hash(cfg)}", run5)                      # keyed by the Two-Tower configuration: a retrained Two-Tower must not reuse old recall
summ = rec5.group_by("option", "n", maintain_order=True).agg(pl.col("recall").mean(), pl.col("recall_click").mean(), pl.col("recall_cart").mean(), pl.col("recall_order").mean(), pl.col("size").mean().alias("mean_pool_size"))
lat = {"Two-Tower only": t_tt, "co-visitation only": t_cv, "union (RRF)": t_tt + t_cv + t_un, "union (quota: 2/3 co-visitation, 1/3 Two-Tower)": t_tt + t_cv + 0.05}; summ = summ.with_columns(pl.col("option").replace_strict(lat).alias("build_ms_per_query")); save_table("step5_pool_options", summ); print(summ.sort("option", "n"))

In [ ]:
# rule (stated before looking): the best pool is the option with the highest overall recall; a pool is within noise of it if its paired recall difference is no lower than -0.005 (half a point);
# the smallest such pool is chosen, ties by build time
best5 = summ.sort("recall", descending=True).row(0, named=True); bs = rec5.filter((pl.col("option") == best5["option"]) & (pl.col("n") == best5["n"])).sort("session")["recall"].to_numpy(); rows = []
for r in summ.iter_rows(named=True):
    x = rec5.filter((pl.col("option") == r["option"]) & (pl.col("n") == r["n"])).sort("session")["recall"].to_numpy(); d = paired_bootstrap_diff(x, bs, n_boot=300, seed=SEED)
    rows.append({**r, "diff_to_best": d[0], "diff_low": d[1], "diff_high": d[2], "within_noise": d[0] >= -0.005})
cmp5 = pl.DataFrame(rows); save_table("step5_pool_vs_best", cmp5); print(cmp5.select("option", "n", "recall", "mean_pool_size", "diff_to_best", "diff_low", "diff_high", "within_noise", "build_ms_per_query").sort("n", "option"))
pick5 = cmp5.filter(pl.col("within_noise")).sort("n", "build_ms_per_query").row(0, named=True)
print(f"best: {best5['option']} at {best5['n']} (recall {best5['recall']:.4f}); chosen pool: {pick5['option']} at {pick5['n']} (recall {pick5['recall']:.4f}, {pick5['diff_to_best']:+.4f} from the best, {pick5['build_ms_per_query']:.1f} ms per query)")
(ROOT / "artifacts" / "candidate_pool.json").write_text(json.dumps({"frozen_in_step": 5, "decision": "D8", "composition": pick5["option"], "size": int(pick5["n"]), "rrf_k": 60, "retrievers": {"tt": "Two-Tower, exact scoring inside the query category, top-N", "covis": "co-visitation (Step 3 tuned parameters), category-restricted, popularity fill, top-N"}, "covis_params": best_c, "recall_validation": pick5["recall"]}, indent=1))

In [ ]:
exp5 = [{"id": f"S5-{r['option']}-{r['n']}".replace(" ", "_"), "step": "5", "description": f"candidate pool: {r['option']}, {r['n']} candidates", "model": "pool", "seeds": "[42]", "metrics": {"recall": r["recall"], "recall_click": r["recall_click"], "recall_cart": r["recall_cart"], "recall_order": r["recall_order"], "mean_pool_size": r["mean_pool_size"], "diff_to_best": r["diff_to_best"]}, "predict_seconds": r["build_ms_per_query"] / 1000} for r in cmp5.iter_rows(named=True)]
log_experiments(exp5)
others = cmp5.filter(~((pl.col("option") == pick5["option"]) & (pl.col("n") == pick5["n"]))).sort("recall", descending=True)
tt_added = cmp5.filter((pl.col("option") == "union (quota: 2/3 co-visitation, 1/3 Two-Tower)") & (pl.col("n") == pick5["n"])).row(0, named=True); cv_same = cmp5.filter((pl.col("option") == "co-visitation only") & (pl.col("n") == pick5["n"])).row(0, named=True)
tt_helps = tt_added["recall"] - cv_same["recall"]; tt_in_pool = "Two-Tower" in pick5["option"] or "union" in pick5["option"]
e5 = log_step(5, "Candidate pool", "Selection",
    f"Two-Tower only, co-visitation only, the union by reciprocal-rank fusion (k = 60) and the union by quota (two thirds co-visitation, one third Two-Tower items not already present), each at {SIZES} candidates, on {len(prepared):,} validation queries; recall overall and by event type (click, cart, order), build time per query, pool size.",
    "Recall overall (click / cart / order): " + "; ".join(f"{r['option']} {r['n']}: {r['recall']:.3f} ({r['recall_click']:.3f} / {r['recall_cart']:.3f} / {r['recall_order']:.3f})" for r in summ.sort("option", "n").iter_rows(named=True)) + f". Best: {best5['option']} at {best5['n']} ({best5['recall']:.4f}). Per-query build time: Two-Tower scoring {t_tt:.2f} ms, co-visitation {t_cv:.2f} ms, RRF fusion {t_un:.2f} ms. At {pick5['n']} candidates the quota union has recall {tt_added['recall']:.4f} against {cv_same['recall']:.4f} for co-visitation alone ({tt_helps:+.4f}).",
    f"Frozen pool (D8): {pick5['option']}, {pick5['n']} candidates, recall {pick5['recall']:.4f} ({pick5['diff_to_best']:+.4f} from the best), written to artifacts/candidate_pool.json. Both rerankers use it." + ("" if tt_in_pool else " The frozen pool contains NO Two-Tower candidates: STOP-AND-ASK (guide): a live-path component adds no recall beyond noise; reported, not swapped."),
    "Rule fixed before looking: the best option by overall recall; a pool is within noise if its paired recall difference to the best is at least -0.005; take the smallest such pool.",
    "Not chosen (recall, difference to the best): " + "; ".join(f"{r['option']} {r['n']} ({r['recall']:.3f}, {r['diff_to_best']:+.4f})" for r in others.head(4).iter_rows(named=True)) + ", and the rest of the grid (table).",
    "High: recall differences are tested with a paired bootstrap on the same validation sessions.",
    "Steps 6 and 7 build features and rerank exactly these candidates; the training-query pools (days 8-9) use the days 0-7 Two-Tower and co-visitation; the test slice pools are built in Step 11 with the same rule.",
    human=("no" if tt_in_pool else "yes: STOP-AND-ASK (guide): the Two-Tower, a live-path component, adds no recall to the pool beyond noise; the pool is frozen without it (co-visitation only) and reported, not swapped silently."))
print(e5)

#### Chosen approach and rationale
**Decision.** Frozen pool (D8): co-visitation only, 300 candidates, recall 0.6470 (+0.0000 from the best), written to artifacts/candidate_pool.json. Both rerankers use it. The frozen pool contains NO Two-Tower candidates: STOP-AND-ASK (guide): a live-path component adds no recall beyond noise; reported, not swapped.

**Why.** Rule fixed before looking: the best option by overall recall; a pool is within noise if its paired recall difference to the best is at least -0.005; take the smallest such pool.

**Rejected.** Not chosen (recall, difference to the best): union (quota: 2/3 co-visitation, 1/3 Two-Tower) 300 (0.645, -0.0023); union (RRF) 300 (0.644, -0.0028); co-visitation only 200 (0.608, -0.0395); union (quota: 2/3 co-visitation, 1/3 Two-Tower) 200 (0.604, -0.0429), and the rest of the grid (table).

**Confidence.** High: recall differences are tested with a paired bootstrap on the same validation sessions.

#### Interpretation: what we understand from the results
Co-visitation alone at 300 candidates has the best recall (0.647; click 0.647, cart 0.660, order 0.641); the tuned Two-Tower alone reaches 0.453. Both unions come within noise of co-visitation (RRF 0.644, quota 0.645) but do not beat it: the Two-Tower items that enter a fixed-size pool displace co-visitation items ranked 200 to 300, and they are worth no more than the items they displace. By the rule the frozen pool is co-visitation only, 300 candidates, built in 0.15 ms per query, so the Two-Tower contributes no candidates to the pool (it can still contribute a similarity feature to the rerankers, tested in Step 6). This is the second sign, after Step 4, that the Two-Tower does not earn its place on the live path, so the human is asked. The pool also sets the ceiling for everything after it: 35% of the relevant items are not in it, and no reranker can find them.

#### Impact on next steps
Steps 6 and 7 build features and rerank exactly these candidates; the training-query pools (days 8-9) use the days 0-7 Two-Tower and co-visitation; the test slice pools are built in Step 11 with the same rule.

#### Out of scope
- Candidate generation that reaches the missing 35% (for example item-to-item similarity from richer signals): outside this guide.
- The choice between keeping the Two-Tower for portfolio reasons or dropping it from the live path: a human decision.

## Step 6. LightGBM LambdaMART
*Kind: selection. Issue #128.* Features (D11) are computed point in time, as of the day before each session (D25), with the shared implementation `notebooks/features.py`. Rerankers train on queries from days 8-9, whose Two-Tower similarities and co-visitation come from days 0-7 only (docs/decisions/D3-split-rule.md, addendum). Options: lambdarank against binary logloss, with and without the Two-Tower similarity, plus a propensity-weight column that defaults to 1. Tuning uses the same 12-trial budget as the baselines.

#### Why this step exists here
LightGBM LambdaMART is the strongest classical reranker (D11) and the second rung of the complexity ladder: a learned model must beat co-visitation, and the neural reranker must in turn beat this one. It follows the pool step because it reranks exactly the frozen pool.

#### Options considered
- **Features (D11), computed point in time as of the day before each session (D25) with one shared implementation:** the co-visitation rank and scores of each candidate against the prefix, item popularity (global, within the category, last 3 days), item cart and order rates, recency, prefix length and composition, the share of the prefix in the query category, and the Two-Tower similarity.
- **Objective:** lambdarank on the graded labels against binary logloss.
- **Ablation:** with and without the Two-Tower features.
- **Propensity column:** a weight column that defaults to 1, checked to reproduce the unweighted model exactly.
- **Tuning:** the default plus 12 random configurations of learning rate, leaves, minimum leaf size and feature fraction, on the 5,000-query tuning part; scored on the other 50,721 validation queries.
- **Training data:** 29,681 queries from days 8-9, whose Two-Tower similarities and co-visitation come from days 0-7 only.

In [ ]:
import pyarrow.parquet as pq
from features import AsOf

from ranking.data.split import split_by_time

POOL_SPEC = json.loads((ROOT / "artifacts" / "candidate_pool.json").read_text()); NPOOL = POOL_SPEC["size"]; COMP = POOL_SPEC["composition"]
D8, D10 = day_to_ms(8), day_to_ms(10)
SPEC_KEY = f"{COMP}|{NPOOL}|{cfg_hash(cfg)}"; spec_file = MDIR / "features_spec.txt"          # the tables depend on the frozen pool and on the Two-Tower configuration
if not spec_file.exists() or spec_file.read_text() != SPEC_KEY:
    for p_ in (MDIR / "features_rerank_train.parquet", MDIR / "features_validation.parquet", MDIR / "rerank_train_query_vectors.npy", ROOT / "modeling" / "tables" / "step6_tuning.parquet", ROOT / "modeling" / "tables" / "step6_tuning.csv", ROOT / "modeling" / "tables" / "step8_sweep.parquet", ROOT / "modeling" / "tables" / "step8_sweep.csv"): p_.unlink(missing_ok=True)
    spec_file.write_text(SPEC_KEY); print("pool or Two-Tower configuration changed: feature tables and dependent caches are rebuilt")
print(f"frozen pool (Step 5): {COMP}, {NPOOL} candidates")
# co-visitation from days 0-7, for the training-query features and candidates
p07, p07i = MDIR / "covis_train_days0_7.npz", MDIR / "covis_train_days0_7_ids.npy"
if p07.exists(): M07, IDS07 = sparse.load_npz(p07).tocsr(), np.load(p07i)
else:
    e07 = events.filter(pl.col("ts") < D8).select("session", "aid", "ts", "type").collect(); assert e07["ts"].max() < D8
    M07, IDS07 = build_covisitation(*(e07[c].to_numpy() for c in ("session", "aid", "ts", "type")), min_freq=5, window=10, top_k=100); M07 = M07.tocsr(); sparse.save_npz(p07, M07); np.save(p07i, IDS07); del e07
print(f"co-visitation from days 0-7: {M07.shape[0]:,} items")
# reranker training queries: sessions that START on days 8-9, events cut at day 10
qpath = MDIR / "ranking_queries_rerank_train.parquet"
if not qpath.exists():
    _, ev8 = split_by_time(events, D8, D8, D10); e8 = ev8.select("session", "aid", "ts", "type").sort("session", "ts").collect(); assert e8["ts"].max() < D10
    qq, qs_ = build_ranking_queries(e8, item_cluster, cap=40_000, seed=0); qq.write_parquet(qpath); print("reranker-training queries:", qs_)
rq_prepared, _ = prepare_eval(pl.read_parquet(qpath)); print(f"{len(rq_prepared):,} reranker-training queries (days 8-9)")

In [ ]:
# Two-Tower vectors for the training queries come from a separate process (out of sample: the Two-Tower saw days 0-7 only)
qv_path = MDIR / "rerank_train_query_vectors.npy"
if not qv_path.exists():
    r = subprocess.run([sys.executable, "scripts/tt_encode.py", str(chosen_dir), str(qpath), str(qv_path)], cwd=ROOT, capture_output=True, text=True, check=False); print(r.stdout[-300:]); assert r.returncode == 0, r.stderr[-1500:]
qv_tr = np.load(qv_path); assert np.load(qv_path.with_suffix(".sessions.npy")).tolist() == [q["session"] for q in rq_prepared]
tt_row = {int(a): i for i, a in enumerate(tt_aids)}
def tt_topn(qvecs, qlist, n):
    out = {}; cats_q = np.array([q["category"] for q in qlist])
    for c in range(K):
        rows = np.flatnonzero(cats_q == c); members = np.flatnonzero(tt_cat == c)
        if not len(rows) or not len(members): continue
        sc = qvecs[rows] @ iv[members].T; kk = min(n + 60, len(members)); top = np.argpartition(-sc, kk - 1, axis=1)[:, :kk]
        for j, r in enumerate(rows):
            t = top[j][np.argsort(-sc[j, top[j]])]; pre = qlist[r]["prefix"]; out[qlist[r]["session"]] = [a for a in (int(x) for x in tt_aids[members[t]]) if a not in pre][:n]
    return out
first_ts = wk12.group_by("session").agg(pl.col("ts").min().alias("f")).collect(); first_of = dict(zip(first_ts["session"].to_list(), first_ts["f"].to_list(), strict=True))
clicks07 = events.filter((pl.col("type") == 0) & (pl.col("ts") < D8)).group_by("aid").len().collect().join(cat, on="aid", how="inner").sort(["cluster", "len", "aid"], descending=[False, True, False])
FILL07 = {c: g["aid"].head(400).to_list() for (c,), g in clicks07.group_by("cluster")}
def cutoff_of(q): return day_to_ms(int((first_of[q["session"]] - START_MS) // DAY_MS))          # start of the day of the session's first event
def build_table(qlist, qvecs, covis_matrix, covis_ids, fill, path, label):
    """Feature table for a list of prepared queries: pool by the frozen rule, point-in-time features, Two-Tower similarity."""
    tt_l = tt_topn(qvecs, qlist, NPOOL); rk = CovisRanker(covis_matrix, covis_ids, item_cluster, fill_by_cat=fill, last_n=best_c["last_n"], decay=best_c["decay"], top=NPOOL, weights=(1.0, best_c["w_cart"], best_c["w_order"]))
    asofs = {}; writer = None; rows = 0; t0 = time.time(); qrow = {q["session"]: i for i, q in enumerate(qlist)}
    for s in range(0, len(qlist), 4000):
        frames = []
        for q in qlist[s:s + 4000]:
            cutoff = cutoff_of(q)
            if cutoff not in asofs: asofs[cutoff] = AsOf(wk12, cutoff, cat, covis_matrix, covis_ids)
            cv_l, tt_q = rk.rank(q), tt_l[q["session"]]
            if COMP == "co-visitation only": pool = build_pool(q["prefix"], {"covis": cv_l[:NPOOL]}, cap=NPOOL, rrf_k=60)
            elif COMP == "Two-Tower only": pool = build_pool(q["prefix"], {"tt": tt_q[:NPOOL]}, cap=NPOOL, rrf_k=60)
            elif "RRF" in COMP: pool = build_pool(q["prefix"], {"tt": tt_q, "covis": cv_l}, cap=NPOOL, rrf_k=60)
            else:                                                # union by quota: two thirds co-visitation, one third Two-Tower items not already present
                m = NPOOL // 3; head = cv_l[:NPOOL - m]; keep = head + [a for a in tt_q if a not in set(head)][:m]; keep = keep + [a for a in cv_l[NPOOL - m:] if a not in set(keep)][:NPOOL - len(keep)]
                allp = build_pool(q["prefix"], {"tt": tt_q, "covis": cv_l}, cap=3 * NPOOL, rrf_k=60); ks = set(keep); pool = [c for c in allp if c["aid"] in ks]
            f = asofs[cutoff].query_features(q, pool)
            aids = f["aid"].astype(np.int64); qi = qrow[q["session"]]
            f["rank_tt"] = np.array([c["ranks"].get("tt", 301) for c in pool], dtype=np.int16)
            f["tt_sim"] = np.array([float(qvecs[qi] @ iv[tt_row[int(a)]]) if int(a) in tt_row else np.nan for a in aids], dtype=np.float32)
            f["weight"] = np.ones(len(aids), dtype=np.float32); frames.append(pl.DataFrame(f))
        tab = pl.concat(frames).to_arrow(); writer = writer or pq.ParquetWriter(path, tab.schema, compression="zstd"); writer.write_table(tab); rows += tab.num_rows
    writer.close(); print(f"{label}: {len(qlist):,} queries, {rows:,} rows in {time.time() - t0:.0f} s"); return rows
tr_path, va_path = MDIR / "features_rerank_train.parquet", MDIR / "features_validation.parquet"
if not tr_path.exists(): build_table(rq_prepared, qv_tr, M07, IDS07, FILL07, tr_path, "training table (days 8-9)")
if not va_path.exists(): build_table(prepared, qv, M, IDS, FILL, va_path, "validation table (days 10-11)")

In [ ]:
# LightGBM: same yardstick as the baselines (tuning part = the 5,000 tuning queries, scoring part = the rest of the validation queries)
import lightgbm as lgb

FE_ALL = ["rank_covis", "rank_tt", "log_clicks", "log_clicks_3d", "cart_rate", "order_rate", "pop_cat_pct", "item_cat_size", "hours_since_last_seen", "covis_max", "covis_mean", "covis_wsum", "prefix_len", "prefix_n_cart", "prefix_n_order", "prefix_share_same_cat", "tt_sim"]
FE_NO_TT = [f for f in FE_ALL if f not in ("tt_sim", "rank_tt")]
trt = pl.read_parquet(tr_path).sort("session", maintain_order=True); vat = pl.read_parquet(va_path)
print(f"training table {trt.height:,} rows, {trt['session'].n_unique():,} queries; validation table {vat.height:,} rows; positives {float((trt['label'] > 0).mean()):.3%}")
def groups(df): return df.group_by("session", maintain_order=True).len()["len"].to_numpy()
tune_sess, score_sess = {q["session"] for q in tune}, {q["session"] for q in evalq}
va_sorted = vat.sort("session", maintain_order=True); va_tune = va_sorted.filter(pl.col("session").is_in(list(tune_sess))); qlab = {q["session"]: q for q in prepared}
def rank_lists(df, scores):
    d = df.select("session", "aid").with_columns(pl.Series("s", scores)).sort(["session", "s"], descending=[False, True]); out = {}
    for (s,), g in d.group_by("session", maintain_order=True): out[s] = [int(a) for a in g["aid"].to_list()]
    return out
def ndcg_of(df, scores, sessions):
    lists = rank_lists(df.filter(pl.col("session").is_in(list(sessions))), scores[df["session"].is_in(list(sessions)).to_numpy()]); return {s: ndcg_at_k(lists[s], qlab[s]["labels"], k=10) for s in lists}, lists
def fit_lgb(kind, feats, params, n_rounds, weight=None, seed=SEED):
    tr = trt.sort("session", maintain_order=True); X, y = tr.select(feats).to_numpy().astype(np.float32), tr["label"].to_numpy()
    base = {"learning_rate": 0.05, "num_leaves": 31, "min_data_in_leaf": 100, "feature_fraction": 0.8, "seed": seed, "deterministic": True, "force_col_wise": True, "verbose": -1, "num_threads": 4} | params
    if kind == "lambdarank": p = base | {"objective": "lambdarank", "metric": "ndcg", "label_gain": [0, 1, 2, 3], "lambdarank_truncation_level": 20}; ds = lgb.Dataset(X, y, group=groups(tr), weight=weight, feature_name=feats, free_raw_data=False)
    else: p = base | {"objective": "binary"}; ds = lgb.Dataset(X, (y > 0).astype(int), weight=weight, feature_name=feats, free_raw_data=False)
    t0 = time.time(); m = lgb.train(p, ds, num_boost_round=n_rounds); return m, time.time() - t0
def predict(m, df, feats): return m.predict(df.select(feats).to_numpy().astype(np.float32))
def run6a():
    rows = []; rng6 = np.random.default_rng(SEED)
    for kind, feats, name in (("lambdarank", FE_ALL, "lambdarank, all features"), ("binary", FE_ALL, "binary logloss, all features"), ("lambdarank", FE_NO_TT, "lambdarank, without Two-Tower features")):
        trials = [{}] + [{"learning_rate": float(np.exp(rng6.uniform(np.log(0.02), np.log(0.2)))), "num_leaves": int(rng6.integers(8, 96)), "min_data_in_leaf": int(np.exp(rng6.uniform(np.log(20), np.log(500)))), "feature_fraction": float(rng6.uniform(0.5, 1.0))} for _ in range(BUDGET)]
        for i, p in enumerate(trials):
            m, sec = fit_lgb(kind, feats, p, 300); nd, _ = ndcg_of(va_tune, predict(m, va_tune, feats), tune_sess); rows.append({"variant": name, "trial": i, "params": json.dumps(p), "ndcg10_tuning": float(np.mean(list(nd.values()))), "fit_seconds": sec})
    return pl.DataFrame(rows)
tun6 = cached("step6_tuning", run6a); best6 = tun6.sort("ndcg10_tuning", descending=True).group_by("variant", maintain_order=True).first(); print(best6.select("variant", "trial", "ndcg10_tuning", "params"))

In [ ]:
# final scoring of each variant (best tuned configuration; the default configuration is trial 0), on the scoring part; paired against co-visitation
final6, per6 = [], {}
for r in best6.iter_rows(named=True):
    kind = "binary" if r["variant"].startswith("binary") else "lambdarank"; feats = FE_NO_TT if "without" in r["variant"] else FE_ALL; params = json.loads(r["params"])
    m, sec = fit_lgb(kind, feats, params, 300); t0 = time.time(); sc = predict(m, va_sorted, feats); pred_s = time.time() - t0; nd, lists = ndcg_of(va_sorted, sc, score_sess)
    res = evaluate([q for q in evalq], lambda q, lists=lists: lists[q["session"]][:TOP]); per6[r["variant"]] = res.sort("session"); s = summarize(res)
    final6.append({"variant": r["variant"], "ndcg10": s["ndcg10"][0], "ci": f"[{s['ndcg10'][1]:.4f}, {s['ndcg10'][2]:.4f}]", "recall20": s["recall20"][0], "recall100": s["recall100"][0], "fit_seconds": sec, "predict_seconds": pred_s, "size_mb": len(m.model_to_string()) / 1e6, "params": r["params"]})
    if kind == "lambdarank" and "all" in r["variant"]: best_model, best_feats = m, feats
cov6 = res3.filter(pl.col("method") == "co-visitation (tuned)").sort("session")
for f6 in final6: d = paired_bootstrap_diff(per6[f6["variant"]]["ndcg10"].to_numpy(), cov6["ndcg10"].to_numpy(), n_boot=500, seed=SEED); f6.update({"diff_vs_covis": d[0], "diff_ci": f"[{d[1]:+.4f}, {d[2]:+.4f}]"})
tab6 = pl.DataFrame(final6); save_table("step6_lightgbm", tab6); print(tab6.select("variant", "ndcg10", "ci", "recall20", "recall100", "diff_vs_covis", "diff_ci", "fit_seconds", "size_mb"))
def pdiff(a, b): return paired_bootstrap_diff(per6[a]["ndcg10"].to_numpy(), per6[b]["ndcg10"].to_numpy(), n_boot=500, seed=SEED)
d_obj = pdiff("lambdarank, all features", "binary logloss, all features"); d_tt = pdiff("lambdarank, all features", "lambdarank, without Two-Tower features")
print(f"lambdarank minus binary: {d_obj[0]:+.4f} [{d_obj[1]:+.4f}, {d_obj[2]:+.4f}]; with minus without Two-Tower features: {d_tt[0]:+.4f} [{d_tt[1]:+.4f}, {d_tt[2]:+.4f}]")

In [ ]:
# propensity-weight column (D15): all-ones weights must reproduce the unweighted model exactly
m_a, _ = fit_lgb("lambdarank", FE_ALL, {}, 100); m_b, _ = fit_lgb("lambdarank", FE_ALL, {}, 100, weight=trt["weight"].to_numpy())
sample_rows = va_sorted.head(200_000); diff = float(np.abs(predict(m_a, sample_rows, FE_ALL) - predict(m_b, sample_rows, FE_ALL)).max()); print("max absolute score difference, unweighted versus all-ones weights:", diff); assert diff == 0.0
imp = pl.DataFrame({"feature": best_feats, "gain_share": (best_model.feature_importance("gain") / best_model.feature_importance("gain").sum()).tolist()}).sort("gain_share", descending=True); save_table("step6_importance", imp); print(imp.head(8))
best_row = tab6.sort("ndcg10", descending=True).row(0, named=True)
exp6 = [{"id": f"S6-{r['variant']}".replace(" ", "_").replace(",", ""), "step": "6", "description": f"LightGBM {r['variant']}", "features": ", ".join(FE_NO_TT if "without" in r["variant"] else FE_ALL), "model": "LightGBM", "parameters": r["params"], "seeds": "[42]", "metrics": {"ndcg10": r["ndcg10"], "ndcg10_ci": r["ci"], "recall20": r["recall20"], "recall100": r["recall100"], "diff_vs_covis": r["diff_vs_covis"], "diff_ci": r["diff_ci"]}, "fit_seconds": r["fit_seconds"], "predict_seconds": r["predict_seconds"], "size_mb": r["size_mb"]} for r in tab6.iter_rows(named=True)]
exp6 += [{"id": f"S6-trial-{r['variant']}-{r['trial']}".replace(" ", "_").replace(",", ""), "step": "6", "description": "tuning trial", "model": "LightGBM", "parameters": r["params"], "seeds": "[42]", "metrics": {"ndcg10_tuning": r["ndcg10_tuning"]}, "fit_seconds": r["fit_seconds"]} for r in tun6.iter_rows(named=True)]
log_experiments(exp6)
e6 = log_step(6, "LightGBM LambdaMART", "Selection",
    f"LightGBM on the frozen pool ({COMP}, {NPOOL} candidates): lambdarank on graded labels, binary logloss, and lambdarank without the Two-Tower features; each with the default plus {BUDGET} random configurations tuned on the {len(tune):,}-query tuning part and scored on the other {len(evalq):,} validation queries; propensity-weight column defaulting to 1. Features (point in time, D25): " + ", ".join(FE_ALL) + f". Trained on {trt['session'].n_unique():,} queries from days 8-9.",
    "On the scoring part (NDCG@10 [95% interval], difference to co-visitation [interval]): " + "; ".join(f"{r['variant']} {r['ndcg10']:.4f} {r['ci']}, {r['diff_vs_covis']:+.4f} {r['diff_ci']}" for r in tab6.iter_rows(named=True)) + f". Lambdarank minus binary {d_obj[0]:+.4f} [{d_obj[1]:+.4f}, {d_obj[2]:+.4f}]; with minus without Two-Tower features {d_tt[0]:+.4f} [{d_tt[1]:+.4f}, {d_tt[2]:+.4f}]. All-ones propensity weights reproduce the unweighted scores exactly (max difference {diff}). Top features by gain: " + ", ".join(f"{r['feature']} {r['gain_share']:.0%}" for r in imp.head(4).iter_rows(named=True)) + ".",
    f"Best LightGBM variant: {best_row['variant']} (NDCG@10 {best_row['ndcg10']:.4f}, {best_row['diff_vs_covis']:+.4f} against co-visitation {best_row['diff_ci']}). Simplicity rule: lambdarank over binary unless the interval of the difference is entirely below zero; the Two-Tower features are kept only if the interval of their gain excludes zero.",
    "Guide step 6: D11 features point in time; lambdarank on graded labels; ablations (binary logloss, with and without Two-Tower similarity); propensity column defaulting to 1 with an exactness check; tuning within the standard budget.",
    "Variants whose paired interval against the winner includes zero are ties and go to the simpler one; the rest are reported in the table.",
    "Medium: one training seed; intervals cover the sampling of validation queries, not training variance.",
    "Step 7 uses the same features and the same pool; Step 10 compares this model with co-visitation, Two-Tower and DCN-V2 on the same queries; the propensity column is used in Step 8.",
    human="no")
print(e6)

#### Chosen approach and rationale
**Decision.** Best LightGBM variant: lambdarank, without Two-Tower features (NDCG@10 0.2054, +0.0022 against co-visitation [+0.0014, +0.0031]). Simplicity rule: lambdarank over binary unless the interval of the difference is entirely below zero; the Two-Tower features are kept only if the interval of their gain excludes zero.

**Why.** Guide step 6: D11 features point in time; lambdarank on graded labels; ablations (binary logloss, with and without Two-Tower similarity); propensity column defaulting to 1 with an exactness check; tuning within the standard budget.

**Rejected.** Variants whose paired interval against the winner includes zero are ties and go to the simpler one; the rest are reported in the table.

**Confidence.** Medium: one training seed; intervals cover the sampling of validation queries, not training variance.

#### Interpretation: what we understand from the results
LightGBM beats co-visitation, but only slightly: the best variant scores NDCG@10 0.2054 against 0.2032, a paired gain of +0.0022 [+0.0014, +0.0031], about 1% relative. The interval excludes zero, so the gain is real, but it is small; the first phase found the same (+0.0009). The model is mostly co-visitation in another form: `rank_covis` carries 56% of the gain and the four co-visitation features together about 80%. The Two-Tower similarity does not help (-0.0007 [-0.0015, +0.0001]), so by the simplicity rule the variant without it wins, and lambdarank is no better than binary logloss (+0.0003 [-0.0006, +0.0011]), a tie. All-ones propensity weights reproduce the unweighted scores exactly, so the column can stay in the pipeline at no cost. Two consequences carry forward: the reranking headroom inside this pool is small, and the pool itself (recall 0.647 at 300 candidates) is the bigger limit.

#### Impact on next steps
Step 7 uses the same features and the same pool; Step 10 compares this model with co-visitation, Two-Tower and DCN-V2 on the same queries; the propensity column is used in Step 8.

#### Out of scope
- A larger reranker training set: days 0-7 could train the rerankers if the Two-Tower similarity were dropped for good.
- Deeper tuning: a modest equal budget is the guide's rule for a learning project; the gains of tuning were small.
- The Two-Tower as a feature: revisit only if a better Two-Tower exists.

## Step 7. DCN-V2 with MMoE
*Kind: selection. Issue #129.* DCN-V2 cross layers feed a multi-gate mixture of experts with click, cart and order towers (D13); the loss is a weighted sum of per-task binary cross-entropy. Same features and same pool as LightGBM. Options: cross layers, experts, task loss weights, blend weights (OTTO 0.10, 0.30, 0.60 by default, tuned on the tuning part). Ablations: single-task models and a shared-bottom multi-task model, each with the same budget. Training runs as a script with a config file, in its own process.

#### Why this step exists here
DCN-V2 with MMoE is the top rung of the complexity ladder and a live-path component (D13). It must earn its place against LightGBM and co-visitation, so it is trained on the same features and the same pool, with the same modest tuning budget.

#### Options considered
- **Architecture:** DCN-V2 cross layers into a multi-gate mixture of experts with click, cart and order towers; the loss is a weighted sum of per-task binary cross-entropy.
- **Options with the same budget (six random configurations each):** cross layers 1-3, experts 2-8, expert width, learning rate, task loss weights 1/1/1 or 1/2/4.
- **Blend weights:** OTTO's 0.10, 0.30, 0.60 against a simplex grid tuned on the tuning part.
- **Ablations, same budget:** single-task models (one per head) and a shared-bottom multi-task model, the baseline that MMoE is meant to beat.
- **Compute:** CPU training time against the one-hour target (D21).

In [ ]:
DCN_FE = FE_ALL + ["tt_missing"]
dcn_cfg = {"spec": SPEC_KEY, "seed": 42, "pool": NPOOL, "epochs": 4, "trials": 6, "features": DCN_FE, "log_columns": ["covis_max", "covis_mean", "covis_wsum", "item_cat_size", "prefix_len", "prefix_n_cart", "prefix_n_order", "hours_since_last_seen"],
           "train_table": "data/modeling/features_rerank_train.parquet", "validation_table": "data/modeling/features_validation.parquet", "tune_sessions": "data/modeling/tune_sessions.json", "out": "data/modeling/dcn"}
(MDIR / "tune_sessions.json").write_text(json.dumps(sorted(int(s) for s in tune_sess))); (ROOT / "configs" / "dcn.json").write_text(json.dumps(dcn_cfg, indent=2) + "\n")
dres = ROOT / dcn_cfg["out"] / "results.json"
if not (dres.exists() and json.loads(dres.read_text())["config"] == dcn_cfg):
    r = subprocess.run([sys.executable, "scripts/train_dcn.py", "configs/dcn.json"], cwd=ROOT, capture_output=True, text=True, check=False); print(r.stdout[-1800:]); assert r.returncode == 0, r.stderr[-2000:]
dc = json.loads(dres.read_text()); print("rows", dc["rows"], "| peak memory", round(dc["peak_memory_gb"], 1), "GB")
tr7 = pl.DataFrame(dc["trials"]); save_table("step7_trials", tr7); print(tr7.group_by("family").agg(pl.col("ndcg10_tuning").max().alias("best_tuning"), pl.col("ndcg10_tuning").mean().alias("mean_tuning"), pl.col("train_seconds").mean().alias("mean_train_seconds")))

In [ ]:
# score the validation queries: blend the three task probabilities (OTTO default and the weights tuned on the tuning part), same scoring sessions as every other method
OTTO_W = tuple(dc["results"]["blend"]["otto_default"]); TUNED_W = tuple(dc["results"]["blend"]["best_weights"])
def dcn_scored(fam_key, w):
    p = pl.read_parquet(ROOT / dcn_cfg["out"] / f"validation_predictions_{fam_key}.parquet"); p = p.filter(pl.col("session").is_in(list(score_sess))).sort("session", maintain_order=True)
    sc = p.select(["p_click", "p_cart", "p_order"]).to_numpy() @ np.array(w); lists = rank_lists(p.select("session", "aid"), sc)
    return evaluate(evalq, lambda q, lists=lists: lists[q["session"]][:TOP]).sort("session")
fam_key = {"MMoE": "MMoE", "shared-bottom multi-task": "shared_bottom", "single-task (one model per head)": "single_task"}
rows, per7 = [], {}
for fam, key in fam_key.items():
    for wname, w in (("OTTO weights", OTTO_W), ("tuned weights", TUNED_W)):
        res = dcn_scored(key, w); per7[(fam, wname)] = res; s = summarize(res); d = paired_bootstrap_diff(res["ndcg10"].to_numpy(), cov6["ndcg10"].to_numpy(), n_boot=500, seed=SEED); dl = pdiff_lgb = paired_bootstrap_diff(res["ndcg10"].to_numpy(), per6[best_row["variant"]]["ndcg10"].to_numpy(), n_boot=500, seed=SEED)
        rows.append({"family": fam, "blend": wname, "weights": str(tuple(round(x, 2) for x in w)), "ndcg10": s["ndcg10"][0], "ci": f"[{s['ndcg10'][1]:.4f}, {s['ndcg10'][2]:.4f}]", "recall20": s["recall20"][0], "recall100": s["recall100"][0], "diff_vs_covis": d[0], "diff_vs_covis_ci": f"[{d[1]:+.4f}, {d[2]:+.4f}]", "diff_vs_lgbm": dl[0], "diff_vs_lgbm_ci": f"[{dl[1]:+.4f}, {dl[2]:+.4f}]", "diff_vs_lgbm_high": dl[2]})
tab7 = pl.DataFrame(rows); save_table("step7_dcn", tab7); print(tab7.select("family", "blend", "weights", "ndcg10", "ci", "diff_vs_covis", "diff_vs_lgbm", "diff_vs_lgbm_ci"))
mm = tab7.filter(pl.col("family") == "MMoE").sort("ndcg10", descending=True).row(0, named=True)
d_sb = paired_bootstrap_diff(per7[("MMoE", mm["blend"])]["ndcg10"].to_numpy(), per7[("shared-bottom multi-task", mm["blend"])]["ndcg10"].to_numpy(), n_boot=500, seed=SEED); d_st = paired_bootstrap_diff(per7[("MMoE", mm["blend"])]["ndcg10"].to_numpy(), per7[("single-task (one model per head)", mm["blend"])]["ndcg10"].to_numpy(), n_boot=500, seed=SEED)
d_bl = paired_bootstrap_diff(per7[("MMoE", "tuned weights")]["ndcg10"].to_numpy(), per7[("MMoE", "OTTO weights")]["ndcg10"].to_numpy(), n_boot=500, seed=SEED)
print(f"MMoE minus shared-bottom {d_sb[0]:+.4f} [{d_sb[1]:+.4f}, {d_sb[2]:+.4f}]; MMoE minus single-task {d_st[0]:+.4f} [{d_st[1]:+.4f}, {d_st[2]:+.4f}]; tuned minus OTTO blend {d_bl[0]:+.4f} [{d_bl[1]:+.4f}, {d_bl[2]:+.4f}]")
print("per-head validation AUC (click, cart, order):", {k: [round(x, 3) for x in v] for k, v in dc["head_auc"].items()})

In [ ]:
# CPU training time against the one-hour target (D21); seesaw check on the rare order head
mm_r = dc["results"]["MMoE"]; per_epoch = mm_r["seconds_per_epoch"]; run_min = per_epoch * dcn_cfg["epochs"] / 60; fam_min = {f: r["train_seconds"] / 60 for f, r in dc["results"].items() if f != "blend"}
print(f"MMoE: {per_epoch:.0f} s per epoch, one {dcn_cfg['epochs']}-epoch run about {run_min:.1f} minutes; best-configuration training minutes: " + ", ".join(f"{k} {v:.1f}" for k, v in fam_min.items()))
seesaw_gap = dc["head_auc"]["MMoE"][2] - dc["head_auc"]["single-task (one model per head)"][2]; print(f"order-head AUC, MMoE minus single-task: {seesaw_gap:+.4f} (a clearly negative value would be a seesaw effect; PLE is then the follow-up, not built)")
exp7 = [{"id": f"S7-{r['family']}-{r['blend']}".replace(" ", "_").replace("(", "").replace(")", ""), "step": "7", "description": f"DCN-V2 {r['family']}, {r['blend']}", "features": ", ".join(DCN_FE), "model": "DCN-V2 (PyTorch)", "parameters": json.dumps(dc["results"][r["family"]]["params"]), "seeds": "[42]", "metrics": {"ndcg10": r["ndcg10"], "ndcg10_ci": r["ci"], "diff_vs_covis": r["diff_vs_covis"], "diff_vs_lgbm": r["diff_vs_lgbm"]}, "fit_seconds": dc["results"][r["family"]]["train_seconds"]} for r in tab7.iter_rows(named=True)]
exp7 += [{"id": f"S7-trial-{r['family']}-{r['trial']}".replace(" ", "_").replace("(", "").replace(")", ""), "step": "7", "description": "tuning trial", "model": "DCN-V2", "parameters": r["params"], "seeds": "[42]", "metrics": {"ndcg10_tuning": r["ndcg10_tuning"]}, "fit_seconds": r["train_seconds"]} for r in tr7.iter_rows(named=True)]
log_experiments(exp7)
loses = bool(mm["diff_vs_lgbm_high"] < 0)
(ROOT / "docs" / "decisions" / "D21-training-compute.md").write_text(f"""# D21. Training compute (settled in modelling Steps 4 and 7)

Status: Accepted for Level 0.

## Decision
Level 0 trains everything on the Mac CPU (4 threads): the Two-Tower takes about {per_ep:.0f} s per epoch ({full_run / 60:.1f} minutes for {best_params['epochs']} epochs) and DCN-V2 with MMoE about {per_epoch:.0f} s per epoch ({run_min:.1f} minutes for {dcn_cfg['epochs']} epochs); both are far under the one-hour target, so no spot training is needed for Level 0. Training runs as scripts with a config file (`configs/two_tower.json`, `configs/dcn.json`), each in its own process because PyTorch, FAISS and LightGBM cannot share a process on macOS.

## Why
Measured, not estimated (Steps 4 and 7). Peak memory {dc['peak_memory_gb']:.1f} GB (DCN) and {tt['peak_memory_gb']:.1f} GB (Two-Tower) fit in 8 GB.

## Alternatives considered
Colab or spot training: unnecessary at this size; revisit at Level 1 if the weekly data grows.
""")
e7 = log_step(7, "DCN-V2 with MMoE", "Selection",
    f"DCN-V2 cross layers plus MMoE with click, cart and order towers on the frozen pool and the Step 6 features; families: MMoE, shared-bottom multi-task, single-task (one model per head), each with {dcn_cfg['trials']} random configurations (cross layers 1-3, experts 2-8, expert width, learning rate, task loss weights 1/1/1 or 1/2/4) tuned on the tuning part; blend weights: OTTO 0.10/0.30/0.60 against a simplex grid tuned on the tuning part; scored on the same {len(evalq):,} validation queries.",
    "Scoring part (NDCG@10 [95% interval]; difference to co-visitation; difference to the best LightGBM): " + "; ".join(f"{r['family']} / {r['blend']} {r['ndcg10']:.4f} {r['ci']}, {r['diff_vs_covis']:+.4f}, {r['diff_vs_lgbm']:+.4f} {r['diff_vs_lgbm_ci']}" for r in tab7.iter_rows(named=True)) + f". MMoE minus shared-bottom {d_sb[0]:+.4f} [{d_sb[1]:+.4f}, {d_sb[2]:+.4f}]; MMoE minus single-task {d_st[0]:+.4f} [{d_st[1]:+.4f}, {d_st[2]:+.4f}]; tuned minus OTTO blend {d_bl[0]:+.4f} [{d_bl[1]:+.4f}, {d_bl[2]:+.4f}]. Order-head AUC MMoE minus single-task {seesaw_gap:+.4f}. Training: {per_epoch:.0f} s per epoch, {run_min:.1f} minutes per run.",
    f"Best MMoE: {mm['blend']} (NDCG@10 {mm['ndcg10']:.4f}; against LightGBM {mm['diff_vs_lgbm']:+.4f} {mm['diff_vs_lgbm_ci']}). " + ("STOP-AND-ASK: the live-path component DCN-V2 with MMoE loses to LightGBM beyond noise; reported, not swapped. " if loses else "") + "Blend weights: keep OTTO's unless the tuned weights beat them beyond noise. CPU training is far below the one-hour target (D21, docs/decisions/D21-training-compute.md).",
    "Guide step 7: cross layers, experts, task weights and blend weights compared with the same budget; ablations (single-task, shared-bottom); watch for seesaw effects on orders (PLE is the follow-up, not built); measure CPU training time.",
    "Any variant whose paired interval against the MMoE includes zero is a tie and the simpler one wins on the ladder; PLE is not built.",
    "Medium: one training seed per configuration and six configurations per family.",
    "Step 10 compares DCN-V2 with the other four methods on the same queries; Step 11 exports it to ONNX with a parity check; Step 8 reuses the propensity-weight argument of `train_model`.",
    human=("yes: STOP-AND-ASK (guide): DCN-V2 with MMoE, a live-path component, loses to LightGBM beyond noise (difference " + mm["diff_vs_lgbm_ci"] + " NDCG@10). Reported, not swapped." if loses else "no"))
print(e7)

#### Chosen approach and rationale
**Decision.** Best MMoE: tuned weights (NDCG@10 0.2031; against LightGBM -0.0024 [-0.0035, -0.0014]). STOP-AND-ASK: the live-path component DCN-V2 with MMoE loses to LightGBM beyond noise; reported, not swapped. Blend weights: keep OTTO's unless the tuned weights beat them beyond noise. CPU training is far below the one-hour target (D21, docs/decisions/D21-training-compute.md).

**Why.** Guide step 7: cross layers, experts, task weights and blend weights compared with the same budget; ablations (single-task, shared-bottom); watch for seesaw effects on orders (PLE is the follow-up, not built); measure CPU training time.

**Rejected.** Any variant whose paired interval against the MMoE includes zero is a tie and the simpler one wins on the ladder; PLE is not built.

**Confidence.** Medium: one training seed per configuration and six configurations per family.

#### Interpretation: what we understand from the results
DCN-V2 does not beat the simpler methods. The best MMoE (tuned blend) scores NDCG@10 0.2031 [0.2002, 0.2059]: level with co-visitation (-0.0001) and 0.0024 below LightGBM [-0.0035, -0.0014], a loss beyond noise, so the guide's stop-and-ask applies. Among the neural variants, MMoE beats the shared-bottom model (+0.0010 [+0.0001, +0.0018]) but loses to the single-task models (-0.0010 [-0.0018, -0.0001]); there is no seesaw on the rare order head (order-head AUC 0.907 for MMoE against 0.852 for the single-task order model), so PLE is not needed. The blend weights matter more than the architecture: the tuned weights (0.5 click, 0.5 cart, 0.0 order) gain +0.0018 [+0.0013, +0.0023] over OTTO's, which suggests, as a hypothesis, that the rare and noisy order head adds more noise than signal to the ranking inside this pool. The tuned weights differ from OTTO's, which is a finding to report; the default weights are kept in the frozen bundle only as the documented alternative. The models are tiny (about 6,500 to 21,000 parameters) and train in about a minute, far under the one-hour target, so compute is not the limit; the information in the features is.

#### Impact on next steps
Step 10 compares DCN-V2 with the other four methods on the same queries; Step 11 exports it to ONNX with a parity check; Step 8 reuses the propensity-weight argument of `train_model`.

#### Out of scope
- Deeper or wider networks and PLE: parked by the plan; only if the human asks.
- Whether the live path keeps DCN-V2 given the result: a human decision, asked in the log.

## Step 8. Position-bias study
*Kind: selection. Issue #130.* OTTO has no displayed positions, so position bias is studied on a semi-synthetic simulation with known propensities (D15). A logging policy (popularity plus noise) shows a ranked list; clicks come from a position-based examination model, click probability = examination probability of the position x true relevance of the item, and the true relevance comes from held-out behaviour. A naive click model is compared with inverse-propensity-weighted (IPW) models with clipping and self-normalisation, judged against the known ground truth. Real OTTO labels stay unweighted in the main models.

#### Why this step exists here
Real OTTO clicks carry no displayed positions, so position bias cannot be measured from the data. A controlled simulation with known propensities is the only way to see what naive training loses to position bias and whether inverse-propensity weighting (IPW) recovers it (D15).

#### Options considered
- **Simulator:** a logging policy (popularity plus Gaussian noise) shows the top 20; a click happens with probability (examination probability of position k) x (true relevance from held-out behaviour); the examination probability (1/k)^eta is known exactly.
- **Models compared:** a naive click model, and IPW models with clipping at 2, 5, 10, 20 and none, each self-normalised.
- **Sweeps:** bias strength eta of 0.5, 1 and 2; five seeds; true NDCG@10 against the known relevance on 10,000 validation queries.
- **Checks:** the simulator reproduces its propensities; the weight column is exact in LightGBM and in the PyTorch loss.

In [ ]:
import matplotlib.pyplot as plt
from clicksim import R_BY_LABEL, simulate_clicks

cnt = trt.group_by("session", maintain_order=True).len(); full_sessions = cnt.filter(pl.col("len") == NPOOL)["session"].to_list(); tr8 = trt.filter(pl.col("session").is_in(full_sessions)).sort("session", maintain_order=True)
NQ = tr8.height // NPOOL; X_all = tr8.select(FE_ALL).to_numpy().astype(np.float32); labels = tr8["label"].to_numpy().reshape(NQ, NPOOL).astype(np.int64); popularity = tr8["log_clicks"].to_numpy().reshape(NQ, NPOOL).astype(np.float64)
te8 = va_sorted.head(10_000 * NPOOL); te8 = te8.filter(pl.col("session").is_in(te8.group_by("session").len().filter(pl.col("len") == NPOOL)["session"].to_list()))
X_test = te8.select(FE_ALL).to_numpy().astype(np.float32); gain_test = R_BY_LABEL[te8["label"].to_numpy().reshape(-1, NPOOL)]; pop_test = te8["log_clicks"].to_numpy().reshape(-1, NPOOL)
print(f"simulation training queries {NQ:,} x {NPOOL} candidates; ground-truth test queries {len(gain_test):,}; features {len(FE_ALL)}")
# the simulator reproduces its own propensities: observed click rate at position k divided by the mean relevance shown there equals theta_k
s0 = simulate_clicks(popularity, labels, seed=0, eta=1.0)
chk = pl.DataFrame({"position": s0["position"], "click": s0["click"], "rel": s0["relevance"]}).group_by("position").agg(pl.col("click").mean().alias("ctr"), pl.col("rel").mean().alias("mean_rel")).sort("position").with_columns((pl.col("ctr") / pl.col("mean_rel")).alias("implied_propensity"), (1 / pl.col("position")).alias("theta"))
chk = chk.with_columns(((pl.col("implied_propensity") - pl.col("theta")).abs() / pl.col("theta")).alias("relative_error")); save_table("step8_simulator_check", chk); print(chk.head(8)); print(f"largest relative error over positions 1-20: {chk['relative_error'].max():.1%}")
assert chk.filter(pl.col("position") <= 10)["relative_error"].max() < 0.15, "the simulator does not reproduce the known propensities"

In [ ]:
# ground truth NDCG, the references, and the sweep: bias strength eta x clip x seeds
DISC = 1.0 / np.log2(np.arange(2, 12))
def ndcg_true(scores, gains):
    top = np.argsort(-scores, axis=1, kind="stable")[:, :10]; disc = DISC[: top.shape[1]]; dcg = (np.take_along_axis(gains, top, axis=1) * disc).sum(1); ideal = (-np.sort(-gains, axis=1)[:, : top.shape[1]] * disc).sum(1); return dcg / ideal
g = np.array([[0.9, 0.02, 0.5]]); assert abs(ndcg_true(np.array([[3.0, 1.0, 2.0]]), g)[0] - 1.0) < 1e-12
rng8 = np.random.default_rng(0); random_ndcg = float(ndcg_true(rng8.random(gain_test.shape), gain_test).mean()); logging_ndcg = float(ndcg_true(pop_test, gain_test).mean())
P8 = {"objective": "binary", "learning_rate": 0.05, "num_leaves": 15, "min_data_in_leaf": 100, "feature_fraction": 0.8, "bagging_fraction": 0.8, "bagging_freq": 1, "verbose": -1, "seed": 0, "deterministic": True, "force_col_wise": True, "num_threads": 4}
ref_model = lgb.train(P8, lgb.Dataset(X_all, (labels.reshape(-1) > 0).astype(int)), 150); ref_ndcg = float(ndcg_true(ref_model.predict(X_test).reshape(-1, NPOOL), gain_test).mean())
print(f"random order {random_ndcg:.4f}; logging policy (popularity) {logging_ndcg:.4f}; model trained on the real labels of all candidates {ref_ndcg:.4f} (about 10x more positives than the click data, so not a fair upper bound)")
ETAS, CLIPS, N_SEEDS = [0.5, 1.0, 2.0], [1, 2, 5, 10, 20, 1e9], 5                  # clip 1 = naive (all weights 1); 1e9 = no clipping
def weights_for(click, theta, clip):
    w = np.where(click == 1, np.minimum(1.0 / theta, clip), 1.0); return w / w.mean()             # self-normalisation
def run8():
    rows = []; t0 = time.time()
    for eta in ETAS:
        for seed in range(N_SEEDS):
            sim = simulate_clicks(popularity, labels, seed=seed, eta=eta); idx = sim["query"] * NPOOL + sim["candidate"]; Xs, click, theta = X_all[idx], sim["click"], sim["propensity"].astype(np.float64)
            for clip in CLIPS:
                w = weights_for(click, theta, clip); model = lgb.train(P8, lgb.Dataset(Xs, click, weight=w), 150); sc = model.predict(X_test).reshape(-1, NPOOL)
                rows.append({"eta": eta, "seed": seed, "clip": float(clip), "ndcg_true": float(ndcg_true(sc, gain_test).mean()), "clicks": int(click.sum()), "max_weight": float(w.max()), "weight_sd": float(w.std())})
        print(f"eta {eta} done ({time.time() - t0:.0f} s)")
    return pl.DataFrame(rows)
res8 = cached("step8_sweep", run8)

In [ ]:
# naive versus IPW: paired over seeds, per bias strength; bias against variance
rows = []
for eta in ETAS:
    naive = res8.filter((pl.col("eta") == eta) & (pl.col("clip") == 1.0)).sort("seed")["ndcg_true"].to_numpy()
    for clip in CLIPS:
        v = res8.filter((pl.col("eta") == eta) & (pl.col("clip") == float(clip))).sort("seed")["ndcg_true"].to_numpy(); d = v - naive; t = 2.776 * d.std(ddof=1) / np.sqrt(len(d))    # t(4) 97.5%
        rows.append({"eta": eta, "clip": "1 (naive)" if clip == 1 else "none" if clip >= 1e8 else str(int(clip)), "mean_ndcg_true": v.mean(), "sd_over_seeds": v.std(ddof=1), "variance": v.var(ddof=1), "bias_to_reference": ref_ndcg - v.mean(), "mse_to_reference": (ref_ndcg - v.mean()) ** 2 + v.var(ddof=1),
                     "diff_to_naive": d.mean(), "diff_low": d.mean() - t, "diff_high": d.mean() + t, "seeds_better_than_naive": int((d > 0).sum())})
tab8 = pl.DataFrame(rows); save_table("step8_naive_vs_ipw", tab8); print(tab8.select("eta", "clip", "mean_ndcg_true", "sd_over_seeds", "bias_to_reference", "diff_to_naive", "diff_low", "diff_high", "seeds_better_than_naive"))
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6)); labs = ["1\n(naive)", "2", "5", "10", "20", "none"]
for eta in ETAS:
    p = tab8.filter(pl.col("eta") == eta); axes[0].errorbar(range(len(CLIPS)), p["mean_ndcg_true"], yerr=p["sd_over_seeds"], fmt="o-", capsize=3, label=f"eta = {eta}"); axes[1].plot(p["variance"], p["bias_to_reference"] ** 2, "o-", label=f"eta = {eta}")
    for i, c in enumerate(p["clip"]): axes[1].annotate(c.split()[0], (p["variance"][i], p["bias_to_reference"][i] ** 2), fontsize=7)
axes[0].axhline(logging_ndcg, color="tab:red", ls="--", label="logging policy"); axes[0].axhline(ref_ndcg, color="tab:green", ls="-.", label="real-label reference"); axes[0].set_xticks(range(len(CLIPS)), labs); axes[0].set_xlabel("clip M"); axes[0].set_ylabel("true NDCG@10 of the click model"); axes[0].legend(fontsize=7)
axes[1].set_xlabel("variance over seeds"); axes[1].set_ylabel("squared gap to the reference"); axes[1].set_title("bias against variance (labels = clip M)"); axes[1].legend(fontsize=7); plt.tight_layout(); plt.savefig(ROOT / "modeling" / "figures" / "step8_bias_variance.png", dpi=110); plt.close()
best_ipw = tab8.filter(pl.col("clip") != "1 (naive)").sort("diff_to_naive", descending=True); wins = best_ipw.filter(pl.col("diff_low") > 0)
print("IPW settings whose paired interval over seeds excludes zero (better than naive):", wins.select("eta", "clip", "diff_to_naive").to_dicts() or "none")

In [ ]:
# the propensity column: LightGBM all-ones equals unweighted (Step 6); the PyTorch loss and training path are checked in their own process
out8 = subprocess.run([sys.executable, str(ROOT / "notebooks" / "check_weights_torch.py")], capture_output=True, text=True, check=False); assert out8.returncode == 0, out8.stderr[-1500:]
tres = json.loads([ln for ln in out8.stdout.splitlines() if ln.startswith("RESULT ")][-1][7:]); print("neural loss checks:", tres)
assert abs(tres["loss_plain"] - tres["loss_ones"]) < 1e-6 and tres["train_ones_equals_plain"]
log_experiments([{"id": f"S8-eta{r['eta']}-clip{r['clip']}".replace(" ", "_").replace("(", "").replace(")", ""), "step": "8", "description": f"click model trained on simulated clicks, bias strength eta {r['eta']}, clip {r['clip']}", "features": ", ".join(FE_ALL), "model": "LightGBM binary", "parameters": json.dumps(P8), "seeds": str(list(range(N_SEEDS))), "metrics": {"mean_ndcg_true": r["mean_ndcg_true"], "sd": r["sd_over_seeds"], "diff_to_naive": r["diff_to_naive"], "diff_ci": [r["diff_low"], r["diff_high"]]}} for r in tab8.iter_rows(named=True)])
naive_m = {eta: tab8.filter((pl.col("eta") == eta) & (pl.col("clip") == "1 (naive)"))["mean_ndcg_true"][0] for eta in ETAS}
e8 = log_step(8, "Position-bias study", "Selection",
    f"Semi-synthetic simulator (logging policy: popularity plus Gaussian noise, top 20 shown; click probability = position examination probability (1/k)^eta times true relevance 0.02/0.5/0.7/0.9 for labels 0-3 from held-out behaviour); naive click model against IPW with clipping (M = 1 naive, 2, 5, 10, 20, none) and self-normalisation; bias strength eta in {ETAS}; {N_SEEDS} seeds each; LightGBM binary click model on {len(FE_ALL)} features; true NDCG@10 on {len(gain_test):,} validation queries against the known relevance; simulator check against the known propensities; weight-column exactness checks (LightGBM, PyTorch).",
    f"Simulator: implied propensity equals (1/k)^eta within {chk.filter(pl.col('position') <= 10)['relative_error'].max():.1%} for positions 1-10. References (true NDCG@10): random order {random_ndcg:.4f}, logging policy {logging_ndcg:.4f}, model trained on real labels of all candidates {ref_ndcg:.4f}. Naive click model by eta: " + ", ".join(f"{k} {v:.4f}" for k, v in naive_m.items()) + ". IPW against naive (mean difference, paired 95% interval over seeds), best setting per eta: " + "; ".join(f"eta {eta}: clip {r['clip']} {r['diff_to_naive']:+.4f} [{r['diff_low']:+.4f}, {r['diff_high']:+.4f}]" for eta in ETAS for r in [best_ipw.filter(pl.col("eta") == eta).row(0, named=True)]) + f". Settings whose interval excludes zero: {len(wins)} of {best_ipw.height}. Figure: modeling/figures/step8_bias_variance.png.",
    "Real OTTO labels stay unweighted in the main models (D15). The propensity column defaults to 1 and is verified exact in LightGBM (Step 6) and in the PyTorch loss and training path. IPW is kept as supported machinery, not used in the main pipeline.",
    "Guide step 8: simulate with known propensities, compare naive with IPW (clipping, self-normalisation), sweep bias strength, clipping and seeds, plot bias against variance, keep real labels unweighted, then ask the human.",
    "Training the main pipeline on simulated exposure: not done (asked below). Estimating propensities from data: out of scope (no positions exist in OTTO).",
    "Medium: five seeds per cell; intervals are t-intervals over seeds and small samples widen them.",
    "Nothing in Steps 9-11 changes; the propensity column stays in the feature tables and the training scripts.",
    human="yes: the guide asks after this step whether the main pipeline should train on simulated exposure. Recommendation: no, keep real OTTO labels unweighted. " + ("In this simulation IPW never beat the naive click model beyond noise, so there is nothing to gain even where the bias is known, and OTTO has no real exposure data to weight with." if len(wins) == 0 else "IPW helped at some settings (see the table), but OTTO has no real exposure data to weight with."))
print(e8)

#### Chosen approach and rationale
**Decision.** Real OTTO labels stay unweighted in the main models (D15). The propensity column defaults to 1 and is verified exact in LightGBM (Step 6) and in the PyTorch loss and training path. IPW is kept as supported machinery, not used in the main pipeline.

**Why.** Guide step 8: simulate with known propensities, compare naive with IPW (clipping, self-normalisation), sweep bias strength, clipping and seeds, plot bias against variance, keep real labels unweighted, then ask the human.

**Rejected.** Training the main pipeline on simulated exposure: not done (asked below). Estimating propensities from data: out of scope (no positions exist in OTTO).

**Confidence.** Medium: five seeds per cell; intervals are t-intervals over seeds and small samples widen them.

#### Interpretation: what we understand from the results
The bias hurts: the naive click model scores a true NDCG@10 of 0.513, 0.469 and 0.452 as eta goes from 0.5 to 1 to 2, against 0.429 for the logging policy's own order and 0.565 for a model trained on the real labels of all candidates (random order 0.396). IPW does not repair it in this setting. At eta = 0.5 every clip level is worse than naive (clip 2: -0.0065 [-0.0114, -0.0015]; no clipping: -0.0222); at eta = 1 clip 2 is +0.0030 [-0.0042, +0.0101] and at eta = 2 it is +0.0011 [-0.0071, +0.0093], both ties, and larger clips are worse or tied. None of the 15 IPW settings beats naive beyond noise. The bias-variance picture points to the likely reason (a hypothesis, not tested further): weights up to 1/theta_k multiply the few clicks that were made at low positions, so the variance over seeds rises about threefold at eta = 2 without clipping (sd 0.013 against 0.004) while the true NDCG does not improve, so the extra variance outweighs any bias removed at this data size. Clipping at 2 is the least harmful setting (a tie at eta 1 and 2, a small loss at eta 0.5). The propensity column is exact (all-ones weights give the unweighted model bit for bit) and stays in the pipeline. Real OTTO labels stay unweighted, and the human is asked whether the main pipeline should train on simulated exposure; the recommendation is no.

#### Impact on next steps
Nothing in Steps 9-11 changes; the propensity column stays in the feature tables and the training scripts.

#### Out of scope
- Estimating propensities from data (no positions exist in OTTO).
- A doubly-robust or counterfactual ranking estimator: parked.

## Step 9. Claude roles
*Kind: selection. Issue #131.* Classifier (D17), explainer (D18) and judge (D19) all run on Bedrock, and this machine has no AWS access, no Bedrock and no Anthropic key. This step therefore does everything that needs no model call: the pass criteria are fixed before any evaluation, the classifier baseline is measured, the explainer's faithfulness test and templated fallback are built and tested, the DCN-V2 attribution methods are compared, and the judge's statistics and the hand-labelling sheet are prepared. The Claude evaluations themselves are recorded as blocked.

#### Why this step exists here
The classifier, explainer and judge (D16-D19) run on Bedrock, and they need an evaluation each with a pass criterion fixed in advance. This machine has no AWS access, no Bedrock and no Anthropic key, so this step does everything that needs no model call and records the rest as blocked, instead of simulating results.

#### Options considered
- **Criteria first:** written before anything is evaluated (docs/decisions/D17-D19-role-criteria.md).
- **Classifier baseline:** a TF-IDF character n-gram prototype classifier (category name plus dev queries), temperature and out-of-scope threshold tuned on half of the dev part, scored on the held-out part.
- **Explainer:** a numeric payload, a templated fallback, a faithfulness test that checks every number and feature-group name against the payload (with tests); attribution for DCN-V2, leave-one-feature-group-out against integrated gradients, on sanity checks, agreement, stability and latency.
- **Judge:** agreement statistics (weighted kappa, Spearman) and bias checks (position, verbosity) with tests, and a 180-row hand-labelling sheet.

In [ ]:
# A. pass criteria, written before anything is evaluated
(ROOT / "docs" / "decisions" / "D17-D19-role-criteria.md").write_text("""# D17-D19. Pass criteria for the Claude roles (written before evaluation; Level 0)

Status: Accepted (criteria only; the evaluations are blocked until Bedrock or API access exists).

## Rule for every role
Pick the smallest Bedrock model that passes the role's criteria (order: claude-haiku-4-5, then claude-sonnet-5), keep the LLM only if it beats the non-LLM baseline by the stated margin, cache every call by the D19 keys, and log calls, tokens and dollars in modeling/bedrock_costs.csv.

## Classifier (D17), on the held-out part of the classifier evaluation set (artifacts/classifier_queries_v2.jsonl)
- Top-3 accuracy at least 0.05 above the TF-IDF prototype baseline (below), and top-1 accuracy at least 0.05 above it.
- Expected calibration error at most 0.10 after calibration on the dev part.
- Out-of-scope F1 at least the baseline's.
- Below the confidence threshold: blend the top categories (decided from the calibrated confidence histogram in Step 10).

## Explainer (D18)
- The faithfulness test (numbers and feature-group names cited must appear in the payload) passes on 100 percent of recorded fixtures; any unfaithful output falls back to the templated explanation.
- Attribution: the method with the higher removal-test lift; if within 10 percent, the lower latency.

## Judge (D19)
- Weighted kappa and Spearman against the human labels both at least 0.60 on the 150-200 labelled rows; position effect (range of mean score across presentation slots) at most 0.30; verbosity correlation (absolute Spearman) at most 0.30.
- The judge mainly measures category coherence with the session, not true product relevance; that limit is stated wherever judge scores appear.
""")
print("criteria written to docs/decisions/D17-D19-role-criteria.md")

In [ ]:
# B. classifier baseline (no LLM): TF-IDF prototype classifier; prototypes from the category name plus dev queries, temperature and out-of-scope threshold tuned on the other half of the dev part
import itertools

from sklearn.feature_extraction.text import TfidfVectorizer

cq2 = [json.loads(ln) for ln in (ROOT / "artifacts" / "classifier_queries_v2.jsonl").read_text().splitlines()]; dev_q = [r for r in cq2 if r["split"] == "dev"]; test_q = [r for r in cq2 if r["split"] == "test"]
cat_names_list = [cname[c] for c in range(K)]; perm = np.random.default_rng(SEED).permutation(len(dev_q)); half_a = [dev_q[i] for i in perm[: len(dev_q) // 2]]; half_b = [dev_q[i] for i in perm[len(dev_q) // 2:]]
def build_protos(rows):
    vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4), sublinear_tf=True).fit(cat_names_list + [r["query"] for r in rows]); P = []
    for c in range(K):
        texts = [cat_names_list[c]] + [r["query"] for r in rows if c in r["true_cluster_ids"] and not r["out_of_scope"]]; v = np.asarray(vec.transform(texts).mean(0)).ravel(); P.append(v / (np.linalg.norm(v) + 1e-12))
    return vec, np.stack(P)
def probs_of(vec, P, rows, T):
    S = (vec.transform([r["query"] for r in rows]) @ P.T); S = S.toarray() if hasattr(S, "toarray") else S; Z = S / T; Z -= Z.max(1, keepdims=True); E = np.exp(Z); return E / E.sum(1, keepdims=True)
def evaluate_clf(prob, rows, thr):
    ins = [i for i, r in enumerate(rows) if not r["out_of_scope"]]; top = np.argsort(-prob, axis=1)[:, :3]; true = [set(r["true_cluster_ids"]) for r in rows]
    t1 = np.mean([top[i, 0] in true[i] for i in ins]); t3 = np.mean([bool(set(top[i]) & true[i]) for i in ins]); conf = prob.max(1); ok = np.array([top[i, 0] in true[i] for i in range(len(rows))])
    ece = 0.0; edges = np.linspace(0, 1, 11)
    for lo, hi in itertools.pairwise(edges):
        m = np.array([(lo < conf[i] <= hi) for i in ins]);
        if m.sum(): ece += m.mean() * abs(np.mean([ok[i] for i, k in zip(ins, m, strict=True) if k]) - np.mean([conf[i] for i, k in zip(ins, m, strict=True) if k]))
    flag = conf < thr; oos = np.array([r["out_of_scope"] for r in rows]); prec = float((flag & oos).sum() / max(flag.sum(), 1)); rec = float((flag & oos).sum() / max(oos.sum(), 1)); f1 = 2 * prec * rec / max(prec + rec, 1e-12)
    return {"top1": float(t1), "top3": float(t3), "ece": float(ece), "oos_precision": prec, "oos_recall": rec, "oos_f1": f1}
vec_a, P_a = build_protos(half_a); best_T, best_ll = None, 1e9
for T in (0.02, 0.05, 0.1, 0.2, 0.5):
    pr = probs_of(vec_a, P_a, [r for r in half_b if not r["out_of_scope"]], T); ll = -np.mean([np.log(sum(pr[i, c] for c in r["true_cluster_ids"]) + 1e-9) for i, r in enumerate([r for r in half_b if not r["out_of_scope"]])])
    if ll < best_ll: best_T, best_ll = T, ll
pb = probs_of(vec_a, P_a, half_b, best_T); thr = max(np.linspace(0.05, 0.9, 35), key=lambda t: evaluate_clf(pb, half_b, t)["oos_f1"])
vec_f, P_f = build_protos(dev_q); base_res = evaluate_clf(probs_of(vec_f, P_f, test_q, best_T), test_q, thr)
clf_tab = pl.DataFrame([{"classifier": f"TF-IDF prototype baseline (held-out part, {len(test_q)} queries)", **base_res, "temperature": best_T, "oos_threshold": float(thr)}]); save_table("step9_classifier_baseline", clf_tab); print(clf_tab)
print("sentence-embedding baseline: not run (needs a model download and the sentence-transformers package; left for when the LLM evaluation runs)")

In [ ]:
# C. explainer: faithfulness tests, templated fallback, and the DCN-V2 attribution comparison (own process)
r = subprocess.run([sys.executable, "-m", "pytest", "tests/test_explain.py", "tests/test_judge.py", "-q", "--no-header", "--color=no"], cwd=ROOT, capture_output=True, text=True, check=False); print("faithfulness and judge-statistics tests:", r.stdout.strip().splitlines()[-1]); assert r.returncode == 0
r = subprocess.run([sys.executable, "scripts/attribution.py"], cwd=ROOT, capture_output=True, text=True, check=False); assert r.returncode == 0, r.stderr[-1500:]
att = json.loads((ROOT / dcn_cfg["out"] / "attribution.json").read_text()); save_table("step9_attribution", pl.DataFrame([{k: v for k, v in att.items() if k != "groups"}])); print({k: (round(v, 3) if isinstance(v, float) else v) for k, v in att.items() if k != "groups"})
attr_choice = "leave-one-group-out" if (att["removal_lift_logo"] >= att["removal_lift_ig"] * 1.10 or (abs(att["removal_lift_logo"] - att["removal_lift_ig"]) <= 0.10 * max(att["removal_lift_logo"], att["removal_lift_ig"]) and att["latency_ms_logo"] <= att["latency_ms_ig"])) else "integrated gradients"
print("attribution method chosen by the pre-stated rule (higher removal-test lift; within 10 percent, lower latency):", attr_choice)

In [ ]:
# D. judge: labelling sheet for the human (rows = one method's top 10 for one validation pair; method identity hidden, order shuffled); items are shown by their synthetic category name
rng9 = np.random.default_rng(SEED); pairs = [evalq[i] for i in rng9.choice(len(evalq), 60, replace=False)]; sel = {q["session"] for q in pairs}
vs9 = va_sorted.filter(pl.col("session").is_in(list(sel))); lg = rank_lists(vs9, predict(best_model, vs9, best_feats))
dp = pl.read_parquet(ROOT / dcn_cfg["out"] / "validation_predictions_MMoE.parquet").filter(pl.col("session").is_in(list(sel))).sort("session", maintain_order=True); dl = rank_lists(dp.select("session", "aid"), dp.select(["p_click", "p_cart", "p_order"]).to_numpy() @ np.array(TUNED_W))
def cat_label(a): c = item_cluster.get(int(a)); return cname[c] if c is not None else "(no category)"
LISTS9 = {"co-visitation": lambda q: FNS["co-visitation (tuned)"](q)[:10], "BM25": lambda q: FNS["BM25 (tuned)"](q)[:10], "Two-Tower": lambda q: TT_L[q["session"]][:10], "LightGBM": lambda q: lg[q["session"]][:10], "DCN-V2 + MMoE": lambda q: dl[q["session"]][:10]}
allrows = [{"session": q["session"], "method": m, "prefix": " > ".join(cat_label(a) for a in q["prefix_list"][-5:]), "query_category": cname[q["category"]], "results": " | ".join(cat_label(a) for a in fn(q))} for q in pairs for m, fn in LISTS9.items()]
order9 = rng9.permutation(len(allrows))[:180]; sheet = pl.DataFrame([{"row_id": i, "prefix_last5_categories": allrows[j]["prefix"], "query_category": allrows[j]["query_category"], "top10_result_categories": allrows[j]["results"], "score_0_to_4": ""} for i, j in enumerate(order9)])
key = pl.DataFrame([{"row_id": i, "session": allrows[j]["session"], "method": allrows[j]["method"]} for i, j in enumerate(order9)])
sheet.write_csv(ROOT / "modeling" / "artifacts" / "judge_labeling_sheet.csv"); key.write_csv(ROOT / "modeling" / "artifacts" / "judge_labeling_key.csv"); print(sheet.head(3)); print(f"labelling sheet: {sheet.height} rows from {len(pairs)} validation pairs x 5 methods; key file kept separate")

In [ ]:
blocked = pl.DataFrame([
    {"item": "classifier: Claude top-3 with confidences, calibration on the dev part, comparison with the baseline", "status": "blocked", "needs": "Bedrock or Anthropic API access"},
    {"item": "classifier: sentence-embedding baseline", "status": "not run", "needs": "sentence-transformers and a model download"},
    {"item": "explainer: Claude explanations, live faithfulness run", "status": "blocked", "needs": "Bedrock or Anthropic API access"},
    {"item": "judge: about 1,500 batch calls, weighted kappa and Spearman against hand labels, position and verbosity checks", "status": "blocked", "needs": "API access and the human's 150-200 labels (modeling/artifacts/judge_labeling_sheet.csv)"},
    {"item": "Bedrock model availability in the chosen region, cost alarm", "status": "blocked", "needs": "AWS region and credentials"}]); save_table("step9_blocked", blocked); print(blocked)
log_experiments([{"id": "S9-classifier-tfidf", "step": "9", "description": "TF-IDF prototype classifier baseline on the held-out classifier set", "model": "TF-IDF char n-grams, nearest prototype", "parameters": json.dumps({"temperature": best_T, "oos_threshold": float(thr)}), "seeds": "[42]", "metrics": base_res},
                 {"id": "S9-attribution", "step": "9", "description": "DCN-V2 attribution: leave-one-group-out vs integrated gradients", "model": "DCN-V2 + MMoE", "seeds": "[0]", "metrics": {k: v for k, v in att.items() if k != "groups"}}])
e9 = log_step(9, "Claude roles", "Selection",
    "Done locally (no model call): pass criteria fixed before evaluation (docs/decisions/D17-D19-role-criteria.md); TF-IDF prototype classifier baseline on the held-out classifier set; explainer numeric payload, templated fallback and faithfulness test with tests; DCN-V2 attribution comparison (leave-one-feature-group-out against integrated gradients on 1,000 top results); judge statistics (weighted kappa, Spearman, position and verbosity checks) with tests; a 180-row hand-labelling sheet. NOT done (blocked): every Claude call (classifier, explainer, judge), calibration on the dev part, the sentence-embedding baseline, kappa against hand labels, Bedrock availability and cost.",
    f"Classifier baseline on {len(test_q)} held-out queries: top-1 {base_res['top1']:.3f}, top-3 {base_res['top3']:.3f}, expected calibration error {base_res['ece']:.3f}, out-of-scope precision {base_res['oos_precision']:.3f}, recall {base_res['oos_recall']:.3f}, F1 {base_res['oos_f1']:.3f} (temperature {best_T}, threshold {thr:.2f}); so the Claude classifier must reach top-3 of at least {base_res['top3'] + 0.05:.3f} and top-1 of at least {base_res['top1'] + 0.05:.3f}. Attribution: removal-test lift LOGO {att['removal_lift_logo']:.2f}, integrated gradients {att['removal_lift_ig']:.2f}; IG completeness error {att['ig_completeness_median_relative_error']:.3f}; agreement (Spearman) {att['agreement_spearman']:.2f}; stability under noise LOGO {att['stability_logo']:.2f}, IG {att['stability_ig']:.2f}; latency per result LOGO {att['latency_ms_logo']:.2f} ms, IG {att['latency_ms_ig']:.2f} ms. Faithfulness and judge-statistics tests pass.",
    f"Attribution method for the explainer: {attr_choice} (rule fixed in advance). Pass criteria for the three roles are fixed. The Claude evaluations cannot run here: they stay open (issue #131) and the classifier, explainer and judge are unresolved.",
    "Guide step 9 order: criteria first, baselines, faithfulness test with fixtures and templated fallback, attribution comparison, judge preparation. Everything that needs no model call was done so that the evaluations can run in one pass once access exists.",
    "Running the LLM roles with a simulated client: rejected, it would produce numbers that mean nothing. Building the judge calls before the labelling sheet is labelled: rejected, calibration against the human is the point.",
    "High for what was done; the LLM decisions are open, not low-confidence.",
    "Step 10 uses the TF-IDF classifier as a stand-in to measure how category errors propagate through the live path (labelled as a stand-in); judge scores and the NDCG-versus-judge disagreement analysis wait for API access and the human labels.",
    human="yes: (1) AWS region and Bedrock or Anthropic API access, (2) the human hand-labels the sheet modeling/artifacts/judge_labeling_sheet.csv (180 rows, score 0-4), (3) confirm the classifier pass criteria in docs/decisions/D17-D19-role-criteria.md")
print(e9)

#### Chosen approach and rationale
**Decision.** Attribution method for the explainer: leave-one-group-out (rule fixed in advance). Pass criteria for the three roles are fixed. The Claude evaluations cannot run here: they stay open (issue #131) and the classifier, explainer and judge are unresolved.

**Why.** Guide step 9 order: criteria first, baselines, faithfulness test with fixtures and templated fallback, attribution comparison, judge preparation. Everything that needs no model call was done so that the evaluations can run in one pass once access exists.

**Rejected.** Running the LLM roles with a simulated client: rejected, it would produce numbers that mean nothing. Building the judge calls before the labelling sheet is labelled: rejected, calibration against the human is the point.

**Confidence.** High for what was done; the LLM decisions are open, not low-confidence.

#### Interpretation: what we understand from the results
The non-LLM classifier is a strong bar: top-1 0.764, top-3 0.843, expected calibration error 0.038 and out-of-scope F1 0.626 on 2,077 held-out queries, so the Claude classifier must reach a top-3 of at least 0.893 and a top-1 of at least 0.814 to be worth its cost and latency. For attribution, leave-one-group-out and integrated gradients agree (Spearman 0.81) and pass the same sanity checks: removing the group each method calls most important moves the score 1.8 times more than removing a random group, integrated gradients complete to within 4.5%, and both are stable under small noise (0.98-0.99); leave-one-group-out is chosen by the pre-stated rule because it is as good and about nine times faster (0.01 ms against 0.09 ms per result). The faithfulness test accepts explanations that cite the payload, rejects invented numbers and invented feature groups, and the templated explanation passes it by construction. What cannot be done is every Claude call: the classifier, the explainer and the judge evaluations, calibration on the dev part, the sentence-embedding baseline and kappa against your labels are blocked on API access and on your 180 labels.

#### Impact on next steps
Step 10 uses the TF-IDF classifier as a stand-in to measure how category errors propagate through the live path (labelled as a stand-in); judge scores and the NDCG-versus-judge disagreement analysis wait for API access and the human labels.

#### Out of scope
- All Claude evaluations and the cost log (blocked).
- The sentence-embedding baseline (needs a model download).
- Judge scores on the benchmark: also wait for Step 11, because the benchmark lives in the test slice.

## Step 10. Comparison and analysis
*Kind: diagnostic. Issue #132.* All five methods are scored on the same validation sessions with the same metric code. The test slice is not touched here: it is scored once, in Step 11, after everything is frozen (so the judge scores on the benchmark, which lives in the test slice, also wait for Step 11). Ties between methods go to the simpler one on the ladder: co-visitation, LightGBM, Two-Tower, DCN-V2 with MMoE (BM25 is the floor).

#### Why this step exists here
Until now each method was measured in its own step against the baseline. The comparison step puts all five on the same sessions with the same metric code, so the reader can see the ladder in one table, find where each method wins or fails, and see whether any component of the live path is not earning its place.

#### Options considered
- **Comparison:** the five methods on the same validation sessions, paired bootstrap between methods, and the ladder tie rule (a tie goes to the simpler method).
- **Slices:** prefix length, popularity of the held-out item (head, torso, tail), and category, plus the popularity bias of the top 10.
- **Ablations:** every ablation of Steps 4 to 8 in one table.
- **Live path with a classifier in the loop:** the true category against a category predicted by the TF-IDF baseline of Step 9 (a stand-in, not Claude).
- **Cost:** latency and size per component.

In [ ]:
# A. five methods, same sessions, same metric code
five = {"BM25": res3.filter(pl.col("method") == "BM25 (tuned)").sort("session"), "co-visitation": res3.filter(pl.col("method") == "co-visitation (tuned)").sort("session"), "Two-Tower": per[CHOSEN],
        "LightGBM LambdaMART": per6[best_row["variant"]], "DCN-V2 + MMoE": per7[("MMoE", mm["blend"])]}
sess_ref = five["co-visitation"]["session"].to_list(); assert all(v["session"].to_list() == sess_ref for v in five.values()), "the methods are not scored on the same sessions"
KIND = {"BM25": "retrieval-style", "co-visitation": "retrieval-style", "Two-Tower": "retrieval-style", "LightGBM LambdaMART": "reranker (frozen pool)", "DCN-V2 + MMoE": "reranker (frozen pool)"}
rows = []
for m, r in five.items():
    s = summarize(r); rows.append({"method": m, "kind": KIND[m], "ndcg10": s["ndcg10"][0], "ndcg10_ci": f"[{s['ndcg10'][1]:.4f}, {s['ndcg10'][2]:.4f}]", "recall20": s["recall20"][0], "recall50": s["recall50"][0], "recall100": s["recall100"][0], "recall200": s["recall200"][0], "wrecall20": s["wrecall20"][0]})
five_tab = pl.DataFrame(rows); save_table("step10_five_methods", five_tab); print(five_tab); print(f"{len(sess_ref):,} validation sessions; the rerankers rerank the frozen pool ({COMP}, {NPOOL}), so their recall@K is inside that pool")

In [ ]:
# B. paired bootstrap between methods; ties go to the simpler method on the ladder
LADDER = ["co-visitation", "LightGBM LambdaMART", "Two-Tower", "DCN-V2 + MMoE"]; nd = {m: five[m]["ndcg10"].to_numpy() for m in five}
best_m = max(LADDER, key=lambda m: nd[m].mean()); tie = []; pair_rows = []
for m in LADDER:
    d = paired_bootstrap_diff(nd[m], nd[best_m], n_boot=500, seed=SEED) if m != best_m else (0.0, 0.0, 0.0); pair_rows.append({"method": m, "diff_to_best": d[0], "ci_low": d[1], "ci_high": d[2], "tie_with_best": bool(d[1] <= 0 <= d[2])})
    if pair_rows[-1]["tie_with_best"]: tie.append(m)
winner = min(tie, key=LADDER.index); pair_tab = pl.DataFrame(pair_rows); save_table("step10_ties", pair_tab); print(pair_tab)
print(f"best by NDCG@10: {best_m}; methods tied with it (interval of the difference includes zero): {tie}; winner by the ladder: {winner}")
d_ld = paired_bootstrap_diff(nd["LightGBM LambdaMART"], nd["co-visitation"], n_boot=500, seed=SEED); d_dl = paired_bootstrap_diff(nd["DCN-V2 + MMoE"], nd["LightGBM LambdaMART"], n_boot=500, seed=SEED); d_tc = paired_bootstrap_diff(nd["Two-Tower"], nd["co-visitation"], n_boot=500, seed=SEED)
print(f"LightGBM minus co-visitation {d_ld[0]:+.4f} [{d_ld[1]:+.4f}, {d_ld[2]:+.4f}]; DCN-V2 minus LightGBM {d_dl[0]:+.4f} [{d_dl[1]:+.4f}, {d_dl[2]:+.4f}]; Two-Tower minus co-visitation {d_tc[0]:+.4f} [{d_tc[1]:+.4f}, {d_tc[2]:+.4f}]")

In [ ]:
# C. slices: prefix length, popularity of the held-out click (head / torso / tail), and category; popularity bias of the top 10
vqf = pl.read_parquet(MDIR / "ranking_queries_validation.parquet").select("session", pl.col("label_aid").list.first().alias("target"), pl.col("prefix_aid").list.len().alias("plen")).filter(pl.col("session").is_in(sess_ref)).sort("session")
pop_rank = {int(a): i for i, a in enumerate(clicks["aid"].to_list())}; tr_ = vqf["target"].to_list(); bucket_pop = np.array(["head (top 1,000)" if pop_rank.get(int(a), 10**9) < 1000 else "torso (1,000-20,000)" if pop_rank.get(int(a), 10**9) < 20_000 else "tail or unseen" for a in tr_])
pl_ = vqf["plen"].to_numpy(); bucket_len = np.array(["1" if n == 1 else "2-3" if n <= 3 else "4-10" if n <= 10 else "11+" for n in pl_]); qs_sorted = sorted(evalq, key=lambda q: q["session"]); cats_ = np.array([q["category"] for q in qs_sorted])
def slice_table(kind, labels_):
    out = []
    for lab in sorted(set(labels_)):
        m_ = labels_ == lab
        for meth in five: ci = bootstrap_ci(nd[meth][m_], n_boot=200, seed=SEED); out.append({"slice_type": kind, "slice": str(lab), "queries": int(m_.sum()), "method": meth, "ndcg10": ci[0], "ci_low": ci[1], "ci_high": ci[2]})
    return pl.DataFrame(out)
sl = pl.concat([slice_table("prefix length", bucket_len), slice_table("popularity of the held-out click", bucket_pop)]); save_table("step10_slices", sl)
sl_wide = sl.pivot(on="method", index=["slice_type", "slice", "queries"], values="ndcg10").sort("slice_type", "slice"); print(sl_wide)
cat_tab = pl.DataFrame([{"category": cname[int(c)], "queries": int((cats_ == c).sum()), **{m: float(nd[m][cats_ == c].mean()) for m in five}} for c in sorted(set(cats_.tolist()))]); save_table("step10_slices_by_category", cat_tab)
big = cat_tab.filter(pl.col("queries") >= 100); print("weakest categories (co-visitation, at least 100 queries):", big.sort("co-visitation").head(3).select("category", "queries", "co-visitation", "LightGBM LambdaMART").to_dicts()); print("strongest:", big.sort("co-visitation", descending=True).head(3).select("category", "queries", "co-visitation", "LightGBM LambdaMART").to_dicts())
head_set = set(list(pop_rank)[:1000]); sub_q = qs_sorted[:6000]; sub_ids = [q["session"] for q in sub_q]
share_rel = float(np.mean([np.mean([int(a) in head_set for a in set().union(*q["labels"].values())]) for q in sub_q]))
lgb_full = va_sorted.filter(pl.col("session").is_in(sub_ids)); lgb_lists = rank_lists(lgb_full, predict(best_model, lgb_full, best_feats))
dcn_p = pl.read_parquet(ROOT / dcn_cfg["out"] / "validation_predictions_MMoE.parquet").filter(pl.col("session").is_in(sub_ids)).sort("session", maintain_order=True); W_USED = np.array(TUNED_W if mm["blend"] == "tuned weights" else OTTO_W)
dcn_lists = rank_lists(dcn_p.select("session", "aid"), dcn_p.select(["p_click", "p_cart", "p_order"]).to_numpy() @ W_USED)
LISTS10 = {"BM25": lambda q: FNS["BM25 (tuned)"](q), "co-visitation": lambda q: FNS["co-visitation (tuned)"](q), "Two-Tower": lambda q: TT_L[q["session"]], "LightGBM LambdaMART": lambda q: lgb_lists[q["session"]], "DCN-V2 + MMoE": lambda q: dcn_lists[q["session"]]}
head_share = {m: float(np.mean([np.mean([int(a) in head_set for a in fn(q)[:10]]) for q in sub_q])) for m, fn in LISTS10.items()}
print(f"share of top-1,000 popular items: among the relevant items {share_rel:.3f}; in the top 10 of each method: " + ", ".join(f"{k} {v:.3f}" for k, v in head_share.items()))

In [ ]:
# D. all ablations of Steps 4-8 in one place
def rd(name): return pl.read_csv(ROOT / "modeling" / "tables" / f"{name}.csv")
abl = []
for r in rd("step4_two_tower_same_sessions").iter_rows(named=True): abl.append({"step": 4, "ablation": r["variant"], "ndcg10": r["ndcg10"], "difference": f"{r['diff_vs_covis']:+.4f} vs co-visitation {r['diff_ci']}"})
for r in rd("step5_pool_vs_best").filter(pl.col("n") == NPOOL).iter_rows(named=True): abl.append({"step": 5, "ablation": f"pool: {r['option']}, {r['n']} candidates", "ndcg10": None, "difference": f"recall {r['recall']:.4f} ({r['diff_to_best']:+.4f} vs best)"})
for r in rd("step6_lightgbm").iter_rows(named=True): abl.append({"step": 6, "ablation": r["variant"], "ndcg10": r["ndcg10"], "difference": f"{r['diff_vs_covis']:+.4f} vs co-visitation {r['diff_ci']}"})
for r in rd("step7_dcn").iter_rows(named=True): abl.append({"step": 7, "ablation": f"{r['family']} / {r['blend']}", "ndcg10": r["ndcg10"], "difference": f"{r['diff_vs_lgbm']:+.4f} vs LightGBM {r['diff_vs_lgbm_ci']}"})
for r in rd("step8_naive_vs_ipw").iter_rows(named=True): abl.append({"step": 8, "ablation": f"click model, eta {r['eta']}, clip {r['clip']}", "ndcg10": r["mean_ndcg_true"], "difference": f"{r['diff_to_naive']:+.4f} vs naive"})
abl_tab = pl.DataFrame(abl, schema_overrides={"ndcg10": pl.Float64}); save_table("step10_ablations", abl_tab); print(abl_tab.height, "ablation rows written to modeling/tables/step10_ablations.csv")

In [ ]:
# E. live path with a classifier in the loop (STAND-IN: the TF-IDF baseline classifier of Step 9, not Claude): how do category errors propagate?
rng10 = np.random.default_rng(SEED); live_q = [qs_sorted[i] for i in rng10.choice(len(qs_sorted), 2000, replace=False)]
pool_txt = {}
for r_ in test_q + dev_q:
    if not r_["out_of_scope"] and r_["kind"] in ("paraphrase", "misspelling") and len(r_["true_cluster_ids"]) == 1: pool_txt.setdefault(r_["true_cluster_ids"][0], []).append(r_["query"])
live_q = [q for q in live_q if q["category"] in pool_txt]; texts = [{"query": pool_txt[q["category"]][int(rng10.integers(len(pool_txt[q["category"]])))], "out_of_scope": False, "true_cluster_ids": [q["category"]]} for q in live_q]
pred_prob = probs_of(vec_f, P_f, texts, best_T); pred_cat = pred_prob.argmax(1); acc = float(np.mean([p == q["category"] for p, q in zip(pred_cat, live_q, strict=True)])); print(f"{len(live_q):,} live queries; the stand-in classifier gets the category right for {acc:.1%}")
m_noTT, _ = fit_lgb("lambdarank", FE_NO_TT, json.loads(tun6.filter(pl.col("variant").str.contains("without")).sort("ndcg10_tuning", descending=True)["params"][0]), 300)
asv = {d: AsOf(wk12, day_to_ms(d), cat, M, IDS) for d in (10, 11)}; rk10 = CovisRanker(M, IDS, item_cluster, fill_by_cat=FILL, last_n=best_c["last_n"], decay=best_c["decay"], top=NPOOL, weights=(1.0, best_c["w_cart"], best_c["w_order"]))
def live_ndcg(q, category):
    qq = {**q, "category": int(category)}; cvl = rk10.rank(qq); pool = build_pool(qq["prefix"], {"covis": cvl}, cap=NPOOL, rrf_k=60); a = asv[(cutoff_of(q) - START_MS) // DAY_MS]
    f = a.query_features(qq, pool); X = np.stack([f[c] for c in FE_NO_TT], 1).astype(np.float32); order_ = np.argsort(-m_noTT.predict(X), kind="stable"); ranked = [int(f["aid"][i]) for i in order_]
    return ndcg_at_k(cvl, q["labels"], k=10), ndcg_at_k(ranked, q["labels"], k=10)
res_true = np.array([live_ndcg(q, q["category"]) for q in live_q]); res_pred = np.array([live_ndcg(q, p) for q, p in zip(live_q, pred_cat, strict=True)])
right = np.array([p == q["category"] for p, q in zip(pred_cat, live_q, strict=True)])
live_tab = pl.DataFrame([{"path": "co-visitation", "true category": res_true[:, 0].mean(), "predicted category (TF-IDF stand-in)": res_pred[:, 0].mean(), "correct-category queries": res_pred[right, 0].mean(), "wrong-category queries": res_pred[~right, 0].mean()},
                         {"path": "LightGBM (no Two-Tower features)", "true category": res_true[:, 1].mean(), "predicted category (TF-IDF stand-in)": res_pred[:, 1].mean(), "correct-category queries": res_pred[right, 1].mean(), "wrong-category queries": res_pred[~right, 1].mean()}])
save_table("step10_live_path_standin", live_tab); print(live_tab); drop_rel = (res_true[:, 1].mean() - res_pred[:, 1].mean()) / res_true[:, 1].mean(); print(f"classifier accuracy {acc:.1%}; NDCG@10 falls by {drop_rel:.1%} (LightGBM path) when the category comes from the stand-in classifier")

In [ ]:
# F. latency, sizes; Bedrock cost is blocked
sel10 = [q for q in live_q[:400] if (cutoff_of(q) - START_MS) // DAY_MS == 10][:200]; tf = time.time(); [asv[10].query_features(q, build_pool(q["prefix"], {"covis": rk10.rank(q)}, cap=NPOOL, rrf_k=60)) for q in sel10]; feat_ms = (time.time() - tf) / max(len(sel10), 1) * 1000
Xp = np.random.default_rng(0).random((NPOOL, len(FE_NO_TT))).astype(np.float32); tp = time.time(); [m_noTT.predict(Xp) for _ in range(200)]; lgb_ms = (time.time() - tp) / 200 * 1000
lr_ = subprocess.run([sys.executable, "scripts/latency_dcn.py"], cwd=ROOT, capture_output=True, text=True, check=False); assert lr_.returncode == 0, lr_.stderr[-1000:]; lat_dcn = json.loads(lr_.stdout.strip().splitlines()[-1])
tt_dir = ROOT / next(r["dir"] for r in tt["results"] if r["variant"] == CHOSEN); tt_mb = (tt_dir / "model.pt").stat().st_size / 1e6
lat_tab = pl.DataFrame([{"component": "co-visitation candidates (pool)", "ms_per_query": t_cv, "size_mb": (MDIR / "covis_train_days0_9.npz").stat().st_size / 1e6},
    {"component": "point-in-time features (shared implementation)", "ms_per_query": feat_ms, "size_mb": None}, {"component": "LightGBM scoring of the pool", "ms_per_query": lgb_ms, "size_mb": float(tab6["size_mb"].min())},
    {"component": "DCN-V2 + MMoE scoring of the pool (1 thread, p95)", "ms_per_query": lat_dcn["p95_ms_1_threads"], "size_mb": lat_dcn["file_mb"]}, {"component": "Two-Tower query tower and exact item search", "ms_per_query": t_tt + flat_row["p95_ms"], "size_mb": tt_mb + flat_row["size_mb"]},
    {"component": "Bedrock classifier and explainer (cost per query)", "ms_per_query": None, "size_mb": None}]); save_table("step10_latency_size", lat_tab); print(lat_tab)
print("Bedrock cost per query: not measurable (no access); the cost log modeling/bedrock_costs.csv is empty")

In [ ]:
flags = []
if d_tc[2] < 0: flags.append("Two-Tower loses to co-visitation beyond noise")
if d_dl[2] < 0: flags.append("DCN-V2 with MMoE loses to LightGBM beyond noise")
if d_dl[1] <= 0 <= d_dl[2]: flags.append("DCN-V2 with MMoE ties LightGBM: the simpler LightGBM wins on the ladder")
log_experiments([{"id": f"S10-{r['method']}".replace(" ", "_"), "step": "10", "description": f"five-method comparison on {len(sess_ref):,} validation sessions: {r['method']}", "model": r["method"], "seeds": "[42]", "metrics": {"ndcg10": r["ndcg10"], "ci": r["ndcg10_ci"], "recall20": r["recall20"], "recall100": r["recall100"], "recall200": r["recall200"]}} for r in five_tab.iter_rows(named=True)])
slice_txt = "; ".join(f"{r['slice_type']} {r['slice']}: {r['co-visitation']:.3f} / {r['LightGBM LambdaMART']:.3f}" for r in sl_wide.iter_rows(named=True))
e10 = log_step(10, "Comparison and analysis", "Diagnostic",
    f"Five methods scored on the same {len(sess_ref):,} validation sessions with the same metric code (NDCG@10 with 95% bootstrap intervals, recall@20/50/100/200, OTTO weighted recall@20); paired bootstrap between methods and the ladder tie rule; slices by prefix length, popularity of the held-out click and category; popularity bias of the top 10; the ablations of Steps 4-8 in one table; the live path with a stand-in classifier (TF-IDF, not Claude) on {len(live_q):,} queries; latency and sizes. NOT done (blocked): judge scores on the benchmark (the benchmark lives in the test slice, so it waits for Step 11 anyway, and the judge needs API access), the NDCG-versus-judge disagreement analysis, the live path with the real Claude classifier, Bedrock cost per query.",
    "NDCG@10 [95% interval]: " + "; ".join(f"{r['method']} {r['ndcg10']:.4f} {r['ndcg10_ci']}" for r in five_tab.iter_rows(named=True)) + f". Paired differences: LightGBM minus co-visitation {d_ld[0]:+.4f} [{d_ld[1]:+.4f}, {d_ld[2]:+.4f}]; DCN-V2 minus LightGBM {d_dl[0]:+.4f} [{d_dl[1]:+.4f}, {d_dl[2]:+.4f}]; Two-Tower minus co-visitation {d_tc[0]:+.4f} [{d_tc[1]:+.4f}, {d_tc[2]:+.4f}]. Best {best_m}; tied with it: {', '.join(tie)}; winner by the ladder: {winner}. Slices (NDCG@10, co-visitation / LightGBM): {slice_txt}. Popularity bias: the share of top-1,000 popular items is {share_rel:.2f} among relevant items and " + ", ".join(f"{k} {v:.2f}" for k, v in head_share.items()) + f" in the top 10. Live path with the TF-IDF stand-in classifier ({acc:.1%} correct): LightGBM path NDCG@10 {res_true[:, 1].mean():.4f} with the true category, {res_pred[:, 1].mean():.4f} with the predicted one; queries with a wrong category score {res_pred[~right, 1].mean():.4f}. Latency per query: candidates {t_cv:.2f} ms, features {feat_ms:.1f} ms, LightGBM {lgb_ms:.1f} ms, DCN-V2 {lat_dcn['p95_ms_1_threads']:.1f} ms (p95, one thread).",
    f"Report: {winner} is the method to beat on the ladder. " + ("; ".join(flags) + ". " if flags else "") + "No component is swapped silently; stop-and-ask items for live-path components are listed under 'needs human input'.",
    "Guide step 10: score all five methods with intervals, recall@K for the retrieval-style ones, slices, popularity bias, the ablation tables, the live path with a classifier, the cost measures, an honest report.",
    "Judge-based comparison and the disagreement analysis: blocked, not simulated. Scoring the test slice here: rejected, it is used once in Step 11.",
    "High for the ranking of methods (paired intervals on 50,000+ sessions); Medium for the stand-in live-path result (a TF-IDF classifier is not the Claude classifier).",
    "Step 11 freezes the models and scores the test slice once with the same code; the disagreement analysis and the judge columns of the frontend exports stay empty until API access exists.",
    human=("yes: STOP-AND-ASK (guide): " + "; ".join(f_ for f_ in flags if "loses" in f_) + ". Reported, not swapped; a decision is needed on whether the live path keeps these components." if any("loses" in f_ for f_ in flags) else "no"))
print(e10)

#### Chosen approach and rationale
**Decision.** Report: LightGBM LambdaMART is the method to beat on the ladder. Two-Tower loses to co-visitation beyond noise; DCN-V2 with MMoE loses to LightGBM beyond noise. No component is swapped silently; stop-and-ask items for live-path components are listed under 'needs human input'.

**Why.** Guide step 10: score all five methods with intervals, recall@K for the retrieval-style ones, slices, popularity bias, the ablation tables, the live path with a classifier, the cost measures, an honest report.

**Rejected.** Judge-based comparison and the disagreement analysis: blocked, not simulated. Scoring the test slice here: rejected, it is used once in Step 11.

**Confidence.** High for the ranking of methods (paired intervals on 50,000+ sessions); Medium for the stand-in live-path result (a TF-IDF classifier is not the Claude classifier).

#### Interpretation: what we understand from the results
The ranking of methods is clear: LightGBM LambdaMART 0.2054, co-visitation 0.2032, DCN-V2 with MMoE 0.2031, Two-Tower 0.0537 and BM25 0.0003. The gain from learning is small (LightGBM +0.0022 over co-visitation, about 1% relative), and the Two-Tower is far behind, so the extra machinery mostly does not pay. The Two-Tower recommends more popular items (0.484 of its top 10 are among the 1,000 most popular items against 0.382 for co-visitation) and is close to useless on tail items (0.006), which is where a retrieval model should help most. Co-visitation, LightGBM and DCN-V2 all score highest on head items. A wrong category is costly: with the stand-in classifier at 78.5% accuracy, NDCG@10 falls by 23.5% on the LightGBM path, so the classifier accuracy target of Step 9 matters more than any of the model gaps. The slice results are descriptive: they show where methods differ, not why.

#### Impact on next steps
Step 11 freezes the models and scores the test slice once with the same code; the disagreement analysis and the judge columns of the frontend exports stay empty until API access exists.

#### Out of scope
- Judge scores on the benchmark (need API access and the human labels, and the benchmark lives in the test slice).
- The live path with the real Claude classifier and Bedrock cost per query: blocked.
- Any change to the live path: the human decides after the stop-and-ask items.

## Step 11. Freeze and Level 0 exit
*Kind: diagnostic. Issue #133.* Order matters: (1) the expected test range is written from validation, (2) the pipeline is frozen, (3) the test slice is scored exactly once (a guard file makes a rerun reload the recorded result instead of scoring again), (4) the bundle, the ONNX exports with a parity check, the feature parity test and the frontend exports are produced. Uploading the bundle to S3 by hand needs AWS access and stays open.

#### Why this step exists here
This is where Level 0 ends. The test slice is a one-time check, so everything that could change a model must be frozen before it is opened, and the range of scores that would be unsurprising must be written down before the result is known. Otherwise a surprise can be explained away afterwards.

#### Options considered
- **Expected range:** validation NDCG@10 plus or minus two bootstrap standard deviations, written to a file before scoring (the range covers session sampling error only).
- **Freeze:** pipeline, hyper-parameters, feature lists, seeds, data version and gains in one config file.
- **Scoring:** the test slice scored once for all five methods, guarded by a file so a rerun reloads the recorded result.
- **Bundle:** models, ONNX exports of DCN-V2 and the Two-Tower query tower with an output parity check, the exact item index, co-visitation matrix, item feature tables, pool spec, prompts and a manifest with hashes.
- **Feature parity test:** the training path against a serving path rebuilt from the bundle.

In [ ]:
import shutil

ART = ROOT / "modeling" / "artifacts"; BUNDLE = ROOT / "data" / "modeling" / "bundle_v1"; BUNDLE.mkdir(parents=True, exist_ok=True); EXPORTS = ROOT / "data" / "modeling" / "exports"; EXPORTS.mkdir(parents=True, exist_ok=True)
def sha(path): return hashlib.sha256(Path(path).read_bytes()).hexdigest()
# 1. the expected range of the test-slice NDCG@10, from validation (mean +/- 2 standard deviations of the bootstrap distribution), written BEFORE any test scoring
exp_path = ART / "expected_test_range.json"; guard = ART / "test_scored.json"
if not exp_path.exists():
    ranges = {}
    for m, r in five.items():
        mean_, lo_, hi_ = summarize(r)["ndcg10"]; sd_ = (hi_ - lo_) / 3.92; ranges[m] = {"validation_ndcg10": mean_, "expected_low": mean_ - 2 * sd_, "expected_high": mean_ + 2 * sd_}
    exp_path.write_text(json.dumps({"written_before_test_scoring": True, "snapshot_id": snapshot_id(), "commit_sha": commit_sha(), "ranges": ranges}, indent=1))
expected = json.loads(exp_path.read_text())["ranges"]; print("expected test range (NDCG@10) from validation:", {m: [round(v["expected_low"], 4), round(v["expected_high"], 4)] for m, v in expected.items()})
assert not guard.exists() or exp_path.stat().st_mtime <= guard.stat().st_mtime, "the expected range must be written before the test slice is scored"

In [ ]:
# 2. freeze: the pipeline, hyper-parameters, feature lists, weights, seeds and data version
lg_row = tab6.sort("ndcg10", descending=True).row(0, named=True); lg_kind = "binary" if lg_row["variant"].startswith("binary") else "lambdarank"; lg_feats = FE_NO_TT if "without" in lg_row["variant"] else FE_ALL
final_lgb, _ = fit_lgb(lg_kind, lg_feats, json.loads(lg_row["params"]), 300); final_lgb.save_model(str(BUNDLE / "lightgbm.txt"))
frozen = {"snapshot_id": snapshot_id(), "commit_sha": commit_sha(), "seed": SEED, "split": "docs/decisions/D3-split-rule.md", "gains": {"click": 1, "cart": 2, "order": 3}, "categories": {"k": K, "file": "artifacts/category_names_v2.json"},
          "candidate_pool": json.loads((ROOT / "artifacts" / "candidate_pool.json").read_text()), "covis_baseline_params": best_c, "bm25_params": best_b,
          "two_tower": {"variant": CHOSEN, "config": cfg_hash(cfg)}, "lightgbm": {"variant": lg_row["variant"], "params": json.loads(lg_row["params"]), "features": lg_feats}, "dcn_mmoe": {"params": dc["results"]["MMoE"]["params"], "blend_weights": [float(x) for x in W_USED], "features": dcn_cfg["features"]},
          "feature_schema_sha256": hashlib.sha256(json.dumps({"lgbm": lg_feats, "dcn": dcn_cfg["features"]}).encode()).hexdigest()}
(ART / "frozen_config.json").write_text(json.dumps(frozen, indent=1)); print("frozen:", {k: (v if not isinstance(v, dict) else "...") for k, v in frozen.items()})

In [ ]:
# 3. the test slice, scored ONCE (a rerun of the notebook reloads the recorded result)
import os

DRY = os.environ.get("STEP11_DRY") == "1"                                   # dry run: the same code on 2,000 VALIDATION queries, nothing is written to the guard, then stop
if guard.exists() and not DRY:
    test_rec = json.loads(guard.read_text()); print("the test slice was already scored once; reloading the recorded result from", guard.name)
else:
    tq_path = MDIR / ("ranking_queries_validation.parquet" if DRY else "ranking_queries_test.parquet"); tprep, tstats = prepare_eval(pl.read_parquet(tq_path)); tag = "dry" if DRY else "test"
    print(f"{'DRY RUN on validation queries' if DRY else 'test queries'}: {len(tprep):,}"); qvt = MDIR / f"{tag}_query_vectors.npy"
    r = subprocess.run([sys.executable, "scripts/tt_encode.py", str(chosen_dir), str(tq_path), str(qvt)], cwd=ROOT, capture_output=True, text=True, check=False); assert r.returncode == 0, r.stderr[-1500:]; qv_te = np.load(qvt)
    if DRY: tprep, qv_te = tprep[:2000], qv_te[:2000]
    tt_te = tt_topn(qv_te, tprep, 200); build_table(tprep, qv_te, M, IDS, FILL, MDIR / f"features_{tag}.parquet", f"{tag} table")
    tes = pl.read_parquet(MDIR / f"features_{tag}.parquet").sort("session", maintain_order=True); print(f"test table {tes.height:,} rows")
    lists = {"BM25": {q["session"]: FNS["BM25 (tuned)"](q) for q in tprep}, "co-visitation": {q["session"]: FNS["co-visitation (tuned)"](q) for q in tprep}, "Two-Tower": tt_te}
    lg_sc = np.concatenate([predict(final_lgb, tes[s:s + 3_000_000], lg_feats) for s in range(0, tes.height, 3_000_000)]); lists["LightGBM LambdaMART"] = rank_lists(tes, lg_sc)
    r = subprocess.run([sys.executable, "scripts/score_dcn.py", str(MDIR / f"features_{tag}.parquet"), str(MDIR / f"{tag}_dcn_predictions.parquet")], cwd=ROOT, capture_output=True, text=True, check=False); assert r.returncode == 0, r.stderr[-1500:]
    dp_te = pl.read_parquet(MDIR / f"{tag}_dcn_predictions.parquet").sort("session", maintain_order=True); lists["DCN-V2 + MMoE"] = rank_lists(dp_te.select("session", "aid"), dp_te.select(["p_click", "p_cart", "p_order"]).to_numpy() @ W_USED)
    test_frames = {}
    for m, lst in lists.items(): test_frames[m] = evaluate(tprep, lambda q, lst=lst: lst[q["session"]][:TOP]).sort("session"); test_frames[m].write_parquet(MDIR / f"{tag}_results_{m.split()[0].replace('-', '_')}.parquet")
    rows = {}
    for m, fr in test_frames.items():
        s = summarize(fr); rows[m] = {"ndcg10": s["ndcg10"][0], "ci": [s["ndcg10"][1], s["ndcg10"][2]], "recall20": s["recall20"][0], "recall50": s["recall50"][0], "recall100": s["recall100"][0], "recall200": s["recall200"][0], "wrecall20": s["wrecall20"][0], "in_expected_range": bool(expected[m]["expected_low"] <= s["ndcg10"][0] <= expected[m]["expected_high"])}
    nd_te = {m: fr["ndcg10"].to_numpy() for m, fr in test_frames.items()}; diffs = {}
    for a_, b_ in (("LightGBM LambdaMART", "co-visitation"), ("DCN-V2 + MMoE", "LightGBM LambdaMART"), ("Two-Tower", "co-visitation"), ("DCN-V2 + MMoE", "co-visitation")): diffs[f"{a_} minus {b_}"] = list(paired_bootstrap_diff(nd_te[a_], nd_te[b_], n_boot=500, seed=SEED))
    test_rec = {"queries": len(tprep), "methods": rows, "paired_differences": diffs, "scored_once": True, "snapshot_id": snapshot_id(), "commit_sha": commit_sha()}
    if DRY:
        print({m: round(v["ndcg10"], 4) for m, v in rows.items()}, {k: [round(x, 4) for x in v] for k, v in diffs.items()}); raise SystemExit("dry run on validation queries finished: the pipeline code works; nothing was scored on the test slice")
    guard.write_text(json.dumps(test_rec, indent=1))
test_tab = pl.DataFrame([{"method": m, "test_ndcg10": v["ndcg10"], "test_ci": f"[{v['ci'][0]:.4f}, {v['ci'][1]:.4f}]", "expected_low": expected[m]["expected_low"], "expected_high": expected[m]["expected_high"], "validation_ndcg10": expected[m]["validation_ndcg10"], "in_range": v["in_expected_range"], "recall20": v["recall20"], "recall100": v["recall100"], "recall200": v["recall200"]} for m, v in test_rec["methods"].items()])
save_table("step11_test_results", test_tab); print(test_tab); print({k: [round(x, 4) for x in v] for k, v in test_rec["paired_differences"].items()}); out_of_range = [m for m, v in test_rec["methods"].items() if not v["in_expected_range"]]; print("methods outside the expected range:", out_of_range or "none")

In [ ]:
# 4. the bundle: models, ONNX exports with output parity, the item index, item and category tables, prompts and configuration, manifest
t0 = time.time(); shutil.copy(ROOT / cfg["out"] / next(r["dir"] for r in tt["results"] if r["variant"] == CHOSEN).split("/")[-1] / "model.pt", BUNDLE / "two_tower.pt"); shutil.copy(ROOT / dcn_cfg["out"] / "mmoe_best.pt", BUNDLE / "dcn_mmoe.pt")
tt_src = ROOT / next(r["dir"] for r in tt["results"] if r["variant"] == CHOSEN); shutil.copy(tt_src / "item_embeddings.npy", BUNDLE / "item_embeddings.npy"); shutil.copy(tt_src / "catalog_aids.npy", BUNDLE / "catalog_aids.npy"); shutil.copy(tt_src / "faiss" / "flat.index", BUNDLE / "item_index_flat.faiss")
shutil.copy(MDIR / "covis_train_days0_9.npz", BUNDLE / "covis_matrix.npz"); shutil.copy(MDIR / "covis_train_days0_9_ids.npy", BUNDLE / "covis_item_ids.npy"); shutil.copy(MDIR / "categories" / "item_category.parquet", BUNDLE / "item_category.parquet")
shutil.copy(ROOT / "artifacts" / "category_names_v2.json", BUNDLE / "category_names.json"); shutil.copy(ROOT / "artifacts" / "candidate_pool.json", BUNDLE / "candidate_pool.json"); shutil.copy(ROOT / "configs" / "llm.json", BUNDLE / "llm_config.json"); shutil.copy(ROOT / "notebooks" / "llm.py", BUNDLE / "llm_prompts_and_client.py")
shutil.copy(ART / "frozen_config.json", BUNDLE / "frozen_config.json"); shutil.copy(ROOT / "configs" / "dcn.json", BUNDLE / "dcn_config.json"); shutil.copy(ROOT / "configs" / "two_tower.json", BUNDLE / "two_tower_config.json")
asof14 = AsOf(wk12, day_to_ms(14), cat, M, IDS); asof14.export_state(BUNDLE / "item_features_asof_day14.npz")
r = subprocess.run([sys.executable, "scripts/export_onnx.py", str(BUNDLE), str(tt_src)], cwd=ROOT, capture_output=True, text=True, check=False); print(r.stdout[-500:]); assert r.returncode == 0, r.stderr[-1500:]
parity = json.loads((BUNDLE / "onnx_parity.json").read_text()); print("ONNX parity (maximum absolute difference against PyTorch):", {k: v for k, v in parity.items() if k.endswith("diff")})
files = sorted(f for f in BUNDLE.iterdir() if f.is_file()); manifest = {"bundle": "bundle_v1", "snapshot_id": snapshot_id(), "commit_sha": commit_sha(), "created_seconds": round(time.time() - t0), "feature_schema_sha256": frozen["feature_schema_sha256"],
    "model_versions": {"lightgbm": lg_row["variant"], "dcn_mmoe": "MMoE, " + json.dumps(dc["results"]["MMoE"]["params"]), "two_tower": CHOSEN}, "metrics_test": {m: {"ndcg10": v["ndcg10"], "ci": v["ci"]} for m, v in test_rec["methods"].items()}, "onnx_parity": {k: v for k, v in parity.items() if k.endswith("diff")},
    "files": [{"name": f.name, "mb": round(f.stat().st_size / 1e6, 2), "sha256": sha(f)} for f in files]}
(ART / "bundle_manifest.json").write_text(json.dumps(manifest, indent=1)); print(f"bundle: {len(files)} files, {sum(f.stat().st_size for f in files) / 1e6:.0f} MB, assembled in {time.time() - t0:.0f} s; manifest modeling/artifacts/bundle_manifest.json")

In [ ]:
# 5. feature parity test: the training path and the serving path (rebuilt from the bundle's item tables) must give identical features on a fixture of 200 test queries
tprep_fx, _ = prepare_eval(pl.read_parquet(MDIR / "ranking_queries_test.parquet")); fx = [q for q in tprep_fx if (cutoff_of(q) - START_MS) // DAY_MS == 12][:200]
a_train = AsOf(wk12, day_to_ms(12), cat, M, IDS); a_train.export_state(BUNDLE / "fixture_asof_day12.npz"); a_serve = AsOf.from_state(BUNDLE / "fixture_asof_day12.npz", sparse.load_npz(BUNDLE / "covis_matrix.npz").tocsr(), item_cluster)
rk_fx = CovisRanker(M, IDS, item_cluster, fill_by_cat=FILL, last_n=best_c["last_n"], decay=best_c["decay"], top=NPOOL, weights=(1.0, best_c["w_cart"], best_c["w_order"])); mismatch = 0; checked = 0
for q in fx:
    pool = build_pool(q["prefix"], {"covis": rk_fx.rank(q)}, cap=NPOOL, rrf_k=60); f1, f2 = a_train.query_features(q, pool), a_serve.query_features(q, pool)
    for k in f1: mismatch += int(not np.array_equal(f1[k], f2[k], equal_nan=True)); checked += 1
(BUNDLE / "fixture_asof_day12.npz").unlink(); print(f"feature parity on {len(fx)} fixture queries: {checked:,} feature columns compared, {mismatch} differ"); assert mismatch == 0, "the serving path does not reproduce the training features"

In [ ]:
# 6. frontend exports (aggregates only are committed; per-query results stay in data/modeling/exports because the licence for derived data is unclear)
val_rows = five_tab.select("method", "ndcg10", "ndcg10_ci", "recall20", "recall100", "recall200").to_dicts()
(ART / "exports").mkdir(exist_ok=True); (ART / "exports" / "metrics_v1.json").write_text(json.dumps({"version": "v1", "snapshot_id": snapshot_id(), "commit_sha": commit_sha(), "gains": frozen["gains"], "validation": val_rows, "test": test_tab.select("method", "test_ndcg10", "test_ci", "in_range", "recall20", "recall100", "recall200").to_dicts(),
    "paired_differences_test": test_rec["paired_differences"], "judge_scores": None, "notes": "categories are synthetic; the ranking query is a documented leak (D5); judge scores are empty until API access exists"}, indent=1))
bench = pl.read_parquet(MDIR / "benchmark_v2.parquet"); brow = []
for m in test_rec["methods"]:
    fr = pl.read_parquet(MDIR / f"test_results_{m.split()[0].replace('-', '_')}.parquet").filter(pl.col("session").is_in(bench["session"].to_list())).select("session", pl.col("ndcg10").alias(f"ndcg10_{m}")); brow.append(fr)
bres = bench.select("benchmark_id", "session", "query_category", "prefix_bucket"); [bres := bres.join(f, on="session") for f in brow]; bres.write_csv(EXPORTS / "benchmark_results_v1.csv"); print(f"benchmark results for {bres.height} pairs written to data/modeling/exports/ (not committed)")
bm = {m: float(bres[f"ndcg10_{m}"].mean()) for m in test_rec["methods"]}; print("benchmark mean NDCG@10:", {k: round(v, 4) for k, v in bm.items()})
man = ROOT / "modeling" / "manual_steps.md"; txt = man.read_text()
if "Upload the frozen bundle to S3" not in txt: man.write_text(txt + "| 11 | Upload the frozen bundle to S3 by hand and time it | pending (needs AWS access; the bundle is ready in data/modeling/bundle_v1) |\n")

In [ ]:
# 7. investigation of the methods outside the expected range (observational: nothing is re-scored, nothing is tuned; the co-visitation results are split by day)
dq = {n: pl.read_parquet(MDIR / f"ranking_queries_{n}.parquet").select("session", "prefix_ts_last") for n in ("validation", "test")}
byday = {}
for name, fr in (("validation", five["co-visitation"]), ("test", pl.read_parquet(MDIR / "test_results_co_visitation.parquet"))):
    d = fr.select("session", "ndcg10").join(dq[name], on="session").with_columns(((pl.col("prefix_ts_last") - START_MS) // DAY_MS).alias("day")); g = d.group_by("day").agg(pl.len().alias("queries"), pl.col("ndcg10").mean().alias("ndcg10")).sort("day"); byday.update({int(r["day"]): (r["queries"], r["ndcg10"]) for r in g.to_dicts()})
print("co-visitation NDCG@10 by day of the query (matrix fixed at days 0-9):", {d: (n_, round(v, 4)) for d, (n_, v) in byday.items()})
gaps = {m: v["ndcg10"] - expected[m]["validation_ndcg10"] for m, v in test_rec["methods"].items()}; print("test minus validation NDCG@10 per method:", {m: round(v, 4) for m, v in gaps.items()})
day_spread = max(v for _, v in byday.values()) - min(v for _, v in byday.values()); print(f"spread of the daily co-visitation NDCG@10: {day_spread:.4f}")
inv = f"Investigation (observational, nothing re-scored): every method with a signal is {-max(g_ for m_, g_ in gaps.items() if m_ != 'BM25'):.4f} to {-min(g_ for m_, g_ in gaps.items() if m_ != 'BM25'):.4f} below its validation value, the range was built from session sampling error alone, and the daily co-visitation NDCG@10 moves between {min(v for _, v in byday.values()):.4f} and {max(v for _, v in byday.values()):.4f} across the four days (spread {day_spread:.4f}, larger than the sampling interval); the ordering of the methods and the paired gaps are unchanged (LightGBM minus co-visitation {test_rec['paired_differences']['LightGBM LambdaMART minus co-visitation'][0]:+.4f} on test against {expected['LightGBM LambdaMART']['validation_ndcg10'] - expected['co-visitation']['validation_ndcg10']:+.4f} on validation). The shift is a day-level level effect, possibly with some staleness of the fixed day 0-9 matrix; the data cannot separate the two. It is not a defect of the pipeline, and the range should be widened by day-to-day spread in Level 1."

In [ ]:
tm = test_rec["methods"]; diffs = test_rec["paired_differences"]
log_experiments([{"id": f"S11-test-{m}".replace(" ", "_"), "step": "11", "description": f"frozen {m} on the test slice (scored once)", "model": m, "seeds": "[42]", "metrics": {"test_ndcg10": v["ndcg10"], "ci": v["ci"], "recall20": v["recall20"], "recall100": v["recall100"], "in_expected_range": v["in_expected_range"]}, "notes": "test slice used once"} for m, v in tm.items()])
e11 = log_step(11, "Freeze and Level 0 exit", "Diagnostic",
    "Expected test range written from validation (mean +/- 2 bootstrap standard deviations) before scoring; pipeline frozen (docs and artifacts/frozen_config.json); test slice scored once for all five methods (guard file); bundle assembled (LightGBM, DCN-V2 and Two-Tower weights, ONNX exports of DCN-V2 and the Two-Tower query tower with parity check, exact item index, co-visitation matrix, item feature tables as of day 14, category map, pool spec, prompts and configuration, manifest with hashes); feature parity test between the training path and the serving path rebuilt from the bundle; aggregate frontend exports. NOT done (blocked): the manual upload of the bundle to S3 (no AWS access) and the judge scores for the frontend.",
    f"Test slice ({test_rec['queries']:,} queries, scored once) NDCG@10 [95% interval] against the expected range: " + "; ".join(f"{m} {v['ndcg10']:.4f} [{v['ci'][0]:.4f}, {v['ci'][1]:.4f}] ({'in' if v['in_expected_range'] else 'OUTSIDE'} the range {expected[m]['expected_low']:.4f}-{expected[m]['expected_high']:.4f})" for m, v in tm.items()) + ". Paired differences on test: " + "; ".join(f"{k} {v[0]:+.4f} [{v[1]:+.4f}, {v[2]:+.4f}]" for k, v in diffs.items()) + f". ONNX parity: DCN-V2 {parity['dcn_mmoe_max_abs_diff']:.2e}, Two-Tower query tower {parity['two_tower_query_max_abs_diff']:.2e} (limit 1e-4). Feature parity: {checked:,} feature columns on {len(fx)} fixture queries, {mismatch} differ. Bundle: {len(files)} files, {sum(f.stat().st_size for f in files) / 1e6:.0f} MB. Benchmark mean NDCG@10: " + ", ".join(f"{k} {v:.4f}" for k, v in bm.items()) + ".",
    "Frozen and scored once; the test slice is now spent for these models: any change from here on needs a new slice. The bundle is ready for the manual S3 upload; the manifest records hashes, model versions, feature schema hash, snapshot id and commit SHA. " + ("Methods outside the expected range: " + ", ".join(out_of_range) + ": nothing was tuned on the test slice; " + inv if out_of_range else "Every method landed inside its expected range."),
    "Guide step 11: expected range first, then one scoring; bundle with ONNX parity; feature parity test; upload by hand and time it; frontend exports.",
    "Re-scoring or tuning after the test result: rejected, it would spend the slice. Committing per-query exports: rejected, the licence for derived data is Unclear.",
    "High for the numbers (paired intervals on the full test slice) and for the parity checks; the S3 upload is open.",
    "Step 12 summarises; Level 1 starts from this bundle, the split rule (arrival_slice), the frozen gains and the needs-review list.",
    human="yes: AWS access to upload the bundle to S3 by hand and time it (manual-steps log), plus the open stop-and-ask items listed in the summary")
print(e11)

#### Chosen approach and rationale
**Decision.** Frozen and scored once; the test slice is now spent for these models: any change from here on needs a new slice. The bundle is ready for the manual S3 upload; the manifest records hashes, model versions, feature schema hash, snapshot id and commit SHA. Methods outside the expected range: co-visitation, Two-Tower, LightGBM LambdaMART, DCN-V2 + MMoE: nothing was tuned on the test slice; see the log for the investigation.

**Why.** Guide step 11: expected range first, then one scoring; bundle with ONNX parity; feature parity test; upload by hand and time it; frontend exports.

**Rejected.** Re-scoring or tuning after the test result: rejected, it would spend the slice. Committing per-query exports: rejected, the licence for derived data is Unclear.

**Confidence.** High for the numbers (paired intervals on the full test slice) and for the parity checks; the S3 upload is open.

#### Interpretation: what we understand from the results
Four of the five methods scored below their expected range, all by a similar amount (about 0.003 to 0.006 NDCG@10). The ranking of methods and the paired gaps held (LightGBM +0.0035 over co-visitation on test, DCN-V2 below LightGBM, Two-Tower far below). The investigation, done without re-scoring or tuning, found that the daily co-visitation NDCG@10 varies by more than the sampling interval, so the range built from session sampling error alone was too narrow; the shift is a day-level effect and possibly some staleness of the fixed day 0-9 matrix, and the data cannot separate the two. The test result is therefore a lower but consistent estimate, and the test slice is now spent for these models. The bundle passes its parity checks (ONNX differences under 4e-6, zero differing feature columns), which is the evidence that the serving path can reproduce training.

#### Impact on next steps
Step 12 summarises; Level 1 starts from this bundle, the split rule (arrival_slice), the frozen gains and the needs-review list.

#### Out of scope
- Uploading the bundle to S3 by hand and timing it: needs AWS access.
- Judge scores for the frontend export.
- Any tuning after the test result: rejected, it would spend the slice.

## Step 12. Summary
*Kind: diagnostic. Issue #134.* This step writes `modeling/model_summary.md` from the log, the tables and the test result (every number is read from files, not typed), then runs the final checks of the guide: the test slice was scored once, every experiment is in the table, and the manual-steps log is complete.

#### Why this step exists here
A reader who has not opened the notebook needs one document that states what was built, what won, what lost, what is unknown and what still needs a person. Writing it from the log and tables keeps it consistent with the results.

#### Options considered
- **Format:** one markdown file with the sections the guide lists (approach, decision table, five-method table, ablations, position-bias findings, Claude roles, limits, cost, settled decisions, needs-review list, Level 1 notes).
- **Source of numbers:** read from the log and tables at generation time, not typed.
- **Final checks:** the test slice used once, every experiment in the table, the manual-steps log complete.

In [ ]:
# the summary is generated from files so that it cannot drift from the results
import re


def log_entries():
    text = (ROOT / "modeling" / "model_log.md").read_text(); out = {}
    for m_ in re.finditer(r"(?ms)^### Step (\d+)\. (.*?)\n(.*?)(?=^### Step |\Z)", text): out[int(m_.group(1))] = {"title": m_.group(2), **dict(re.findall(r"(?m)^([A-Za-z ]+): (.*)$", m_.group(3)))}
    return out
LOGE = log_entries(); TAB = lambda n: pl.read_csv(ROOT / "modeling" / "tables" / f"{n}.csv")
def md_table(df, cols=None):
    cols = cols or df.columns; rows = ["| " + " | ".join(cols) + " |", "|" + "---|" * len(cols)]
    for r_ in df.select(cols).iter_rows(): rows.append("| " + " | ".join(("" if v is None else (f"{v:.4f}" if isinstance(v, float) else str(v))) for v in r_) + " |")
    return "\n".join(rows)
five_v = TAB("step10_five_methods"); test_v = TAB("step11_test_results"); abl = TAB("step10_ablations"); bias = TAB("step8_naive_vs_ipw")
five_both = five_v.select("method", "kind", pl.col("ndcg10").alias("validation NDCG@10"), pl.col("ndcg10_ci").alias("validation 95% interval"), "recall20", "recall100", "recall200").join(test_v.select("method", pl.col("test_ndcg10").alias("test NDCG@10"), pl.col("test_ci").alias("test 95% interval"), pl.col("in_range").alias("in expected range")), on="method")
exp_rows = pl.read_csv(ROOT / "modeling" / "experiments.csv"); n_exp = exp_rows.height
tg = json.loads((ROOT / "modeling" / "artifacts" / "test_scored.json").read_text()); dd = tg["paired_differences"]
best_val = five_v.sort("ndcg10", descending=True).row(0, named=True)["method"]; best_test = test_v.sort("test_ndcg10", descending=True).row(0, named=True)["method"]
needs = [(k, v["title"], v["Needs human input"]) for k, v in LOGE.items() if k != 12 and not v["Needs human input"].startswith("no")]
lat = TAB("step10_latency_size")
STEPS = {
 0: ("Sample, licence, environment", "sample is healthy; two licence rows Unclear; Docker and AWS pending"),
 1: ("Split rules (train 0-9, validation 10-11, test 12-13), gain sets", "time split D3; gains 1/2/3 (D7); same method order under 4 gain sets"),
 2: ("K = 50-100, k-means vs Leiden", "K = 80 k-means with taxonomy (tie with K = 90 within noise)"),
 3: ("Co-visitation, BM25, popularity, random", f"co-visitation is the bar (validation NDCG@10 {five_v.filter(pl.col('method') == 'co-visitation')['ndcg10'][0]:.4f}); BM25 is the floor"),
 4: ("Two-Tower variants, catalog size, ANN (exact, IVF-PQ, HNSW)", "in-batch + logQ; exact search; loses to co-visitation"),
 5: ("Pool composition: Two-Tower, co-visitation, RRF union, quota union", "co-visitation only, 300 candidates (Two-Tower adds no recall)"),
 6: ("LambdaRank vs binary; with and without Two-Tower features; 12 trials", f"LambdaRank without Two-Tower features; +0.0022 over co-visitation on validation, {dd['LightGBM LambdaMART minus co-visitation'][0]:+.4f} on test"),
 7: ("MMoE vs shared-bottom vs single-task; blend weights; 6 trials", "MMoE with tuned weights; loses to LightGBM beyond noise"),
 8: ("Naive vs IPW (clip, self-normalisation), 3 bias strengths, 5 seeds", "real labels unweighted; IPW gave no gain beyond noise"),
 9: ("Classifier baseline, attribution methods, judge statistics", "attribution: leave-one-group-out; Claude evaluations blocked"),
 10: ("Five methods, slices, ablations, live path with stand-in classifier", f"best on validation: {best_val}; ladder winner LightGBM"),
 11: ("Freeze, one test scoring, bundle, parity", f"test scored once; best on test: {best_test}"),
}
dtab = pl.DataFrame([{"step": k, "what was compared": v[0], "winner": v[1], "confidence": LOGE[k]["Confidence"].split(";")[0].split(":")[0]} for k, v in STEPS.items()])
bias_pivot = bias.filter(pl.col("clip").is_in(["1 (naive)", "2", "5"])).select("eta", "clip", "mean_ndcg_true", "diff_to_naive", "diff_low", "diff_high")
summary = f"""# Level 0 modelling summary

Written from `modeling/model_log.md` (one entry per step), `modeling/experiments.csv` ({n_exp} rows), the tables in `modeling/tables/` and `notebooks/MODELING.ipynb`. GitHub issues #121 (umbrella) and #122-#134 carry the same trail. Every result is on the synthetic-category ranking task described under Limits.

## 1. Approach

Level 0 builds and compares five ways of ranking items for a session prefix and a category query: BM25 (floor), item co-visitation (the bar), a Two-Tower retrieval model with FAISS, a LightGBM LambdaMART reranker and a DCN-V2 with multi-gate mixture-of-experts reranker. Everything lives inside weeks 1-2 (train days 0-9, validation 10-11, test 12-13, D3); weeks 3 and 4 are untouched and reserved for Level 1. The metric is NDCG@10 with graded gains click 1, cart 2, order 3 (D7), 95% bootstrap intervals over sessions and paired bootstraps for every comparison; ties go to the simpler method on the ladder co-visitation, LightGBM, Two-Tower, DCN-V2. Rerankers share one frozen candidate pool (D8). Each method had the same bounded tuning budget (12 trials; 6 for the neural models). The test slice was scored once, in Step 11, after the expected range had been written from validation and the pipeline had been frozen. The test slice is now spent for these models; any change needs a new slice.

## 2. Decision table

{md_table(dtab)}

## 3. Five-method table (same sessions, same metric code)

Validation: {five_v['ndcg10'].len()} methods scored on the same validation sessions; test: {tg['queries']:,} sessions, scored once.

{md_table(five_both)}

Paired differences on the test slice (NDCG@10, 95% interval): LightGBM minus co-visitation {dd['LightGBM LambdaMART minus co-visitation'][0]:+.4f} [{dd['LightGBM LambdaMART minus co-visitation'][1]:+.4f}, {dd['LightGBM LambdaMART minus co-visitation'][2]:+.4f}]; DCN-V2 minus LightGBM {dd['DCN-V2 + MMoE minus LightGBM LambdaMART'][0]:+.4f} [{dd['DCN-V2 + MMoE minus LightGBM LambdaMART'][1]:+.4f}, {dd['DCN-V2 + MMoE minus LightGBM LambdaMART'][2]:+.4f}]; Two-Tower minus co-visitation {dd['Two-Tower minus co-visitation'][0]:+.4f} [{dd['Two-Tower minus co-visitation'][1]:+.4f}, {dd['Two-Tower minus co-visitation'][2]:+.4f}]. The ordering seen on validation held on the test slice. All four methods with a signal scored below the range predicted from validation; the investigation in Step 11 traces this to a day-level shift (the daily co-visitation NDCG@10 varies by more than the sampling interval), not to a defect, and the paired gaps are unchanged.

In one sentence: nothing beats co-visitation by much. LightGBM adds about one percent relative; the two neural models do not pay for their complexity in this setting, and the Two-Tower adds no recall to the pool.

## 4. Ablations

{md_table(abl, ['step', 'ablation', 'ndcg10', 'difference'])}

## 5. Position-bias findings (Step 8)

A simulation with known propensities (examination probability (1/k)^eta): the naive click model loses true NDCG@10 as the bias strength rises, and inverse-propensity weighting did not repair it beyond noise (paired intervals over five seeds include zero at eta 1.0 and 2.0; at eta 0.5 every clipping level did worse than naive). The data have no real exposure, so the main pipeline keeps real labels unweighted; IPW stays as supported machinery (propensity column, verified exact in LightGBM and PyTorch). Human question (train on simulated exposure?): recommendation no.

{md_table(bias_pivot)}

## 6. Claude role evaluations (Step 9)

Blocked: no AWS or Anthropic access on this machine, so every Claude call (classifier, explainer, judge) is unrun and unscored, and Bedrock cost is unmeasured (`modeling/bedrock_costs.csv` is empty). Done without a model: pass criteria fixed in `docs/decisions/D17-D19-role-criteria.md`; a TF-IDF prototype classifier baseline (top-1 0.764, top-3 0.843, calibration error 0.038 on 2,077 held-out queries; the Claude classifier must reach top-3 0.893 and top-1 0.814); leave-one-group-out attribution chosen over integrated gradients by the pre-stated rule; the explainer's numeric payload, templated fallback and faithfulness test; judge statistics with tests; a 180-row hand-labelling sheet. The live path with the TF-IDF stand-in classifier shows that a wrong category costs a lot (Step 10).

## 7. Limits, stated plainly

- **Categories are synthetic.** OTTO has no text, queries or categories: 80 clusters from co-visitation, named from a fictional taxonomy. Every number is on that constructed task.
- **The query category is a documented leak (D5).** The ranking query's category is derived from the held-out items, so the numbers with the true category are an upper bound; the stand-in classifier run measures the drop when the category is predicted.
- **The judge measures coherence, not relevance to a real need.** It scores whether results fit the category and prefix, which is what a synthetic category can support; it is not a proxy for buyer satisfaction, and it is unrun.
- BM25 has no text to work with, so it is a floor, not a competitor.
- One training seed per neural configuration; intervals cover the sampling of sessions, not training variance or day-to-day shift.
- Licence for OTTO-derived data is Unclear: no per-session or derived data is committed; only aggregates.

## 8. Cost spent

Cloud spend: none (everything trained on the Mac CPU, D21: Two-Tower about 2.3 minutes, DCN-V2 about 1.1 minutes). Bedrock: not used ($0, calls blocked). Latency and size per component: see below.

{md_table(lat)}

## 9. Provisional decisions settled

D3 split rule (`docs/decisions/D3-split-rule.md`, still Provisional until Level 1 reuses it), D4 categories (`docs/decisions/D4-categories.md`, accepted for Level 0), D7 gains (`docs/decisions/D7-gains.md`, accepted), D8 pool (frozen in `artifacts/candidate_pool.json`, Step 5), D15 position bias (Step 8, unweighted), D17-D19 role criteria (`docs/decisions/D17-D19-role-criteria.md`, criteria fixed, evaluations open), D21 training compute (`docs/decisions/D21-training-compute.md`, accepted).

## 10. Needs human review

""" + "\n".join(f"- **Step {k} ({t}):** {h}" for k, t, h in needs) + """

## 11. What Level 1 needs to know

- Start from `data/modeling/bundle_v1` (manifest `modeling/artifacts/bundle_manifest.json`: hashes, feature schema hash, snapshot id, commit SHA) and the arrival rule `arrival_slice` in `src/ranking/data/split.py` (weeks 3-4 arrive in slices; nothing in Level 0 read them).
- Point-in-time features come from one shared implementation (`AsOf`), and the parity test passed (0 differences on 5,200 feature columns), so serving can reuse it.
- The expected test range should include day-to-day spread, not only sampling error (Step 11 investigation).
- The live path today is co-visitation candidates plus LightGBM; Two-Tower and DCN-V2 are decisions for the human (stop-and-ask items above).
- Blocked items: AWS region and access, Docker Desktop, S3 upload of the bundle, Claude evaluations, hand labels.
"""
(ROOT / "modeling" / "model_summary.md").write_text(summary); print(summary[:1500], "...", f"[{len(summary):,} characters written to modeling/model_summary.md]")

In [ ]:
# final checks of the guide
guard_f = ROOT / "modeling" / "artifacts" / "test_scored.json"; man_txt = (ROOT / "modeling" / "manual_steps.md").read_text()
step_ids = sorted({int(str(s)) for s in exp_rows["step"].to_list() if str(s).isdigit()})
checks = {"test slice scored once (guard file present, summary says it is spent)": guard_f.exists(), "experiments table covers steps": step_ids, "manual-steps log has the S3 upload row": "Upload the frozen bundle to S3" in man_txt, "model_summary.md written": (ROOT / "modeling" / "model_summary.md").exists()}
print(checks); assert guard_f.exists() and (ROOT / "modeling" / "model_summary.md").exists()

In [ ]:
e12 = log_step(12, "Summary", "Diagnostic",
    "modeling/model_summary.md written from the log, the tables and the test result: approach, decision table, five-method table, ablations, position-bias findings, Claude role evaluations (blocked), limits, cost, settled Provisional decisions, needs-review list, Level 1 notes; final checks.",
    f"Summary written ({len((ROOT / 'modeling' / 'model_summary.md').read_text()):,} characters). Test slice used once ({tg['queries']:,} sessions). Experiments table: {n_exp} rows. Needs-review items: {len(needs)} steps with an open human decision. Best on validation and on test: {best_val} / {best_test}.",
    "Level 0 modelling is complete except the items that need the human (AWS access, Docker, hand labels, stop-and-ask decisions on Two-Tower and DCN-V2, S3 upload).",
    "Guide step 12: one document a reader can use without opening the notebook, with the limits stated plainly.",
    "A separate results table per step in the summary: rejected, the decision table and the five-method table carry it and the log keeps the detail.",
    "High for the numbers (read from files); the open items are decisions, not doubts about the results.",
    "Level 1 starts from the bundle, the split rule and the needs-review list.",
    human="yes: the needs-review list in modeling/model_summary.md (stop-and-ask decisions, AWS access, Docker, hand labels, licence rows)")
print(e12)

#### Chosen approach and rationale
#### Interpretation: what we understand from the results
Level 0 ends with a clear, modest result: a tuned co-visitation model is a strong bar, LightGBM edges it, the two neural models do not justify themselves in this setting, and the biggest practical risk is the category classifier rather than the ranker. The claims are bounded by three limits stated in the summary: the categories are synthetic, the query category is a documented leak, and the judge would measure coherence. Several decisions belong to the human and are listed rather than made.

#### Out of scope
- Level 1 (weekly retraining on arrivals, monitoring, drift) starts from the bundle and the needs-review list.
- The Claude evaluations, the S3 upload and the hand labels wait for access.